# TTS duration and GPU memory benchmark — Kaggle T4 ×2

This single notebook contains all six adapters: **Kokoro, F5-TTS, EmotiVoice,
StyleTTS2, OmniVoice, and Supertonic 3**. It installs each model in a separate
environment, tests configurable profiles and 10–500-word passages, and creates a
downloadable ZIP with named WAVs, timing/error reports, memory traces, source
provenance, and full generation logs.

This revision fixes the F5 eager-import dependency failure observed in the
uploaded Kaggle run and replaces arbitrary chunk breaks with punctuation-aware
splitting. It removes added per-chunk gaps, protects PCM exports from clipping,
and uses male defaults. **Exact duration alone does not establish word completeness.**
Independent CPU ASR and silence/clipping diagnostics are included after synthesis.

**Before running:** import this notebook into Kaggle, enable **Internet**, select
**GPU T4 x2**, and edit the experiment cell below. Use a fresh kernel. First setup
downloads several GB; runtime environments/weights stay under `/tmp`, and reports
and audio stay under `/kaggle/working`. Nothing is published externally.

Up to two workers run concurrently, each restricted to one physical GPU UUID.
Different models are preferred whenever possible. Generation progress and current
GPU memory appear live. The default target is **5,000,000,000 bytes per GPU**, with
allocator limits and a 10 ms external NVML monitor that terminates an observed
overrun. **Sampling cannot prove an instantaneous peak ceiling.** Both sampled
process/device usage and native allocator peaks are retained separately.

Duration means the **complete exported WAV**, including silence, configured
pauses, and resampling. Frozen duration/frame/token plans are reused in synthesis;
WAV headers are independently checked afterward. F5 and OmniVoice preview a
chosen allocation, rather than predicting an unconstrained natural reading.
StyleTTS2's exact preview needs style diffusion and may be slower. Quality and
intelligibility must be judged by listening, even when duration matches exactly.

This notebook's GPU execution must be validated by your Kaggle run: it has not
been executed on two T4s locally. Failures are reported rather than silently
removing a model from the comparison.


## 1. Experiment configuration

Defaults: all six models, **10/100/500 words**, **two takes per job**, one enabled
profile per model. Enable alternate profiles or add any native option shown in
the saved help logs. Profile-level `n` and `seed` override the global values;
put other controls in `parameters`. GPU selection, text, output paths and the
VRAM target are controlled by the supervisor.

The built-in cases are three distinct, complete English passages: a ten-word
sentence, a one-hundred-word historical vignette, and a five-hundred-word story
with five paragraphs. No sentence is looped to reach a target word count. Edit
`TEXT_CASES` for your own passages or other lengths within 10–500 words.

For reference-based models, supply a clean **mono 1–8 second** reference and
its exact transcript (3–8 seconds is recommended), or keep both fields `None`
to create a disclosed male Kokoro reference on CPU. Synthetic reference quality
can affect cloning quality.

`speaker_mode="auto"` uses your supplied WAV for F5, StyleTTS2 and OmniVoice.
Otherwise defaults are Kokoro **am_fenrir**, EmotiVoice **9017/Neutral**,
Supertonic **M2** (documented deep/serious), and OmniVoice male/low-pitch design;
F5/Style use the male reference. `speaker_mode="reference"` tests only the three
cloning-capable models and reports the others as unsupported. `speaker_mode="default"`
uses male defaults. Style needs only the WAV; F5/Omni also require its transcript.
For Omni clone-specific instructions, use a profile's `clone_parameters` dictionary;
the default male design instruction is removed when your reference is selected.


In [ ]:
import datetime as dt
import uuid
from pathlib import Path

ROOT = Path("/kaggle/working/tts_benchmark")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "runs" / RUN_ID
PACKAGE_DIR = RUN_DIR / "download"
RUN_DIR.mkdir(parents=True, exist_ok=False)
PACKAGE_DIR.mkdir(parents=True, exist_ok=False)

# Change these before running the installation/benchmark cells.
CONFIG = {
    "install_models": ["kokoro", "f5", "emotivoice", "styletts2", "omnivoice", "supertonic"],
    "force_setup": False,
    "torch_cuda_variant": "auto",
    "strict_cuda_driver": False,
    "n": 2,
    "seed": 1234,
    "sample_rate": 48000,
    "pause_ms": 0.0,           # Native punctuation already contributes pauses; no extra gap per chunk.
    "max_vram_gb": 5.0,          # Decimal GB: 5,000,000,000 bytes.
    "nvml_interval_ms": 10.0,
    "enforce_observed_budget": True,
    "budget_scope": "device_total", # Conservative; includes pre-existing allocations.
    "job_timeout_seconds": 3600,
    "gpu_release_timeout_seconds": 20,
    "cpu_threads_per_job": 4,
    "require_two_t4": True,
    "speaker_mode": "auto",  # auto: user WAV if supplied, otherwise male defaults. reference: cloning models only. default: male defaults.
    "reference_audio": None,    # e.g. /kaggle/input/my-voice/reference.wav
    "reference_text": None,     # Exact transcript; use a clean 1-8 second reference.
    "include_reference_in_zip": True,
    "preview_players": 6,
    "quality_checks": True,
    "asr_quality_check": True, # CPU-only word comparison after synthesis; it adds runtime, not GPU memory.
    "asr_model": "base.en",    # Use small.en for a stronger English check, or a local CTranslate2 model path.
    "asr_language": "en",
    "quality_timeout_seconds": 7200,
}

# Three distinct passages, with no repeated sentences. For other lengths,
# supply your own TEXT_CASES rather than cutting or cycling a passage.
WORD_COUNTS = [10, 100, 500]
SAMPLE_PASSAGES = {
    "10": "The final decision required patience, clear evidence, and careful judgment.",
    "100": "Before the library opened, an archivist discovered a folded letter behind a damaged map. Its author described a bridge that had disappeared from the city decades earlier. The location seemed familiar, but the surviving records offered conflicting dates. She compared photographs, examined a survey, and followed the river on foot. Beneath an ordinary railway crossing, she found the weathered old stone bridge foundations. By afternoon, the letter had become part of a much larger story about the neighborhood. The discovery changed no grand theory, yet it restored a small piece of history that residents could finally recognize as their own.",
    "500": "The field station stood above a narrow valley, where pine trees sheltered the road and clouds often concealed the distant peaks. Each morning, a small research team checked the instruments before carrying fresh batteries and supplies to the ridge. Their project concerned a faint signal that appeared in recordings during cold weather. Previous measurements suggested a pattern, but the evidence remained uneven. Some nights produced clear observations; others yielded only noise. The team had arrived with competing explanations and a shared commitment to patient work. They expected the investigation to continue for several weeks, perhaps longer if conditions became difficult.\n\nOn the third evening, an unexpected result interrupted their routine. A sensor beside the eastern window registered a sudden, sharp increase while the equipment outside remained completely stable. Mira, the lead engineer, examined the cables and repeated the calibration. Daniel reviewed the software, searching for changes in timing or numerical precision. A younger colleague suggested that the window itself might be influencing the reading. Everyone agreed to test the possibility. They moved the sensor, exchanged two amplifiers, and recorded another sequence. The signal returned at almost the same moment, leaving the explanation unresolved and making the next experiment more important.\n\nThe following afternoon brought heavy rain and a brief regional power failure. Inside the station, conversation slowed as the team waited for the backup supply to stabilize. Then Mira noticed a quiet mechanical vibration beneath the floor. A ventilation fan had restarted, and its bearings produced a faint rhythm. She asked Daniel to mark the event in the log. When they compared the timestamps, the vibration aligned with the unexplained signal. They switched the fan off, waited, and repeated the measurement. The disturbance disappeared. A simple maintenance problem had shaped an observation that initially seemed to demand a sophisticated explanation.\n\nFinding the cause changed the project without ending it. The earlier records still contained useful information, although several sessions now required particularly careful review. The researchers revised their analysis and documented every exclusion. They also invited a technician from the nearby town to inspect the remaining equipment. His practical suggestions improved the mounting brackets and reduced electrical interference. During dinner, the discussion turned toward the habits that make collaborative work reliable. Clear notes, independent checks, and a willingness to revise assumptions had mattered as much as technical expertise. The team began planning another series with quieter equipment and stronger safeguards.\n\nBy the end of the visit, the valley had emerged from its clouds. The road remained wet, but sunlight reached the station for the first time in several days. Mira prepared a report that described the original question, the misleading observation, and the correction. She included the unsuccessful tests because they explained how the conclusion had developed. Before leaving, the team stored copies of the recordings and checked the inventory together. Their findings were modest, precise, and useful. As the vehicle descended toward town, the station grew smaller behind them, carrying the promise of further questions and more careful measurements."
}
TEXT_CASES = [
    {"name": f"w{count:03d}", "text": SAMPLE_PASSAGES[str(count)]}
    for count in WORD_COUNTS
]
# Example replacement:
# TEXT_CASES = [{"name": "my_passage", "text": Path("/kaggle/input/my-text/passage.txt").read_text()}]

# Each profile is a separate experiment, repeated n times. Toggle enabled,
# change parameters, or add profiles. Names appear in filenames and reports.
# Parameters use each script's own CLI destination names (saved --help files explain them).
MODEL_PROFILES = {
    "kokoro": [
        {"name": "male_fenrir", "enabled": True, "parameters": {"voice": "am_fenrir", "speed": 1.0, "chunk_words": 45, "normalize_peak": True}},
        {"name": "male_george", "enabled": False, "parameters": {"voice": "bm_george", "language": "b", "speed": 1.0, "chunk_words": 45, "normalize_peak": True}},
    ],
    "f5": [
        {"name": "reference_quality", "enabled": True, "parameters": {"speed": 1.0, "steps": 64, "cfg_strength": 2.0, "sway": -1.0, "ode_method": "euler", "max_chunk_seconds": 6.0}},
        {"name": "reference_fast", "enabled": False, "parameters": {"speed": 1.2, "steps": 32, "cfg_strength": 2.0, "sway": -1.0, "max_chunk_seconds": 6.0}},
    ],
    "emotivoice": [
        {"name": "male9017_neutral", "enabled": True, "parameters": {"speaker": "9017", "prompt": "Neutral", "speed": 1.0, "pitch_predictor_scale": 1.0, "energy_predictor_scale": 1.0}},
        {"name": "male9017_sad", "enabled": False, "parameters": {"speaker": "9017", "prompt": "Sad", "speed": 1.0, "pitch_predictor_scale": 1.0, "energy_predictor_scale": 1.0}},
    ],
    "styletts2": [
        {"name": "reference_quality", "enabled": True, "parameters": {"speed": 1.0, "alpha": 0.1, "beta": 0.3, "diffusion_steps": 10, "embedding_scale": 1.0, "pitch_scale": 1.0, "max_chunk_words": 45}},
        {"name": "reference_expressive", "enabled": False, "parameters": {"speed": 1.1, "alpha": 0.5, "beta": 0.9, "diffusion_steps": 7, "embedding_scale": 1.2, "pitch_scale": 1.05, "max_chunk_words": 35}},
    ],
    "omnivoice": [
        {"name": "male_or_user_reference", "enabled": True, "reference": "auto", "parameters": {"instruct": "male, middle-aged, low pitch, british accent", "speed": 1.0, "num_step": 64, "guidance_scale": 2.0, "dtype": "float16", "max_chunk_seconds": 12.0}},
        {"name": "clone_reference", "enabled": False, "reference": True, "parameters": {"speed": 1.0, "num_step": 64, "guidance_scale": 2.0, "dtype": "float16", "max_chunk_seconds": 12.0}},
    ],
    "supertonic": [
        {"name": "m2_deep_serious", "enabled": True, "parameters": {"voice": "M2", "lang": "en", "speed": 1.0, "steps": 16, "max_chunk_chars": 360}},
        {"name": "f1_fast", "enabled": False, "parameters": {"voice": "F1", "lang": "en", "speed": 1.2, "steps": 12, "max_chunk_chars": 300}},
    ],
}
print("Run:", RUN_ID)
print("Cases:", [(c["name"], len(c["text"].split())) for c in TEXT_CASES])
print("Enabled profiles:", {m: [p["name"] for p in profiles if p.get("enabled", True)] for m, profiles in MODEL_PROFILES.items()})


## 2. Notebook helpers and hardware verification


In [ ]:
import csv
import hashlib
import html
import importlib
import json
import os
import platform
import signal
import subprocess
import sys
import threading
import zipfile
from collections import deque


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    temp.replace(path)

# Lightweight notebook/report packages only; model packages stay in their own venvs.
kernel_packages = {
    "pynvml": "nvidia-ml-py>=12,<14", "psutil": "psutil>=5.9", "tqdm": "tqdm>=4.66",
    "soundfile": "soundfile>=0.12", "pandas": "pandas>=2", "matplotlib": "matplotlib>=3.7", "uv": "uv>=0.6",
}
KERNEL_READY = True
for module, package in kernel_packages.items():
    try:
        importlib.import_module(module)
    except ImportError:
        try:
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = ""
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], env=env, check=True)
            importlib.invalidate_caches()
            importlib.import_module(module)
        except Exception as exc:
            KERNEL_READY = False
            atomic_json(RUN_DIR / "kernel_failure.json", {"module": module, "error": str(exc)})
            print("Kernel dependency failed:", module, exc)
            break

if KERNEL_READY:
    import pandas as pd
    import psutil
    import pynvml
    import soundfile as sf
    from IPython.display import HTML, Audio, FileLink, display
    from tqdm.auto import tqdm
print("Kernel helpers ready:", KERNEL_READY)


In [ ]:
GPU_INVENTORY = []
GPU_HANDLES = {}
HARDWARE_READY = False
hardware_error = None
try:
    if not KERNEL_READY:
        raise RuntimeError("Notebook dependencies are unavailable; enable Internet and rerun.")
    if not (0 < CONFIG["max_vram_gb"] <= 5):
        raise ValueError("max_vram_gb must be in (0, 5] decimal GB")
    if not 1 <= CONFIG["nvml_interval_ms"] <= 1000:
        raise ValueError("nvml_interval_ms must be between 1 and 1000")
    if CONFIG["budget_scope"] not in {"device_total", "process_tree"}:
        raise ValueError("budget_scope must be device_total or process_tree")
    pynvml.nvmlInit()
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    physical = list(range(pynvml.nvmlDeviceGetCount()))
    if visible is not None:
        tokens = [token.strip() for token in visible.split(",") if token.strip()]
        handles = [pynvml.nvmlDeviceGetHandleByUUID(token) if token.startswith("GPU-") else pynvml.nvmlDeviceGetHandleByIndex(int(token)) for token in tokens]
    else:
        handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in physical]
    driver = pynvml.nvmlSystemGetDriverVersion()
    driver = driver.decode() if isinstance(driver, bytes) else driver
    for slot, handle in enumerate(handles):
        name = pynvml.nvmlDeviceGetName(handle)
        name = name.decode() if isinstance(name, bytes) else name
        gpu_uuid = pynvml.nvmlDeviceGetUUID(handle)
        gpu_uuid = gpu_uuid.decode() if isinstance(gpu_uuid, bytes) else gpu_uuid
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        GPU_INVENTORY.append({"slot": slot, "uuid": gpu_uuid, "name": name,
                              "total_bytes": info.total, "baseline_used_bytes": info.used, "driver": driver})
        GPU_HANDLES[slot] = handle
    if len(GPU_INVENTORY) != 2:
        raise RuntimeError(f"Expected two GPUs; found {len(GPU_INVENTORY)}. Select Kaggle GPU T4 x2.")
    if CONFIG["require_two_t4"] and not all("T4" in g["name"] for g in GPU_INVENTORY):
        raise RuntimeError("Select GPU T4 x2, or explicitly disable require_two_t4 to test other hardware.")
    HARDWARE_READY = True
except Exception as exc:
    hardware_error = f"{type(exc).__name__}: {exc}"
atomic_json(RUN_DIR / "hardware.json", {"ready": HARDWARE_READY, "error": hardware_error,
    "gpus": GPU_INVENTORY, "kernel_python": sys.version, "platform": platform.platform(),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "peak_measurement": "NVML sampled total/device and process tree; allocator peaks separately. No instantaneous hard-peak certificate."})
print(json.dumps({"hardware_ready": HARDWARE_READY, "error": hardware_error, "gpus": GPU_INVENTORY}, indent=2))


## 3. Extract the embedded standalone adapters

The compressed payload contains the six complete Python adapters and the CPU
quality checker. They are
verified by SHA-256, written locally for inspection, and included in the ZIP.
No external source archive upload is required.


In [ ]:
# Six complete standalone adapters and the CPU quality checker. No Python file uploads are needed.
import base64
import gzip

EMBEDDED_SOURCES = {
  "kokoro_samples.py": {
    "sha256": "734f492085135cbc9519d319da4ebddf5a39b124e9a207dd01366819ec6c2e7d",
    "gzip_base64": "H4sIAAAAAAAC/9U9a3PbOJLfU5X/gOHe1FAeipZk56WMZjeTx25uJo+Kk9m6c1QsSoQkjiWSS1K2Fa/vt18/ABB8yHZSu1V3qYotkUCj0Wj0C93wn7473Bb54SxODmVyLrJduUqTo/v3HMd5nUQyk/AjKcWv6Vmap6IIN9k6TpYiTCKR5bJ/EZ7LRZpvRLTNwzJOE5GtwySBJv79e/fvvSdw4sgfjkRciFzO080GQUa+eJ0UZbhei/e7j2k+XwkAI3bpNkcIJcL0RLmSyfj+PaHQEv2NyOJMxKqns4zLH1dlmRXjw0P4vNrOfBjgcCUvl3kYHZ4R0j68+Uu0mD0ZPAoHo9ksfDx8MJqHx8fHj4aP5uGD6NHjR8eP5vPZ8UzOHeFs4iI8i09lMnVEmYdJgbjIvJhMHvhH/kAU8zjbiSLdJtEiXkuc5stLpIu0UeWxAyKYLHzo0e+X8rLsYx+RhDmTyy8vS3hznsZzKcJNsJBJHufwpMikjMQQxuv3gYDfDFkWZbwJS9lPk/UOcf24krhy57G8EPk2KcRfR+9pOYHYIpyX23AtCJ3+PE2iGEEBImZ5E1lepPkZLF8polQW9+8laYmAxKvBYZLGBUGP4jm29hTQdAtYzAViKWQyTyOZewKWG99WHCTpBfDN83C+ssYsCD25mckoAsaCr7kEVtoW0AaZptglAKiIC1+8UkCLbZaleSmj+/cA1PwsS2Ng4hDYLC4lzDGXjJoaE4DHZSHkJcx/vRMPBwNxADA2Lk4lADx6QpH7/r2wFKNjcfa3Lx7w4Xy9RZTEL+9ODl++OxEFLEEyl754m8JEZb7c9Wch4qleANqb9BwoDJsB8fd5PSK5CLfrEp+GYhOuiYaFLH2978L5XGaAIT8+XKdzvUoWXTwBSwEIwiALmdNwf3/2+1OcKBAMYCdp9aofAuawoTMZniEBeFVgStAcmlZUw/VYbZMzMUOOD/NYEhoARsBeA5qUQG4QEQiUV2q+DmFusBof42QHjZJFvISlnMG0ZLSEDbMII2iKawTTKcQfMAwScb6O52eFuICNnG5htWhWHpOMPsHi8ke9GiRjnr//hJMrLToiFoiC+CKBeH+FBr9/ePbGF88/vXiGbdOs7MfJGNoJkCNAzBJAz7aAHNIPOm9kWACTREY4IZk04zFnwfSIUCDtYAPMZV7Gix0s3yrMUTSmQI2ifxFHMF8cXKxj4DFPgIxB6ogoj89lfgjUwV0BOyfp67EUSsz5MCLBh3Z5ul4TywBzxYle4nU8y8N8h8QJoSWsNUioPM5Ks5ZRDHIXZrjDqccbhT6JeKTfIk83IggWW9wWQaBaCJoXY4Gt9NN8mYV5Ic2DCGQLSBjY5TBQaR6vwmIFmJnv/Aue+BtZhtApNK/+KNLEfAFRtTJfQPhG6ab6Wo1a7ArzGUc3X7bbOFJTygAUDKjn854gA/GefXz9+8vgw7OPL8UEtnIwGAzu3zt59ub9by9PgvcvPwQvPsHL1+/ewuuH9O7dpw/PocfL31+f8GPnzurk/r037168/M10PoHeVyjMhXC0muI93n88euOMhbM4WiyOHgCsJ0N59OThcBgdDedHj+FLeDwMj8LF4NHs+MnjR463F0z/fOgP+19WCG4wlI8eDB7Moofho3AUzo+fPHpw/PBoODyeH4WPHj4YhE8Wj0bDxwjuWmP76jXQ4lZMWRHBYMHAz8rVHRGqeuET3fEal+b+PdjAgvgrd3ui/7NhN/9ZvtyC4VC+p5djHikDFPe0cEHC0C4A/p0EQZTOg8BDbQEMVso8ABlVFBPT+UN48aLq8De5zl7ppj0ei3TXRGQ+iKVgs0U1ud4F8hI0QAE7OVjm6TZzrcbUMFRIuQ7raMcTKwA+cX4DPZSDDP/08VX/Mbd3butMCh4glLtMTpCdNTAGAgKAPvR/efeG0cX2BmzWgsmSuQvgf54Am6ezP0BsiHQhnv/2WovW4iloSTAB5zGAX4egjEvQH6gQ5jIi1bd/vEQPBVrF0wAnQz3o2y1oshzHQ9sH5KAS8h5pBJSAhZxvSyA26BsZFTeMhO87Bzsedffhh8S9yhhzqk6OscvM8r0nPewJVsTAxEoZZ+H8jPQU2rkh4AHsBQIyUoobcOaR7opF/0LGy1VZmIGfN+CC3kjkMiSqbOJL1I2qy1OjDeU/0KZTj78CAzDkl9twiaSYrxCbYrKOCyBwOJOLVfxH9sXp2VQySCqjRfcXaFo8FeHk04l4mSwBxupmLBxl/uolXKzT0OYYf7C3H2rnfhw1UW4IYRvtDtm2f8cYq6hr1/xGzKC1uGV3or8iwwhZG62ULE7IpE4vEphYtI+Pa2uxARKub9ixPDa1UhbX05q5KzZggqN2BcxqBrIxu29bkTka5f2INoE1vKYjfnPBigCREwQ9H/gT3cZD4fiEVEDd91M2Aoekvunm2dYwFHymTbWNwtNxDO7j5fRuVAsv++d5uOkvZ3t46YE/0IO8kHPwlNbir794Aoz8TQoEe/CUbcbKSixhJDDM776LEOk+7v78XPY3s06x9OjhY43FB24pwAIuwHikxWoYhnfEArXGKge+K7pF4V1wR8O/D/7eHhhHDzTarxPwFIF4pHeoGxu8T0FEommPT9iaLUB7gB25DOPkVvxx/RgHEP3gjRZ7VnE02CcSatNhhdJHyamNFVK4el7VIw3YMhattzzll5dsihJQgUCfgvfN+kl7pKIow12hHEZfD3oHVDN0EvqbfRMeVGz7Uqtj7WLOwEOXMgHtn8CCUDgGtjmvwK0Ux2XpR7P9w94Bd3Ty+ujjEf4NOjPENqWPYEZNCj9PiziRAkHhjpQhiVXkLRziKStV2CyFXoQ5uKjAcgNweQr0N8HdKb+G6gkafev4i+yjv2fjHlI8ozIYf0nTtQyTd2QvhutnHO5oT+tjvpWteX2Q0RaNBfYpZylwzMB/8th4vvMVaE5U5bCgy3J1ywyQsbczJK6l9U6d98/fBMOH8Ig+jY7x06vf3j376EwtKata7YUM6GTb8hax7yhLzendhTvsgNS/gsSvwnXRpvHfc1B94I7rUJem7RLDMrBLKYa5M9Gnr+GSQq4XYJAX5b8P+w/bBKyGUi5l3mcmYHuiio6QVi04iBbnEQiMvNwpX7doziaXYAMkIms4WTi/woUf52OBZtJpUeZT8U/xNoVdN6FfdS/sbbiRBdi40nLAtLumRgov4BnZXOCj+wj7dDieiniBUM4xAkHQAXdJTzS9c4n6IgnznScCcrQYxbMEzCRGFGCr1gDN6uCzwTNuEbcAOBhd8NHMKtx2Fx/VY4A6y6UAGHDFxNmWC3CiCrCzNDurITEOExcUfAY56+pBYDPE87JnDc8MJPM8zSs/iy0wjOOAiAXGtHwsp9fkI8T8KvTBiS3H0BajnCHGczI/0A2QotSA8UrElYO8g9tcmYnX1xVYEJpISAALbohBHVaXviuY9emqLnvmtXA+KZAmwEfx4TO5K8biqiDj0lUwetf2FHE20MwT5+F6KxF1jY8Pe3ZTuE1aMnro7DOep9B7Wm8SF8GMdxw0sxYpVHFo1/BwcAIGlES5rPakuHnP2kygKGMNhsHCBl/QrDyBLZoTqRPwCqZxzWwxk5ojFGCnPSpPxUchTIG8hOOc7p6hUTR0YwY9e9+AmZJHt2NGKv4bcXPJviQIvW9BMqEgwu04ot4iFBUPFoQPiab9Swri8RtQwjgkdgXhcjteDQxqfPbvR3Uvlt1DW7h3jFXmu46n+I+hTez+jHev3V5e4smHcD9Co5eIuyd+x6b0uYdRZ2ixZ5xqtq8TgB9HgmYN4gn6XO9fDGVEtebLeCtxW297I62boxOc7/IWBloMknwD+lC7qknmg6wOdCP34MCI8ZrcBrPGxdCcmEyEDjvihsNnPnAJ2AdoROio4sTpkUSmLjAv1LItmV8bGHuRoz8hC+G2wTlseQMG1ODb0LAxQBvBMhzaJgM+8dF4C9B4s4Ar8wjfm8YutSZ3Vm3LHs6pekwUMO/G9jIpzX+yzbL1zpwqIhelC3VYS/xk06eBZiJ+EkM606Hx2I+vPSNHKCDfHJ93IQAOjSdMFEANafn0Zvtn4Gih4+q0iAWe40/gT+lB6cGPGsU+oPPzRIwODh4edY3/3A7ZsodGkVsed4FhgFoAE2wudFUfHvdncdnUN8gcCdifyBzgcaj4oLMJLwMM8QQU4qGvTJkqYuCQMx2QM+2ggxuQg+ugExmgc4mv7BVUth6eRflxAX40mCXuUpZhWebEAB5h0uvttY2u8H0lXhlEi7p0xP/TxKJvC33zGhEawFqYZmrS2OBBF/HfqzWlaHp0yH6zOcZHXkBwVoSMIKkjzn9s4xzo20BY01GzBOL0GM/PsCtPiVY5wGAIPhs+GdFrw7PbKAyU0x5sZginC3UtL2k8z46wUGydInIKio3irODdqZa4J34Ww9Gge2OoOIdZIR006UMPIg78tmAr6qtZ2oyDD/cNYgdCWiPxMK1Rcukvtus1RWrdHCOf/0SquX8ejz9HP/b+7HiMAwdMe90D80szZEf41Kl7hiz6tHMoKbIVsOPpckxL55uMBVlouBj0kRxE+K0QcRznDQWZKUvGxx0drsHNUoyRpevdDyrRQsxlvKZz/W2B7rja74AL6IaNLOO5TyfUFpoNXMQB4QECyT7UBanUE4eH9rNqaiqJYReAqcPTQ0k4JnupNZP32wTzYTityWROwAKcpzGlfeApD7xThxrAgTOMNvCh+WHMIdGimgWaV1kGe34icFQ/pweu89n5vB0NhhH9fPJDb3ptMQUwRCExlA/ccPrdn7HN6OHn7dFgMPq8XSwGQ/w5XEx//A/HMyP02tptZCDqRr4EAcOK2K/JP3se6DHq9uv0gg5m0dmsSz5nk/t2NEQ9KzoedjyLOjpnebpoPy06Wv7R8awo28/OO0aW5bzjob9sP4x92X4Y+psOzBsPLS9csQTQFO16t7HXYY+fPuv/d9j/Mv3s9360l7MdiKitkfEDGH5DLdUZQH0btnjLezomBjymn0fEXHNirhn9DBssxjGcQbW31OiswNg+o43l8VFAJS5MrKnaar9KmdFZB6XP4O7grSouVjGnQ8Huy9WeayQdHeqEI7PRYGo0ZMMwykNMT6vcB7BQSCvScNxhn0WkDjAm4nRamSN4CLvMw2xFqTZ4bE+HGyDUZD2IwdbWpOrADV1rWXme1LCxfLBLoesmTlyVQgRymaRW0RGawMY/WS06/JI5cEuMaTs0nW7PiZQEyNUhzZO+eUwmmKkEJxtjqJLHOB3DoNMejt4tXclwHrUHmrYfKeauUNzj2d1lDv+eeQy7B+uey63z4LWtmp32hw1I6twmzDAT13WE42O6XA3jBhdoZuMm0GI8re18hmhpe23NuJQSqO2twKh4D5hvDXuWLdKxis5Ue/dEJ/LR3mVDE5HkYDXY+MUKzAkNt0+HNH11doM7ugwzPH82m5fztWBxsh369wmFrXkCBeYzTOCRHxaY6LrTOEd0NgHPCbmjUc+fp9lObzAaUm2iHNfWtQ3Vg9r8xKEYgs3a433GQ5I9Map0MsP7uSFcNhgOdwf+AzBB8OcB4jlPCxd+gUygwLk78PBpFnsMpNfrwVQoAmIhbytvHP90TI2n4mBC47TeM1HHpsHpeGw4SS07t6yWnUxsmH4g2XoJyvRMJiC26RcfCHjoH8Sb7SbIVuC+bpT5R0Ic3d6KBz5IShYqztjAYR5EtQR7Lj6PixizPxUQkZ7LHKZ6AYIWtqVkltEp4wjwDaVg477Q2MtNVu40AGCsFVjQSSrevXshFuF6PQvnZ5Q9ex7Gazz505mzPyiPPtvO1vEcs8climdxkeOOyqHHukhFmYOlh6yAmNGBtUaVI2KYy4rppZRIyQBlwgSLv0iO8FE68kzitDAvmuSYhuuJGSa0JhV8Jku0Ddcq6QthVQZ7uF2uSk0eSk2m+Wu5pDYZ0mcmKXmZMzrVYlQaisGiduJVtQ5jOZ+NnnOYA5rrrkQxTWvdyDxoNFRRnznsfrDqYSfD4q1h97oc0DHPtZLsGXNFQ/TZDm4qq5a6bsvQhYPZ62Ar4oEqAsXPCRBdGUaIwBUO+l1+/VTXDRzqEwsg3QkewlYsRJlbOa4yGSOcl1cftm6KoYxoTgMdz9a++YapvUvYIPihaMwKY6IYRqFkIsWoKpF4bE0XbCb2rEqLwzm3rmNS5mQwT5cozd0INzjL/zIFuqnPsL3kunKYLDGwyM1BDfYFQQpUcIequ6LbRRzBZsHEW/4+C3NMqP2TA+IS4Lv8/sAAU93wMLEoI7AB/Qs82XUXzuf8inC5HovTKwAz/umKOl9fT8UVInB9eEUj294UITaZqBlVRGjBdz6bMxjr3WK9LVauRS1qHOAho4tSYCz4eDwLd3jmOOajwQadSokaDrfxhESHj2oywIgRwfApyvUjaPpyk1VJoaoPo8dnlhX6dMoZgcYsXDW0BxIKU9iDEDzxWB0zkwWUlJNRdYBP6wiDfcbszMYhaD1ZpEIhl9kadRlia9HChFbd5ozDAnR92YgrPByALuT8a7KsKNf6htbASMOHpvXxYHgb7OPjYKhaD4eDo1rzhq/iIFuUF7BPVrmUn5MF1iEt0P4HFAnCqW6CkUNqhR+qdpZbYIJsFMZOlhL3ANkUto/NeHTmnR9UIGphDMTDenM80GfnuHF01qUOo6iomS7H0vkDGQ7MOYd63TYhmEW3ZwFUoREdaa/nENwh0G5xSCs+MdAxkpDSEieU1++Djl4EZNxUOQb1g6Uuc7HxztRq4fti0XpfYnaddT6N1QOr7RJTcBbA58Fqa6oIVgv8FujczUanM7UE3PbXN5ij4Ylf3yuro9HaDpEJU+NgRcqUpcEnYK+pAWmL9qkX65MPQCYgWpdGcXS1HZfJcFnfHGt5wDi/SEytCFIJ9E2UztnywVVPdWkYRpNlch6DMsKsGFuJ9HhKgJW24aIYIYDdwxqhXQDi2y1cVReg5R2bWsE2x2CJ3dBK2XCqRj4KQCPmeaoyCjBCUvD4Vh5I1a3nLyXAOZ8XQQx7Hnb01bV6hqG9uAziyCGP0MKHwh64Kwy/dwz43UQ0yka+ZrUWteVS+cK6BBNEDhWSqZGuGuOA4l9Q5OSqjRaYBbVF07YnsD+drcEmCtRpkWsfPOlF4bDyRPXgr64djlY6nSqpgtmO/XOUTq1jiwMxRFmu/1samiDxgTgeImLU2mkfzjAK+BLszcDY/XuMyBqtHTpAuAgLOugAaYSG+5bO+baJgfQULXphYunzbOvb57XW+HwqSbSokYGbwRIEG7lJSd1bvZamVwAmV4b1Y7LQAHy7n6U0dAKwoS16tDa5vdqImN/TefBiUb9uyDcGwNOnuxJUgQeTODwn/6dVW3cTAYEAgaqaU8gH2gR0m1gd1ibpiQ6qG9hUdxGgla/BwqYoi2qlNGuDHuT3XODHNgyab3Xes9nzuw72tDTalaOy2sdCZbM7y2zLO4AmAi/ATHEwbBLgG4rvBQXo92IRywheY5rSdTfNdsl8BYIY/M821ymCsRqt+uD+U5M0Ldqd1Tru76sbtLvGCzL5DXDPAEOfyObTOzDVm2YNJnEZ+WvsAEmuWbY3cYvL9FI0Txz0woBi0dNoxvFp8sw5ZkJm3aop7u+l52466QfNPmYGgU0i6FDb2Y1OxDV6x1Tcw4WozD3N3FLqB8KTOPKZ2ZzzMCOXvF7rSqVtUbNQlRpylalVpto879CBJ6N9G/U3pySUcKOBgp3ae1AbVi4aIugIVZ4m/G6fajXsMdcG7AkNxDO4TPQHT1BFSgCKfcIiUn81MoHDBAFFd9TJr6pl4YROIB2lP5vBVeKlMkZ6NpS6BWJB3p+CaiDoIvWJgsUGCthEizKRJRstVd6KnQXLlT3UfJtZIc8CjBoQXqdDED8PrYAzOi+dnc5knsDEC5A2uu8I+g5HN3ReyiQgJINVmlFX7vngLl2SYLEouf1IqajeV9m7n5JWiVM9U5Xq4tHpVGUhlWe1CbOMTF8O4pkd1WE6WUVeE52WY560+MMqrq3vAL3SXL81UZ6Dq15P6kzNs5jwL4/7oBvvVkP30IAwtoQ8D9fadzIB0EnllrhYqBcgiXgkXbfniW4EeEQSLgoq1SpSBIPOY/CDDobV8nYIDjVWB2COx+a13WNay4ewQN94kqeSrcIS7Y+ipGwrHkmrA1UAiTH6CtAphdzdS8bzso5koGsmK2Snxr+ttWCn4HToD6ZgW2FQ0MK7Vwuj1PcondQxDGL27q5qr2CYtZ2YdEnptJeUlGMmoWHW+rf74h0Xumm9rf2GzMB9m3BvENP5nUpiNYWsAkSaYOFVmVGHVhaYRyoGhsewZWgOYjkZsmMLYsktwGKi4bUDzINTFIvtzLFuduKnStDTnqX8LsOZVqJEhlKJVru+txcOz+qQU7+onUVRxNIYVNShGhNPWbJAK1Nd+ajoFmCZDJUR3ViMwIXHDP6jTArMjaUT6vmZn0TxBnnryDwpVmEmqS4DHmN0avTg4R0cJ8z5l5cZOMIgVKuCZ0HgONZOkx+LK50tx3j9QC1+8DiYVMt7rblzhi+xW88HC8O9E1pvURZix+adKJSqQjg5jcUwR6o0VJMVmI9IkGEirrrNAukGlLKWjrU32WcuCN2VvIzipRXfUmcjGVMcqU1jnw6mfkW0jCo78PmNu0vtJ2rIu2kFjpaIMaIbY/UvQSxw22ziS9RuGyP7NMbEhHSxDYgp9wI8qZoEYHSA9xCjLyCM1eYyDSy1UrsWoZ4Ve1tZjZGhVR6trgfiZzUNoA+n9OeO06I2rT6qxF06NzNkoODiNjPJlZ1BRqyJ4SCkMmO3CZqFepZcEUdPDsVCha2Cq6j09VUofpJeuPD908fnvfH3//X95vvo4/d/+/7N9yfXwRVeUOLjj2OX+OV0/Hh67dRG8jdn8NPlEuqCSwjBHAfDKEjPanoX02LWuDQBYbwvaMqN/6L2mL45JkA97vYq2xtDxSBupEspKnzaS+k71eGvtsUbfrFVXcR1P/U9e6UPqYi91GfkeX2iiPVc6rHKricbA3TsPJw1jwtBSvBReu+6pQxuLF2683ki+xYg47aWEWmQvVLQW2dsVICLmd5RnmaNI7tit5ml6+Kmw0Tix1Nr4qfZ1JIPNq0aFJq2NUNEx7RoTcBHPOAZ4aEkdVGenDoq0c00fHTUK2mhNMVgehspnff6wBx9WSxDtg8paeBDls1qeA6vFRsMUOamOtmmR4LbjKnCnFuSXnNPT8HzOIAXnhhMpzr1Q2tWzCdlCTVphSd4xm2AZiitGL8O7CYsKt2ON04V4IufSdeArQOjCijbkV1gqYch+Wl9NfrCzgiaSdrlvJD4xR4Er6tJkNsCxGji/g/+6vkYfrVNEb7qK6qBCdT9Xy5+ASmP196B5SXdPhgHfTshpeqnrnlLcxbjGoICj87D4nSMPuLj8dTTtPeIWha8Sy73bIJcF+XGjexxzWVwmtKgSjZpHLnNrrolBkf+AOu456OOC0F+TvqYGWwl398EnvOEzPOeP1+Dn+hu4mQy7BFHgADPJXEMkMmub8yVI4SUNwAIix5VONaWY0+Eisp2xsKI4laWqeIQaKM/tlJWI3xbMTfYlW4rymXwg6YVrp1NjVWFTbubwJpTnGvR/VrXiY81jQ46D0C7gklGQZFZQfKiUkd0bInLP60Ha2vuLApNJ5w1g7WBp5Jj8ERTecT+cpTxGI3cupuTpjwKxz/AqEqnqO2LkdclXJsZfGpBA5N0SvXUBjkr9UgN3ASghTBpFBfrVbKCjTv+jHuu0ilqHEvMoDXWrG0pVF32bSSqj00taPhpM2nNSqbVXhpjh6gZKONWiiuZAl0JlXfQ76zj36ZWJhnejWgSiMiHWRak2imcUl1ixDee2jrdPkZtKvaGcrezhUjFIqN8HY90TFhnei6Ldj5xY1TOBhY/idGeTNQ7Eo+28TOBJg9Y7ZxedkM6Et8HyVckzrAIbCHBPtLJSFEX1Toop9lDWwtx0szfWJJioWQja7aUtNnbN2FiQh+QJC/QiBY1SK8DB5xUnNglmURhkOi4MYz1zNutXTUO7U6NCJw2Ujt+3lMA9n9/2Tuu6cFUuzk4gYWqPqPgVfQH+qzIJHRS8f9o7TvEoQVFxRFwedtqvcq6rVURdJRKeK2yTu3krelSA7vwQCWycxZoLZOdAYLVVasMAwjtybLwtsMiOvNPgUdqMryeJ5wX5oJn7XQ2KtcQ8s0hehS8XNmnbvhT9GAZzBcV+c7dUwHuchxquci3OP0NZ1pHF7HkUdebTQRbhfVqyIPaRlbp49y9Ua/GsZemIKAVRSES8+3Z6K7Vxz0QLqkOeon1bWZuaPTIyBrgpto9r1WkWQGiS3ImthnqkH7Cs047+m+950wUk1wCLRupKHZjHc60WpujOKsZXagYVCcZAWZM53Gkz2obZyxVKMk6dzEJOjVkZYmX/qDteR7mnN5i26fa4KataD1mVcyBP+xrhQftZjVCE2mhcfftYY5aonrb5srYHap1rq8ndGsscGen2njkMDT5prufrqDu6GC8qG6EG3E2Wrvak8aA9R3KA9af1ejHN0ileXAGkoqyK/jC8LUM89pl5bhh8B5M2OJgcMV0Cbk+DFSnfuqwzz7AdlgsAeBGZdFVWwdotolP+dPU62hj+WrQznzrbFutUmzJiK6WtucGbauvU79MyWNo+l/XjSwzlDta6FSv7MEcTvmAERrJMbXVNiif6r5qJCs7WkdRD2nBUdrwMbmnhU/PTmbtzGy+fTH28C6CP+142b1YX8mO38T09miInZJu+4h0w0Laryg73b5yrpGSq69eozOlcVcG7h0jxfp+Q1C2qLjGrRuwOGzHig7DENPWwUEz6MJt60zc0cNT8TJuXkUk7OaeeovBiG4wWH0zl1asBzy5sAwwdTqn9DWXn4ecvm1HBpthQK+aRS0BK15Sjmw9Lug2QCk8fDTf1m6vpwOFkU+/94ccDfxTBUKf/WmM64CbgPBMa9iNrfnsb5MCfHAJazyoRQ8xIbcdIRR/qbpaHInlbvRHIdohvleDtwAF/GCZ1IOFd4nY/stjtbdEac3fsNDzqMU8qXc9wtlND/6jCxqGUkWuBu5V9DIUGQNJprVAI4fWfEp4d9vVSDQE5RI07Ux1hpZk1TkrN77rQWuH/HqhtOl5nK4p4ZCCAOrOUftWTjp7ruq09HktIfAUk0XZSrnhgKShgVriq1XHaptf7Ys7zLnYJkziBd6jNyEx5jb1kXKKBp6+9oYTPMgYB7/ohF0Zfa9klYZOXivbR1yGbMpBGE6vVhhBAXn7jh27q+1V4Z9EoDYu/rDIAOvafCu+F+7o4OBo1Gslx8Kst2BpdwGqpnLzsWKNqYNWhXzDWdUeTuWrEgXb8T0mFoXbbCeowwG3B9ZuOFCBRW2tc0eBcMPH74LVUHOtMK1mjdoqW9wBq2gK0L+We8h80n8aZ8KVxBQ0S0K9bxW2HQXEmCSW7WpHxbXSHKv4GeSEtU8aRKbS3RSPviltaDlXrq9t9tvdm5XoFf61uhZXv2g7pBi3UaN6dbfaPO8mkPkIvvJwIA4ORO0yIDCpRoNasAPziCdcy46UwwgSlpXPCoNdr9dBO3NFMOX4Mpvih5/pCt/xHgIcTPiG30Nq3JLZZkTKEGk5Wx2C2/S4q+zW11Wbv2ilLK9bBXdNONumGFqTib4jprJcF0rjBFe1bQFbYDw4iq5J4sA7+HntwywcuyS5qou0oFvlkfbTAuUJVUlWpZEAzsKwWKhaTtPCE/s5zxPqFmXOdlRf9N8MmTh/f/Z7/Xh6gYocxsBP1RA1hcUJ7FbkCdv66uzrkL8xCoiBZWRRldeErrSyu/TbzIEHmq2HNQZrjIIs1tp1MBqP+TMy6oBumbMH7mLLBs9Vy7BN1nFy1oxKd/EkXtIMEg3/TJVYhPEaWI+vhjIe/LnM40XMmXH18pFWNarFHDWDj7W7ZQhp8X4HfxKknikSqKB3en+SUu7xV9frihcq/76i7809jD/ZfNTVLZdrDgmZY3BaWOhNv7u62PrejNUdI22aBl3gVILUWp7LNUK66o7BO1QjwVcc4FE9/qEKjgviC++mTsU8XGPMzBKpd5TPHBsc1u6ab8MPFyVXI1U43aInOsBddy/qreGUpn9vabvOaIpmcB1O0d9r8ZSFcxJiSc8V0+ha/20dvtaU9pjZeyzra39/D7Mo1ZDXjZvj1P1PQP6AZHQQUKg+CLCqOAh0uL5es8uy4GSHMvzlZVy6VIKs1a2+8tWWFfatr554d6I+UHQIr65lP8S6FBY/2UcgigwvP3x49wEvgMX31w6XiEyqCv/eDUgOdZEr9r1/738BM6dP4jpzAAA="
  },
  "f5_samples.py": {
    "sha256": "9b6b0018c9fb89a83d7d37f356f0399d091f8d83eafffc6ebd1ea429dc86e09a",
    "gzip_base64": "H4sIAAAAAAAC/7197Y7buLLgfz8FR2cXR+6x5Y/+SLcznotMPiZZnHwgyWSw6DEEWaLaOm1LGknujpNt4Oz9vS+wu1jsi8zT3PsA8wpbVSRFUpLdnbPAbcykbYksFotVxfoi+y/fjbZlMVom6YinNyzfVassPe45jvOhCtIoWGcpZy9Ohx8/fmA3E1YGm3zNS3abVCsWpIx/DsJqwAq+LXnEom0RVEmWsnwdpF6v946gsWNvMh7CP1NoF2abDU8jHnnsVVpWwXrNNkEVrpL0ilVZEa5G9G+wjZJswMowyXeDXplt0yhO1nzAqt+jDTwPYl7xtMyKcsBW26sr6B4HIfdX2+WA3WRhVs7nY2/ijQHJiFUrDkP/vk0KDoNXZS+L6VkWx0mYBGuYILzPszKBwXcee0qzTqqSwchFyAlIuOLhNcu2Ffvw9pf3T5/7759/evXh1ds3sx5jV0nFQuq1qqq8nI1G8ASQ8WC+ow+/JjdJNBJU9LCp+Cj7DZ8qAtMQOML0/Hh6Oj09O46W8cVkyc+mp3wcTh9dTM+CMA6C8OTkYnJ+FgIEsWJsuGF5krNE0nTImTeqR5Ft4lNfrp+X79hwiFOum8H3in+uhkhmlgdlGVxxr/pcsd8AAP5g+5gXPA35kJaH1d+92+DGeo+QmPMcmYNVRZCWYZHkledAI2CHXg/ofcVTDtySFeyWJ1erqpyxp0+HP/3X4ZunwxNYN7denLzgACNJuVyDPEvSqu/13gCv3cDCBhWsVcmmJ+z65ZfHMISY5JBeFFyxLC44/5xnRQWAfn3yyeu9V/iOymoHsw6ztCqydcmCgrNym4u2Hvu4gmY4RJoBgSOeIwOnFeObjLg9K3q3WREN+SZfBSW0e/LuFTSEEeFzEAV5xQuPfcg5DAxPaimBhcpC+jgA0BVwWQ/oDm8BJeBNluAoSYgk4MCFebWFjxWMkrEY6MZveLEDZk9gAkhwELiPxOhyVjhJHI9EMyhpRusEPrubLM1GijLRgKQZ2a6EtYeOvapINhsQqf5jIH4GvAsSAOzNQeAnw3NWghSDQOiBUDxQWnhzwXtPV9v0usQlhLbQMa2I3uE6AKzYEuU6KBJYHQRxzXnOqiTdsSpI1lLJ4KLQ5D6sYDmAAwgg9lZSzUtc0ZLIC5SFxYJXgCr8l8AEgYkqNvaOWbnObqPsNlWgeHTFWRxEXJAG6AFrk5Q4YJJukyqReg4pAxopRSUDQHdSCQJmW9AlHhEdet8k/BbJrbQiO5LrC2gBiBygAHWvqtWRXG34GhQprU0JY6NmQsWUBhWwwHq9YwEgleVFgnwM0wuuSUkmG+6hwiqrBAQdGA70JothjkgPEjwQj6wHJKgMDvPYO9DJKUKIMl4SBsU2pU4xEIZtsoivgZVJe0bIsE+zddQDCgKtQMsi5dZZEIml0msPMwcsQ5gAkQcEBTGE9eB5gBK43gFfPn33C5IGB4t4HGzXFUGBVSzZF15k7Gdo8On9k9cw6i/Pnvy1ZO92H3EfUDMAvKB/CBThUW/JEeHhcBN8Ht4UwWZ4tRzIHQnlnRegFVBA0izV3TUpSpRoIA5AA6A9sRaroIgEPzKQzCqrSPHQvAixGSHG1smykAwrF9ea6GsegPzC7HG1BKWyHMcEaFLa2JtPr/9Wg96imiWqoTJGdfMYkAFq9HBx+OcQmJpWHXerJU0N0NwC51bAuiAkQDBN26stYALiAm0AxBB3SVpXD/fzXi8usg3z/XgLDMZ9n4m1BSyBBIIyvZ56VlzBrEquvgObrWDu6qv4BQ+8Da9Aw1WBevP3MkvVZ9jYV+pzVqpPRQ203C4lHeonu/pjtSp4gHOvHwBbiSnkABfGVvi/w2HoRbXLkVjy+ZN01+s19mo2Z85DN1en9+vzVz+//Gh1Pj/hp8HJZBxdhIDeScSn8fnJI/4ovDg+uwjOTk7GywgskcDpfXr79O0Hq+845svjOIouphfLMDo7O7ngY+h5Gpydn4fxoygaT6Y8nk6c3psnH199AqyffHw+YC/fvoPe05PxeDxggLea08snH14+/wCvvtIO7ZRFOIItvqrKEQnzKIw3sNE7M8D6HLA9jc8fLR+dB8cXUcingMRkHIYw2+UFP72IzqaTR5N4enLKp8v4LJg8CscXJ0CR49Pp+NwZ7BkC/t0KewKHuZgGk+ji0aPTs2X46NGj4+DRaTgdwySncXgxOV4eT6fjeBrwKT+LxyfTyfQsPg6i8+D0URQAPXGYu16vByqCEff5wIalC//czEDyyuqyrIoF+2/sDRpac/rVZ8Mfa2713gQbXuZgCM4I3xwa1e+eFFdb1LHv8FvhgtqnPQrYfu77URb6/gC1BjAtbNc+7FBlOa87vw9un+kOL/k6f6Ga9sVQXhBFiC+N4TrDIewjcXLlDJAr+RyZFExV6Dh3/ssH4Ids+XcOmwQofDUIqEbQ6amwDVKcyWP29G+vWAb7fJHgNpVUjhhNmqVzOexmi6bBeueDwlhvS7CK/Ksi2+au2bqFIO4Vzr0tyCA0Z7FvvqjAumb7VhvZtpEbiC0LRBY3QdxAS5i5sx++ZX0+DCXTIN0LWVuVnfgrBY76HtQ4uUWwWbw4hdn4NxP/pwDW7vnrJ6ZXsncs2F2D5cFhandIjEcdtuug2B2CmZXDKCm64P6NoEgif8KWTHCmtws2a9qiwD3AvdYnkfbAFdw7UqpGAFoN1FY+n+xrXsJO1NnjZLqvSwRsECK/qaZOmG8dNRn4jCZKuI2C2Zu9WAprq00QBRK/uY72hpz+3gloT6JzHoae3gciR0t3uClVfzC2AgPCeC9TFllZonX6rX2luwbaejg83P+05hGyS4e5MJ6Q98DhGJKpbVjJA7CivemYATwaYx/mhll238DPeJhsgD1//mkAfP852Ww37HTvuqJ6BOngwyxd7wA0GNmovWsl/VOWoY+iJOlJKFwrNeqLYF3yvSudG7zaQHdyL52VSzcUrlEp90v8MQDWz8TkPwIoMOOkI0wmM3UeMFLjZPcR95Qj3ALWQQ4bwu9bMPCSL9Aarf4Cdwk52N6ZVTwvO7n3eK8UhvEV9CuILfYQZW/f8jbY7ekz3KspMuRTDr4WrkG4Qp+2nLsOB+MChdjZJJFS0E5xfeL0DQUhGu1VWdkQ5l9qdnFK0Hbcr4otr/XKL6DBt2mC2z/4XWRBRMItIg2g/K79ar2itRwW+2XVu5+LUHBI6DQbdcI6U2j/RP6K9oE1K+lIHHjDBPN+mRVDL3cV72aXydleZXUb3AzBmsc+1vq9e/ran5zhouGn6Ql+evG3t08+Wgso391LH3Sqhutkk1R7iXxxrmjzDhqzkCe0eDdJwGghq20k1AdD2y3dyuiLOx5MFiaFrlNwdwfMJxNL2KL0SFukol0Si6ae2FNnNbolr8CWuyoBALpEHrnOrtnWQ//GR8PEBRMlQ3GfO9sqHp6r/Qh/BNMimK/io4dW4kw+JzdXfkxgsT1ftQfEjPbk7kMD10Hi4CpI67R/p0cCh/YWWGeOuLsSjkYEABKQkkKMYFO5aopA/ySs+rgxY0/1GCxzBVOTRaww2LNZ4TpPCQe22QKCS4wiKaMYQzBBQsEKWiuwzokbMFYiGaNtLDsaV3L+4dmA3QTrLceZK6y8pOKb0u3bKEkSzhW5L7H3wmoC8zfmHsi9xa03H/8DqpWPoFXUvnN4X+o3UMAfwDbBBTAGogkM2BJA9K32fK2XGGUBYwAgrt8CFMOoZAA2FvabxiQR/JZRXVIq1K3/rcOX/FtGgh2s31xDHEv0qCXHk/qKcBEMI8XFft8eWnFy7LxKBVBy7ZA0IwEIOfErMtN3xZ3BoIqLUa7LXemhQrmczBYktvAZCUtOLk6ZnphyGKQ7MUU2nzPlyFHoDh96QISiKjEepny4udMnTGppqMdvMKGSkksHu/nk+i2kp30fBsJRPIQGtfgncWmjkXvQwpeqv3SPjmrFQ00CQ3U3lLZSD0IjSr/VqZ1FX/mXjuUYOsp1c8jf8oV74QhnQ33TZDNmIykF23MFu07hBgMa3tKtkhwiMIvzbBHD7DqgoJdYgn69E7mBJ+K/pQyNwOLIZ4RT/WLWa+niD5gc2InINfxG3stimRPCNTXXuB6QQqcepZEo3Cq+WWTseqFgdj0HtimS3O1CUYQYBh2JKBTcAMO9fJNXu3YmSsRXKf8XSexT4LkiCWFVtIkReOQFDIwHYBf56Mv4V0vzsTD3fDD3mo3JiPKVJW+8JEvetzvUPl7jeccjtH18sn0sAPGVr+x08zna4KZXoNZqvXYxKOslZQzbawWaUgojioIkSRfpn6zX6jWGQJRto3ZuAczmizH7waYf+2HOTknreim8m4iP5J3A16n8SlkcaDkW3zWh6WEHau/FutJ4hudJo4GwsB/nbIJbAQ4Dn6cDmSn6kcnccG254yM9B4stKAzgU0LxB3Y+Ho/rlwJr4/2PbHIxbbQgenhTxKmDS/DxZGpD1I3IHsdRxXIeoICZ9kQcPU9gQq617Vgwb+phPh4J0DD9kUjnmgyacQEHuSgm26qnrtGQxNFq1YX2O3JtRWpdNdUsBcKc8iuR4aUmlKUAo6Ba4WueBst1Lcya5eZaevDLtItjZFCkHgpY/3I8mI4Xj9k6A5NTIBJiHq/A0B4ydxKsR8DlyPzAPeGqi9e1iNKKdoysHRhzcOF8GBRXzlvNIMa2QMRw8ZstyO1efbFYLWAoR13r0Yxh1ChSXYRah6ao2zpIcQPiN5yI5UBdtI8ginXNeAMyoBBN6omrMwQJrklU8GpbgNkgEwW0R/sigO3iBjGjvZHyAnofVdFltLfqrJNXbFMt5pfOFTmWzvCpQ7YjQeuTZXAzJEMC3758/uSZsxhg9pDSaGL/n6PhP6D0q/xIeNFnIbig6SJoq/Y3Rb8ase/mXXUl4qcIEuDD99sUgxDPpcmJKWCRBqgdqIp9bcC4ewz6HYMEX9VAygxFtV/wNUkYRpxycLywPiFlVnqp7S+BgYB1CHOVE/TKVTA9PXOJXGxUA+0LB5eUiotf8nUAS7R0fit+wzgyfEjB0/VW/HOUXIE351oGkRwGyKJwsy2iTpq8Bjc6Tng0SsCj3uSAB2gJHQwR9JohMQSOihiU3he+7zdzB4wY04sh8DKvah6hR/hvM2lG3EPcQRFJe5uWmHiCyzETf5ANnI86gyIy+JInsC5AJBA2RBOZ+lb5HvA5ctIfacmLCjQQTUktIaLs4JKV2fqGu30wMXs9yoix13yTFbvXWYqlUgI1x0H7QKbZMamer7egPVQqXKfg70uKUzLco1Q1wkXp9n3UN77vlnwdD0SF2Awzuxj0wUzBDBEfMFKqYGTMpGNpBmLWsUf9BuKz6Ce/CGU8Z7IB/ZItXPGrT36yqwZgR2zCL/o2+LLKcp/fYFhirhPX3nN84jbapjebtY+UmCFcO5nZQJoAzQyAH+nToT5xIXxWSYc9TYHPDFIIVx4dOUy1NKSM2PBgx++w4zYKHKX4BRHxEWxPfnATJGvcrJsxF8HNn9CFkbz8FmM9mOIhA0L1k9keud6N4izDpzaGFY4hLaKBrtm0Ig1mdLmqu/hYdANCkYDaMnt71MvfkABobajQ9GVVxpxtktTVzDUQo2E0DPlo7D06NfkHNV0ThG3pdtMKa1JUIQjp/zUPYItG8Oznn0i7U8EKZg5WwDROv5tRMObVHH8kUN5HWSCOL3WkpIavoLkWbEvgOhcKE04VSYOCVIKfW+5ZtWJnU0XWeeQ7lKmeHWOkZyRsr0CBuHYcCHRshCWBc3ZpNHxG4/3Mq5f0+qfdqzTin91EuEgJbo0FKme3q89TLAYDTbmw8UAdJut23IZJYv7crtA9R+Fp6BOUIAyqdgQL8UeUDwG7dL6tpQRa6K2m+YOTEwTBGUrSdA/XuQqtoBiwBpK2m0obMJY4bGEY3H0nmAjkTIzaPwgX8UTYFOjGMQ6j0XLj7sPZyzGMOGdZicoAvrj9B/Wl4kcdeyRQWHL5c74Vm6UItD4YVhsAWNUuGNM/MJWAOfTTnz1oJME53887hntQf5O3aiO36wdr2HIlpB7u86TB7uEhMDV6e1a0Hnk/CHujRZUcfHYbD4GbwOohMnQTtSmJt0GC6nt82u/1Wu3EHm3t/mKzdkV8YS6VAJgtARA5Jcegvw+OiJ0a/CeJ+Jx+4fZu9fyLMJ/AS1R21WP2a5JG2W3Jfn327DVMf0fKBcufS14XHxrVob0W9WsTrEyDvFxlFRGQdBimey7J7AJDbDH7p80K9tW5yrdC/Yu04wzXxJE1IDMmaz8cLM70xfYrgkmqzhHaICHvOqw9uV/t0nBVgLX6ZY8poGt0563OGISR21LdrBuKrDq9B4hqtRcG1sWqcr467ylqc4hINRY1ueong72d1Kh1H/Wg0YVEQ9rj9f5urY0tQY3utZlfGwOqR20O2B2EsWEDbwQ4qd2hxaddTbe/M1kRZV6TR8+adSiCPvvRQOABjuZbVYuLhcsERtTqltImIx8T19PM+kiuFy+0gIVrEEotXbaF0NRCZAy0JE66Cnv1yd8zsEkR/2xL5SwidFJiQlxkoPEf6UrhgvhqFpQiBKTq6sva5RO1B798fDE8ZyJwKIwYUepfYO425UGx3NVV/yKOJyr/eRpRNhadPTkRY1wMDzfdXtP4pVMF1MEygY1YlaQ6HssoB5Li6OOCWkUlTYkGnL0L7t0l+OPjOiaCXdDEoK6Wblvz1MWnHqXruVsn64F5TKIdttxj54kYQzFM2JwM8A42+K64ewyIhLCAAKFdoiFnSEQXE/uhRtEMuAKpUS1Ri+/ZpGfbnPjW7EeLhF8d5gi2oeeXov+M2gOURb9NBXBb9pKBus2N0YGezaGbm9oTxUAYyllzKrWXnCRPpYC0cSzBxe9lBtuSOi1CB0Aa4MItZS1Yma05+Jm3K56yQNCdTnIAmDhZr0u2zT32LKMNE4lfcTqT0oCWb1MMUJELJc+kEKuveUijBOpwy45mQZWuYqWJkvaGCx4jlb7pZULlNRkwF2k0FE/7bDRiU9tmIJuYqpCp/sMtnEvvu3/593/8659//Pc///i/i8vfnL/+2z/+z7/943/+tujfLY7+E+ym0GbwePbnH//jzz/+959//K+ONh0mZAhskURBJZympOkRyQkMFHv0RWgTtFUA+5ArURwIoYLuQ+SgRa/DutMDdZt3gpt1q8vhZNHZcAlLd93YXD08P5JGnbw9gxcLo/BG6Qp43DMUgACEIj+pRUU8Q0w6hIJNDF8eOHlJB9nmTHWaLoBggBD8q+E01Y7q921CVw8xw+T5pQKiobcLKugcU7YtLQSVqmyxHoiU5gHEU3XHaAewLxhzw0kHN+HxLjwQoZZBdbsEgLNF/xBBGoRBSN+kj7vZ42FQ59ZaHqB0e2IzmNkC9hqEfB+/NswE2qopp4WluKV7G9xwGQKV9cu0TQ9YmtNj2q/hd71Tvwfr84aLE05Uzds65KY2blX9i8oKh8F6SHlord6pSVfKKBdqqXGNBkyPlDl0NHWVzJFRv1nTFsLGPcv6xSdemOU7yW9g8uExjjQH2yylykwcEpiLAII/RYWA8JoCnqqkVS7IjBot2NGc4JivhgKh+t3lbKb4rGsBRPwXLT1zAWR9oYy22ktQbYEolyJUXTtM1G6hrah3hQgbwwLyK14M8chmuE7yXCxDkW2vVlSMsW6VaF+ts2WwZldBktaLkxXJFWYpld9Lw7lAHFwp+BUsBf/0paLDznItJ964ng8b2ZBInduwqTAJ+vQai8mOCOgAfDurAxjr1nfw6rAdPBbNHTk0PJCf1EkgFCTwSAQBXPGrlOeBgGxgvFHiWQqBrJTWlmtdE6eCvhLC5XhR23ziEaoy9XIyM/xa0RNjJWS7fq/aD4lwckQZ5x3Il1auUoRT9XREaQythJoJetL7Z7JHuEVGWsiMNSFaiSSVL6zJmPJbo45yS8BMGbbcjIb4/vOSWe8fEnd4D8oInbQU/m+H6wRelzMhr0Aj8V2LL8bGhoQMbhyIuRZ7ejwQD2UHG4v+gb1wD4YCIVwRD4+uli4tWsd0xbh9ixOor+SE5TZZg6e9DmSaMZh1HGQTOWXpmoFr5RvaB78ab6k+zd8gy2iFgxLS0y6cHbqRbEFnJ0WG0hNngLYVnr+W8XWYPPBJ5Yewg4Aj7oNu2qHIIkDM/OOzJPrck/OMfXEsgUwIgS+miHOO+zjuCy/fvusJI/op2vOYI3xxyjZcHGYG6mWYXFVFYer4/oitMlrhiTz24MntCCxBtMJBY9gDyy/K5cGyGFlMM5dWm6Bee6MfMbcLc9Y67WO6rEKiWpU7VHzjnoskooHDUWcl0JEqgVIKWroW86avbrnp0hMU1yig/UEmI/Zsz43MNvJDEuW4LLR69AUQyoNvXAlR1uUDnxJwrWWo+0AOjPC+JLkYt1RPrcqfFRboEf27Wcq9VGsCq0ZwFmbW6JoTAg22cxVYQwAavXyl83ANXAnouznaph7Os09FDq6t8LrLcNrpm7+wFwXnX7g49y+vG9CRVXXKEVRE8deyeV+B1wCF2fxtjpUwwYb9MBmTq49N6QBliIEAKpqizvOxdzxA4y1coUPUACWKZPR1CnQrAF7jkIGuTsItWKXgBYPjjDVPPJLaDh2iESk0r+FKKLwbwkVsrR+M7hGtI8Uto5rTD2hgHe9UWmDetS5HpmSaQ5g8/bDpoLBOhbC2ByfVRSrIOGqgVA5K/9np6fGZkPh6BPHV4MNBl84SUC0GlJB/aGozDFjKb0d0NtxSTOD7TKf3hKBae6wMqAF/iwJavFxERqemU2n+mEiP5EEjqqkRi/CYlXipBk91pGNUK3Fnz54r6yaVeLpyWkNzxkOMJhzVm4aKepo9f9yjS02yfI8QvpkuL0godI5AHomUtOk4scWCuOIi2vP0xWuMdm7yx3RbScDKDQCCl2bcEo1PTBAppSxON+0hF6lgFcOw2nxto05F8zOpo9uvldaEJrUCbbeqRU2eNJxJzutoauzYa6yiqHvotezo1SEKiHT7aUfflEq46kO8M4spOtobMgiNTYm02uokgmW5EfGVM5gFkTgtXbp7rDaxF2GNlahHNGuXmu5hw0fEMPs+y0zZZMBcA/Ys+fgQ+w2zLDTbBOxV3aFxbZZqvYrxm6+ue9HtjdPtIr+lehA18GCAbksnJNR7OnTeU7GA9tUhHpgCWJnoioMVTp9Kiuj2Luf+4jc6d0PCKQ63R5yur0FlZt4DVhf6qRMdtJUYX0ES6UyFPqRrE8J1rGu86CSjeQOAqO3zJ9PTMdZem3cBDOoqz3njRpF+s/hPoQN2SRwnn4kSFqxDaZGPxn1TLAgxV1yKA3TqLq3hOtjh1Qt4W4EJ1hhZJRZqY0rxcgnsIw6xaG5ysWpQszqWQ4bbsso2Eojg2Tmyq2vU8YAkYWiJF3NgYVuTRclmPhlPTwaNMpq8Ws2nU/sp1jWV88mZ/TSO/Q0dULYf0ykYhH466XqzCcprmENExzCpgtdq8/u1nwLGc7T/7DdoyvpAVuDheQPrnPtBVaU+4tnRk94tgxBoGc1FxtbpaEGYyXr3eSPhWaOfbjc+3yxhV5rr1bLbgYPlC+qO9Yu+/ojv0dL0r2/xvNS8kYaGPSBGvdleHAccMxmegfewzTZe3yapft3RHaYIQ2PhawoKldqMm+lscQzEOGcBzcxLcuzm9VToQPJMntgyiHunPwKHAtfXk3bkKXTQ6l6G5wjoq9FekJccFxCOuRYTQ5SxiA1lpdaMJCfG5XWqelXcbmF30nS/5juvoOAtJlWTz67DN4G8pcPpz8SZMevQK/Qwz7wiRFXAbVpP0EyfCMYC24ruPnDwlIzE5q5nwBQNFVU9+lDFlfgQBmvuxXQwrt0Cv1VFdgV7uHdLJSlmdkkgmGe5S4jT2TStN+jMNBUEch93R5c+0v4I34wKGip7xnfim6g7kWWGFIyZnJHNiOfn6mpO83gila/2RUzT6HY8tZSYQIoDeV0sBpX1wfOgrigWESD61zgfUZ8WNGNwePDZFzdaivd0Pm9uNsc6bOOmFrzAwH7Zvral13ajjKEAvL3Dde5yyNkBrNlI3CoDKzmcnlyvvuhD4wqdelOzb7rq99qWqzXH/08k2tN+MCryajvAgWwFD20Vf4XXt4GEkIhqavWtLi1eFGxC2OrJoa8HqlC6/urqGhkJ8XEnFlzbwcMaNTWiZrTAqhVKULCMdri7ePSwPrpG3zy5mw1ED09rafWEVDpZW0BxCsU5dNvYgLR0/6DtZSvcX1JdDyjMMBkiB3pgKuOxOn6pr1wNVexP3Ns5mp6eDQFFZb853Va4HpeW31LL2E0/aGxfxlFeCpda2thuKs75ilambdNqBasur1GQIA3u6WwtOUHDlnzTb+1MqkRnE2BQ8d4L0OokB2k1CmvQPZE5L2JfVje4xrnorlPRFHdDDSTOEMdMnyauD5+J4+j6RPH+OyyGJd4uYQI2T/fWxTm17y/CRl6c4J3Da7dwfvvwvSOC330rjzihU35GXzzoOR4fsItt61BYyToAhGIm6zgoj43znYw9D2BibK0SDt1Q3fIYiRqGx+wKcPmq0bhrMqx1Lk0ckjYO1PSM+nWw3fIdXtCa5ubj+tJjurs1Nl+R8vmPdP3wGmYPVhSjIfU1jtIay7P1TjfF+5m9YFtlNa7woGecUNfn5XEP2wRpEsNCeHhJitP3+GfgcrBW6OxiurPae5jxdB057BFeOuz0W1rqBVDsOUHpXP234iZWAAeGCdZWB2vk4Z26c6SkW1FBiBKgDl5K/NVE4e4xXm6TUTgnBgqsmHXbWIMFNuKUFHC3dWrKlYeNtOFgnZOWQmOW2VtU2FzDvy5wI96ZIA8bEtn87LpRVowZDNhrEIMyJll1Wwf0ldniSJMH7Yz1bbAr/WnUAU5l6Dc8AMUEw84ndjligfuJYZo3cmpBKChCp1evwsgti0E7sdIY0eI1V8zKDO6NRhIwTbb+1veCEufmduUglTpB/WGULVhw4dW5OtUEMOrDttTWQz1175mmd/X9yaLvQN6jrC/QXfLqlvNU1hOp25XN62zwbLqRyy9/L0ROH5eAFqMxzbOTPjs6YtO+HVCWh+gnfHh2H9b1zdgUE45jLGq7wao5uiG6smpaRaJMG9xoU5FKc41Sjr63TUtQufwLd8dtrKyT/s0ipnqA+uMRXWuhrwYYIRQbp3o3675eoskGRsYiEk6B63hobf77P/7VaV7Vo7NUGJ1xep3PmdNwGdGIUBZ+sGzHm/6ZmBOBwlva740xtexxFdzxvzWmU5sZ27yOtXeaG6pwkUdWBJuCuvfbKWbO0ciUB8Im0LlwnQbvTACi57xGq+teXJu4aRBU9QHshxuyec+BnWEYYdzC4G1ZXCHTjo1bEw53lZFtZRvZFTCX+WUz9L0QhaB0MgpJpipJ6iIS8ySJMNN10sUa7Khxm8X3Fp6UixlZOrJnVQ7y2/b5hVaY3hywYSbX2OnWNsKN9kpXNpuJ9KKex6DpDGCOGnpRzSJSrGmvNxicLHbrSaN9i80wr9J81pysuM4yKzBC9eLUSDOJlMdjSiEhZ+tMmsoo1d4TeVuPwWuCr1iFCzBDYphSXfCPRVXN2Qnq+iW42eDPyiSPPEzTdZwiL5CH6SK7CFR76X7VMxYzxbW/G9CfR0jxckgsdduWq4YFocy9NpeIk/GzhoJrIU7mtF/fFTxr3pnQDD2Sw2W2byjHTjfNaG7HE1rcJ+7BgXZfr6VP129cFHczkNdRkOt0IyJqA3Hhzg24YW7QVxG6u2bQU2TwSLU1plW7HfBef2nzJK6KXp/me2B8fIn833glzE2c2OWiFVtFUxbeSOvWq8+H9bs4R6243v4OOQDd3bzbAkgk3EyDB1UbzXVYf46XDvgBeCyJvPAVHzauV7TrPNS1shSg6Tynpo/XqjDQAzYvEaCoAxNGfY6VPjRThTJJ2JaXS0eFDrC+WH7uaCVyUAJJqYcWezc8cy69A+ZWOwQlKkw+ycwbZbsDWKLVhqP2CbCisxRlL/IWj5jjH7Xg6CLiCY4IjF4QBM7KXQqtyqT0GteL+Nps0xUW5N+qGHNdotXXNVrt+0PacMA7IY6/HC8uO9PPi3uPf7VTy9pexjK2epYyCIY3hPCilA6ylbNUenz44eOLjypati//T3/iAv1pl0oy5niR1pG5h+GlmOHc0X9dR5UsYA5yC+I6F1uf08eYQo5xeV427G1UUPIPnNSl/TBQRxW/3F8fJg2mR2eUkXXB4wQM7xrgeJhEPO51HQdHuf88oPVEZDldUYY1moIe3bX6gorkqySpYg7kLffA0WkdJRI1SOI+sSP2Vd4+PMO6V31L8QzvGaOLimfs5O7STCUt9o6hl/fg2Wix9gZCg4PNiSVi50V9mTFnX+Xv79nkbiALT74SLemRcxig4CMc/56GdCNFV8bS4m4maj5vfc2O98wHZDu6CUikoZ975Ouiiu7bFVqsYw7nbXMURXfSPzzuKsuuayVkpLDBwbwCq4sXPny/xbOq2NI1UTx8aP/eCxWoUF/J8wDT53+XsSuFjljPh111gCpvXjtPD+pCt1qhSF3qsqHFw7oq21J2b9YTPRAKSdtcSt3Dupi3ks33XZR4cMzbYOcrAxoUGo/n5r2K93bHAk1To30vtBXeH4QFIuPjwQPvaeA+3l8+F5dlphl9u7/vPXdpYF3b+gF8h7wsc9AHrsPA/Khmy96B6zhqPpab9mShNuQO5jiMXldBEDpMwmbDi6dS0FX1xWYEVBxNcPbPZEOZ3hrLy9lAIrenvG3BoMVsIeJubh93v822ooMQsANM9g8kLi7VWTyR9nJhfLp3C4njDg/0f0DkbN+Q4vcRc0XozASwfzxRWyktQ4LghfkWZixCfX3zyNY+FjGU2AZNYwKz75qPuoZdGHmaTZqxj/2T7uIQOzuZlPTXVjA9GW9L8TcoWR7Q5dD4R/Ds2x47sGxGhg2Mu+LD92H3OrPoxN4Eb0av0lhQag8W1soY5wT180EdujoYfsIjLd1DkNGmCk+NKfb23KXT3Fg7Ja3lPvY6tZSMf5lHpprRrW7EybczY1mH8hL78xN2HGl/qqKJtJ2zkGeGbISMtEVnNqPOX+zlIZ3Y6D1sE1C38ll5V3Hyq9c+1CouSfTpsk4U+yU43TaAYSPwhhkUu0nXujSAiDseDSh7Dtk+wBPTiT7xx0frqx/pb3viE/yrefg3ML/aQ96J+C+8wWrUbcm+2kjeIV9H2xD/QJ+gx1ebPjPv7D/f9R/j3wiV5rUawHng2kgeoStXsNgQnMQvgbyjzTh+WrOScSV0pwA8QMLMP6LQis3EKtVq+Auz8XF0RwaNekpf7igZ2/boYhG6cY1xBqxLFgZM/vkQMLIAlC+/dUh1GmcinYmfTMCdGgAbedrzbzAeuZDQQCCilISF2INZ0cGju2LJ1fYi3Xz6Ox3EdR0aXMduVMxt0Vnhv7/SX8fzAqqwxDCkQRaj6GC/1ejQX+qamS74gcZtDsUwYuvhIQiwBkn48EB/IwIpF1H3NZb5QMfmwSHVUT0/0NWWdTrVYD440FMflTGG3ZMtswIqB2CCIs7WYGbKW6HUH7ZxZp0HvoV637f8d70H7Bo6wCiivwtyPe9VL/9BUVwrSttybRSe4polLKxK8F5Z/DMKvk/3k/k+lln5viyutxxyIekfduB6bp5/TiqXKrJkYr15LRvGMuDZrJE3gc3o/fu372cM9pzwzkENuOZzvIO3rMD2N/5ISWu0SV9EDaFj7/8BPYQlcqt9AAA="
  },
  "emotivoice_samples.py": {
    "sha256": "d896d9e96a308b433d30f3e2391ae30479ebffd83a8b1271d22cc133c3bffd21",
    "gzip_base64": "H4sIAAAAAAAC/719a5PbRpLgd/6KMny7BiUSJFuSx6ZMz8qyZGvOYykk2ROx7Q4ESIAkLBLA4NEtuq/vt18+6gmAbGo8e45wiwDqkVWVla/KzPr8s0lTlZNlmk2S7FoUh3qbZ48Gnue9q6MsjnZ5logX+7xOf83TVSKqaF/sklLcpPVWrKLVNolFAVWSfSLipozqNM8qATVF8jFa1fC3yMta1Ok+zTbBYPBLlYg31Il4FMxm3E69TUS+XqerNNqJMinyKq3z8vBFBQ//bNISGs/qShS7phLVKi0OgXiOgA22dV1U88lkA600y2CV7ydZUidRlYwPeRNH+cSAHkAhAgxgXn0QeVMPnny9nq4ffRknj+Pl7OKriyfrx3+ZPonjx8n0L19N4798HcdfLpfTWSC+z2+yXR7F4rvX738UmyRLYKh5Se1V9WGXiCRb5XFSDqj1Ik8R4Gi1yssYBi7qXHwKrJOb9EM6eVMmdRmlWRKP99D2rgoG72Gm6vxDkqV/JOXkXbr/7sXb94K+ihQ63FW5mrNY+HGyjppdLf4RZZv/Tv7WZJMq3S+Tsh4vsdvVFtqukiEsy4uPtK7zgZAoIBIE5hqBCXnNq6A4iPEYl0dMiqjeTmobZPEb1BVQQE/O2EyFqbAJp9PZ4yn8pyvQ/PUWNu/C2ZePHj+aYekiiT4AAn49nf1FN1GU+b6oxc9JAxO241IwAbNgCr/r5GM9XqewRllUMoYG9ccavgAODgamSZxAsYuWOJeZiKM6msiVwSYmstjFSGQ5YhJM0TKJaXnTLE4+BqapMb1A5N8BUte7AyBdnldJ1VMN/laABnHA2yyHRcIJGfCYoEaZiKiAp4/pPqoTudirPKvLHFBCvKOhAuwprhLuFHhcHkQSpzWh3hbbgqcVfoC5TDYw1u6eXSbrHLr6IWqqKgUwm4LWHZoYwbcVvE6oLXhfl0m0h6bWSQlonwhcL5FuMqgPYCDMu2IbAeSbBuEJxBvuHzZMkdar7QRRZHMYVKtol/AIswQA2cH8RdCeGhzPdJXsgRxkyST+zgx7MHhdINy02skeuqvSavx7BZjLy5hWNdAUceuttk32wZuL6Uh4ctTwdDEaeGrsIcEBL2fBBRZCEK13M3jHANsF7wLxKoNBJUikYCIGsMWRjsVpVeyiQxJP9lGWrpMKZ51gUHMOw0qz1a4hFFjmDVDZ8sC7Wu5wtW8txHRI5B6gELCmVVLzPvD/lm8z8SusGlLtIVPVaJDJ/cAUihEqEM8YYFq5fzz7Fah4htO8IlIfCdrz0F+ZNxskzdB5oQnRwCBvHS13yVMR8VMEK3ijAZYkiMHWCENtYOGbvPyw3uU3weBNk63qhlZhHN0gItBUwRRVW+QbSbxJxDqKcdKQ1t5sc6AVZaLoKvYnorpOMm5kEAH4saC/CJ/4PacuJ6tdWhQEaFmna+BMFU0ScAFA1jop5U7R+wGYTZQBlj1/84uafrUqCMemicoIthIMEQhxLn6AYr++ffZ3YEy/fP8Ma+QKPbG4mg+gENEOBj5usmvoc50CaV8BhRkvGxhoPafNAzheJeU1gHGNvPJ9Xq5gMaHailgOF4UuBkSGxDYqY0YjAWDHJdQqJ7hPkGbB/0lJUHB93OmBeJ0BRcKRrZsdEMYdbJgBrBHsUGoN14EoWJHAHyDI3yyeiB++M4OmDX2dwoKXTcaTwxQS8WwyMHOotj1MB3ZJ3BdlBq6DUyyA6Iib6BqJz15Uhww+wE6Gecx3MQAFy9UUvPQRFChkvyxPMOlAjlQybdMg7w5Pia4CuEDRAT1Wh4HE82ViBhJrJFhCke0+Kj8gIgDVB9aOiLnTEwK4gFtTs9oYeHcJQACdSVfIL5Jr3JnweS4yhCmJaiy1S5fIDhMY/haGKVIELwU6MoiAAuQ0fv0O136dbhRNhlnYAutGkADeOIcJApgr3HMVkEagNbDWICsdAhTYBoM17HARhuumbsokDJEj4CaicXOLg4F6V26KqKwS9Qz0cwugqkf+B14ETZ3u1Fskr+o38KKt+p1X6lepG6yaJRAcII/6W3XQP2H5EoZWIir0pKCVb4B6RsCouBTy4tUuqpCDymL6FZdAFmQ18gahow/1gba9fP8sO4zEq5oFlMHg3etf3j5/Eb598eurd69e/ywWwjtXJvQGPz97/+pXqPzs/QuoOPsSZJrBYABUQtDMhjDFlQ9/rufEjS5hh12J/yN+RjK7oH+GYvytXong5wh4A/K/OUk19LKEkrrAM8lP39AXn0rhf0AdV2VK9GYRhnG+CsMR4tUeCWMZ0iwtdCtvo5vvTYUfk13xUhWlFodW7wFgWqjYuO+Nx4yf3ginNVngLI/EFppYeH97B/MnSWT1VDz/6ZXIATtLwG14/JAcKoESBJApQNkVSgoe94SLDaO0Otw3uPN2hzD5CGyyAiQPN8CNCt9U6ACGL717vpMQaMN+cqhIVvoG+ibNgBEaZlwBsUCZBWVVoCTeyUb7hOOzIWoLymdXJKI1PkT7Xd+AvpeLhkoKDnrCizxB1lnLh4AqH+9E4yL0RgoG7kdvZKEo9bHwTqsiVg2G7accqaLSc2LgochODgKI5o/NhijjS5QYmSy/+l42cB6gyBF3hBTVGPmT1TsQf9xNetN8l4PcEWVK5nxGn7vDewkKWNIew/c5ibGxUiHlaCZajxMEggO6EqQ+ZV/IOl204few9ApMpD1q9V+Qmq76I+3nqRZ1SNC0NS7vrK5Y/VG4Blh0EjtZLrXA86Qe5ykY35FckbB6hAwCVZucRElr39+33KQTKqBA/Ixq0yVoiqqzH9PNlmXudQTCe/lU60xarkE1gIR3VpliEEdRGvXup5+kW4y1XDRmheIoTCfbYp3k39GYPUsg0oylojJel8iSrG2h1rOL+bM21v8dRKN9s4edlDckInFjJO/sko8k7VA/IM6sYZoFq+xyap+KKdCiqiaNUisQywREqDQvP2mXO8phL0GXoyWkBRDUMk9sXZUAS0iprs5Y5330cUyaDPyFEtZWMEty8fjcJdGNfcKCXIA40l6Tt8nvQD+JL1dAeGKzOJZ6oLBasqoUJ+3+AYO+mERx/0Afn6yZ9daZneaE1k52q108Ot1bDPrDKrGJzapoNKGB38haUCW7nBM+XJ1JXE6pddD8efzE5SP3TzqIG0VTj4EtunitmsEn3+ta8rzh6dmlUmPUWnonWZPKKYp0oImwqkoSsPjxj9MSUIGGpPG+OkKnpoYQv5DWM1EBe0RjxTKpb5IkkyYCUinPXB00I4zRjEAdu7uHAejsn0dBZ/s8zyuQUsgcIaJaJKDJSssOtv+ULTLlNcwIzyBwqiarkZTFaYXWCDSOnbOZNhEQ4Xh5fIrOG3WGcv0ONvoY2fKfl27el03SJSlxgxZA6AAI+RJIi5gGX3+FsiSbYt48/7tQxpdjU3TOnCARvAbyN94cm5cnBnW+T1YpDF388N0IV2qfV7V4chovgdmQfVVKgv/Sjj3ZwU10PQalGCGH5ldb3JDV4tKDCQpnX8Ir+nXxGH+9/On1s/felUWlZCnuIYnKHeixoREPWeP8kIGMafROLpyuuXzAkvy8vawVtIKsMUDhtPLtsgHS9BBFLJ+ON2AJF15Tr8dfKQoiO0D5Nq3Y4LJK1LlDBfCDZDI0XVrTA6phXhqVUuyBEZHuD7gvUEUmhTJfIsfyhi3URZhv+WcA+mU9l+9JupA/oRnZVSgr3elWYM/iVEErVVJrcEEdp2dZ3BmhrHFiKGvvF9kqQY4a71zcVmSa8mX14Z01FIQVSo3EdbRrEjpykIAEaZ3sK781cQAFFMdytx6iOSKK1Mfv3JI9sCmYxC38+ay8Q/mW7YcwYjwnsAAz84zWB56MS6h21QZHAl6xTQONdHpNWH+QneDn8yDkFnFmEM47YWx2WbPbtWC0ITjeDZSyUDOS+9bXGzt8h1Im0ja1p0/v+eGw2wf+B6AAuVsQafIJriECtoQ2OuWTHUAl5wrL0/FNVn9Sw1D+rHaJSn5Cy4BexO2p3l1PF1VyujVrtqnNkQDxfdi3LrhoVKu/wRaCvMq4B0IUGp3Gky5i2JOAiHkGotTloR8OtS1pD+gtEViz1p2lj6ukqIX/Hsq8QPhH4lcsSb+HIqqwxKePurU9gL5AM/boNbEnq9rR4eoReVguJLvYlTRKDhxkMq1RqXOb7Lam27IOte5rSRYN2ZxwCkCn5Cc03W1UTj9QxVCV9h880CyCyiGHbfPeLtfFp0ovBNt7FwvrtZ5R+jYf9HPIdwUIM+sD+1PsDsQb87U810ZtxTJu2ieLjkHW01BR7z1rwOTb+nhqLvshjFwzEsJmXlFrfKC7zOutBVB28EFJiuq6pKkBypMmu3io+QqiO71C2uR70i7raVNq6FhEPbKS2u+GR6eWjbyjY04LOCM97gl85sQ+FnIYCGIGqjlDiBqtZxRiD0TXkPSEUBkDrFdSpbdg7MpT7vRgR0Nc6XqIU9zzUXwjZicllVssdscy1xIPfcnlBpU4aetq4UtrBND+xfT4pLaMH6abWgA7hd8X0zY+ounnG3GiTSqhGsryLEs2pHX2I3YXd4+h9+lOHW+Ok71jP1PxzULOV5qF0ogmFxg/zU7NWcfspvtTau+UxjCb9uKcMm5K7wVtFAyVUVC5MLS/dBEPT/aCtFqnGYihfbhHaEcDDi6e4MB6UXAhHn8KDsIoLrG9kXh89RSIXY1eX9rfA4A64BlelAJNoXPTfzbAFuuDuwIu6LQUZG8I9xUB7bw5vfbKTqEB5EZpDY5iQE//aBoI0WSgYHAxxf6ML0/tK9uKcQQ77J11BCA0LoTxkudjWbVefitmp0CQlom+SVGAjKEBxtQzgEHCglp9uFnakyPaH3FqnpymONI4YCOUPx2JJ1don5fWgDa1IBtEiIYuua61+AqNWGp17AK4gb9Gq+oJemEMZxqMKQ4LGw0Crm4sZAoMtkmKzxaCjJEalDIJ1qD1wLSttn7poUXR/+t8/lv8cPhXb2TXPcHhZOMKnOMWzvugoY9k5gyNmTPEhkLpvdIPg0MDPHKPkVqddsMQdV7jcQA686AXjXQ4CQR6i0qILV+6e6ytukNDJ4+aBAzMLFq3aZllYxixmDboKhnSltNhN9wk8fKsVfY0ZbR6NQRSyn23bkMt4f/frOnqxbfaHbR0PlTwHCV3+C8MLmJxdclgyDGVSd2UGSGedKxQ3nIhHghlm3rr3+RlPCcFE30ptA7teZ7lWoZmR+nxNmKzJArIq3y/h28vss0urbZAC5foXMSHIBOkUWm0qwJ0rKEjz7pMiyJBBRf7DEp64Xu/eb81F9NZTH+//mJ4dWf2E2zgCvQT2r2Xn/0Vi1x8+VvzaDq9+K1Zr6cz/DtbXz38X95Id2BNn5yAC9WeKhIkWVyh55LvBTYDt4eAti1VHDZLUvqs27vbcV8Gll1Yvqq677qv4m7NoszXnZdVt9zv3VdV3Xl13e0zqVfdd8Gm8y4Nks67KNh3AXbfGWOHXH2YQ8RIv0WIgQhfPhv/dzT+4+q3YPjQXruOVdRZEr2buHl3nziLLR9mHSwaPZ0Tpj2mv48IjVaERkv6G7WQiew1QjkmsQxPBl38QxtnJHbpPkXfQ1QoYBNpZyW9lf53khRCOgiiSA9bGDQ62oyVuNmm7I26Rm8f3lfABYEFVuiLDHuumqx2KE2ZzQSjok5dRaWMUoDV2El87zmdrpguuZIiGkppcaUMcv/hzTk81fZ76bWIZDvZF1qU5NMl22drIS6vNCsBchZtyqjY4m7ivoAx1Tv0X7E7RCIhjQRcnsv5LgMB3XeXZETFUPbi8ZFLJLzBHqiZFkXtjGXtPWP/YbI7oYcgulfy0RQ3eUv/2MyCl42ad1sH2gIoQdQVwAcJaup87gDXtWIBe4/TGIWghWrooTDDfMibCvrpNQqa2nI++s1ksK2iD50v3N0IBwGd66b4xUMxa5sJ8fU3GrZqeGIwuJiXvaCwegitszMnGzrUHCVZsycHUO7hcg5dXg2x735mRjaii04/V8fspwa+/nk6A/7/mTHMevu6OrnkR8bQWs3qcjxz2+E9G0QFuv/6nvACdF5zoHVxTe1OLgEF5lc23eX2JNHUKppPmuecnUgt1YBI50js0x3QjATdk4B8kPmc6CkU15T0XVPgEWjFXtu8QxFU440OhGu/p6ALnNIaHePgNxEe8rQ3NJQ9WmFpigMalLNiIIdQ4dHLAl4EUYURLwcGfCRiOjaF9wTco4thsMqLgyRJdBIL1aBzv8Rl9W3t54EzPDERM1BlhkQofO5xKCYTcaFJMTf3rUvf9wV04E+DJ2Is8O8DBHKVVz78AxSUvF9RaYPHIh1xG8PhEMZBxnYLcktGws4v51T2SjxYUDftz2P6PNffL+dzhUNyybmgcuAt8zpZ1SFqJM6qSz1ZLi+aM+UROxWdE2WjRa8bmLpLqoPmlktitFTpyvDVZ0UhDbqbXb4EeQObf6rjGRL21NnmeV0B1DW6pdXbKKPzdCTx+38rMsBq+LOpeAD/yGHCMl9Mla8IjC+UrkELHgmuB2je+A8aEiQayPLsL8XoRN515BYwsdsZMg2zZ5BEJLsrEl+gvj0aWENqvbt8I0u+9axmQrmg6Mw0t9sftYqrAB/6t/2R/NTcpk7OA9e/kziVVqH0e1OmQV/+a/QXRB9DKxKUHVCOKoi3S0Vl8neMyyxRl2X3OXJzl4Q4TTj4suBwrWjjyFwq6Auosxd6RFT4TUCBFlKl+MYjU4xXFZ5QveyTjlZifJSUngbCjBbVgrSiIDDQOpChGBHOtDiEiSHfdPFDA9QwieUAf7h4o+fgbUImP4oWYb9rGC1KlfEXFYZwjvP1mGY1Z2nExFOR5RC6DHgjgFofAoNSUy9AOVRupOhSC7JkwaFrOzRQ4/k7SEgw6YWlRa5zNDtUATX4CsvKnpclaMZbwewHpMlsUxUz0IqbmpqEAakiHAYjy8FernGNqL2myBGLV0l6zfFUICwmFLyFRDEQyDmgLtXhKDl2m4wUeqxhHMtoxTIRzUWVO+CXSbGL0E4RSfSwSUqcgMhOgRIA8DrNlHcmD/a9mf9liv6ZGBmjVoKmOs2ahDzIobEYNv5zduiWKg9vcQ69VEvL64I7IwxRIQpDv0p2a5g2GARyZKS4lkCGHwP5DRUz/mW3Altrp1vR+k2fTkMEjXfPgr75duukGw07grqL20Bq+3Cbqnb8NlBQk/1JIyc22FIrUae8eng5HV9coXmPynMP9FPvnD4ptaMS9MpRaw9REQgY+mzhurRIi6YoeMCMI/msvHuK0YJNJYOY4zIvAH8QvbxOF86MKd/eTMaHV5fTK2virPfAiMnaKLyW4Yg+W0LdsE2CuISkrxHuj1DHdKqTA54JDDKQLByD//Kykk8qJtPS91CSI99leXRD4t2gzdWpOHy4sn4azv4GQ9tysjdJlzecQI5pbHk/y/BftO2R2CUjEfciuokObQZfYzCgxHkdubvg1wFLbThU5uk+snQ5XBD1diD3hMiOp8Ngl2cbKfcpcFQrsKUrQKDLy2MMqw8nYZquyF0NbccLDZoyT1NHGCiJp25kK1W9ZXmG733fGg5onUPxnwqwEcgzIU28dLK7nI/E7CpYFY0/DOqctq9S3aNsQya6o41/01rcf62j7twDsygTX7dkZsBMe6tjq5Bj7tRvR86MjdToJLpX2+jiyZc+xtnMKdyNEBTwl3EwTjdJhYKfDOwLZHnuiuKDsWqQw+byvXLpkVMK+9nPHRe15S5ffcCFRn7kw2iWcTSXJclBEITGi8eAafgPjGrpeS0CxbAETYHKm0/tuSPm79vkI//yUTL4r1YcII7ZpIYI5VcfX1njV6F9l2iEl9uRYkbzBtcKywWrm1hFzBzwNBAPpTDitkbNA8ZFbUryn8MCbuNUvht03IQO6JfAtqGUBAnzSVdV/Rvypfstkz2I+L7VKS8unWuAUk2hZe0xGlcMbLiStl4dbhmUTWbI/6W3ScktYvzcs0YHL6DymGzz+PXHF8++R39TcnNgR1/QtwuKI2Uvc/kSJ51+sscuCI4xfFf2NmP4loABXW/FWLZNcm9BL0v3fVxrjR7gCWUtsBOQcLjdbatZ4FMbYKu3quc7jJ2VeT5s8VF9D9rHRCUIQNcRacCXHqfbwIlRQk5QHOxHkCXbb1bqTV/4HHy5kvuyrA//0nqBiF43BNN4DLxglexASeNH+PtAwX9ltBZrMc27nkU1/vBqcUcnV5fGcP46qohJnS9ioiVHuZhAowQF40kaV1lCB2ai0Od8IooB/qS0Fw9IhY4F9qHlP5KMxjCUygXZkJEfz3V0KJGvgZYxXPYvjSFSOUjikDMzkLxA3/ZR9cE8aXJtXrELhi1nsNc2O2B03ttSBj5zOIWK1TDviRto+UaxZske5kYKYSAlszHyEPIhGE1PeXk2kGfXEi1CNqP6/I8yb4HUuFNCFKhWGciOVRKzzQMwmnmmcwr3ISmzZMenETGKQmzCQpPITh1S+dahS629Tbgz6UsCkuOw5e7CLYcYxcQ+X9gBub9wF/hTdSLlRo2+FuxtgdJFXDl+MRYzlKS4E3i6gAfZj3hoxvJA+AwWV3hIA+KBBHJeZS0akG2Ndtihr23mdkfjox3Rn8lEwYftSi6SU0oihUm+hWgj9ZG1rM7ZKYj50QZtD4gV+a6hfhmuysoNkgi9M3WQ3o/py/SHZz+zeKHEV32e0INlDJaGKKAiwDTN8b00R8uMNIlrklbV4Ksld5zqUB0SqPZsdJako1cAUj2BQE/vqku2mj/QXwCokJemEnPha6P6sL/MVUdRBEpI+E/tMy+5rmZej8ZHmid8nY3onwsE8A+g0lQz4HojYT1dHPFbT3u3/olpo16HstshsnhJJ44eLvRwCO+XrGoKmbgDJjSN0aTAUy3XjnlBy7W73/1cTYbCVJgLa+SfMm57kLK1/19DVGcPx4Fyd0he1caE9gLThMiZ6bGpoA/FvCfVhOv9Ry4+xIjJNbkK0VzljvlzseW4e2BjSbhtloLcXDlPUFQLK8HHU0Fx2y84MdoXFLFAyWhaDT5r6vzvKqkVEJGGEkXFecLuNJhBDg1Jog2YyS/lNAhyd5JdpyBiXHo/vgx//OW78PXLlz+9+vkFOXt7MyM8OEq1xiXKGoIEAVNoJKXOOIJwvlcx+27xQ7SqrPwlGM8EIg2+Dflx4JqxqEelO7qfpG85Ob04HyjxmvRLw99IhvIdaA9mg7hqg65lCrCuir7tSIWki7J0SOeHodupdAZTWi4/+rbnWcs+Ry5Y4fLA543IATuufKAphtPpVP3fakEnOgrp8Fc3NbUR1YKNQzzQoo0+YF6v6Y2Bx+9BWoXRdQScdLlL/KOGtOPCrOvIdhNxikLO8rUkVQP2gUraRD5TO8m1UcjNMCpS9s6ZomQ88glbmlk3GgAHJtAyWLMwbIU54LuQveoWdu2Nro3nGAUovSkIBnZDAdUK96CZloeWgwqFbkKDf/nyK6nvy3+ccieXEQ0hHUwZORCjRCP7ai/miaa/WdjevJ+8nGjhNitJS6MTbYlqD30qS3lNJQm8WCUrmmBGM7bK7wHfQS/7tAUt+FwJ1UE582glokMm/8SYJ868jcRRhLC6o6xxdNamekLFsvLvqStN4xU2kmWBTpelBH/jV4KmpR5DjaFG7ioR/dQKoaShm4sC9GlQso8XhY+q9A8XxYjYUCgPZLq1ZP5MTjUS1jVMcx6T4Br8ntSadv/txft3P6hQkE9qRWazUQ3ZnG/guioQPwirIlkhgXTybGFswirErojNhSonQxdn7aB+nZPJsImJjgzFn2SJ8MxLtEYMjuMmC0YaSBWGwyKWek3RwhxB1B9N1ycQPWe/X8V27ZSGloXHyYHWEv8kBKxVdeeP3/MMIpS+BfHw2DLIoQTJR5h8bsF3+mnVXG9YrTAFguf05HcKBogTIaXHlJxbv2gJQTbCGDmosoQjJT3tDoEgh7xnP/0kKC2SScQJbOb5m19ceYjjoWR6WgDD7skHGIfSwI3561oj4KpIH4wIgJPFPL0daGUEkhFQwUJjr0qxcZOkm23NsltLzeoPe7K6l5HkdC5Nu9BTftVWofOQUBsVOSumFdglo89VKsgvqKMvBI8f+8fD4vLQRkn0WKZDB4zJlCZddIdMP/qexA9Q4KzYTTfo2xqANDx6VyoA3A2/3adVReaTJkvUIBbu+tLycGMhQuxL6EbSfdPOU9A63kTjiuMHgKgKkDuWFwnCeYLTmhM32VMM5EI2waiLHg57zKSIsfLySzual8gaj5JGx/kecb67ZFE2ETZZA8p8uEWmgJH4Htu7fAX/qFvVTKpTw7xuVbo7BqTZas5zt7hJALZwlYuACJjJutql/y4tGXV0o0WvKneC6GN+N0WojnRHuBliwR7Vo61FmsK9EeTn8ypKPHcCcJI3NPR8ntVJVtF3tmVzAhi0pazxJgLS6HOdLucIshB08WiJjACofBZWh/0y31V9BU0qNyyqog+zkGJkq/ZORI2JKlYl2R2sJJN/SrpF7Zxt50IaJVo5hVX2HocTO7TRYsr3SLecjW/hilM0rBPMxurM4TX9yNgX/fvJDOi0ENTiRqYfmxlpODRDMuX+xIJpJmVSzZ9gVLpQD7O6Z6k+F693MXvpYFpdUlt5nkJ02yHCX7Uy1wp2EcDE3BG+qVyBwyGjMsUKYURg8aVhZ1NdJyUztNu7jsO5yzHNTFxa83/VnzWl7ZB+gkef9kaXy+uMrt8eSAmlM+5O1w+k+5XvBXJ+6ZAikBywlO5Rlfwa5GW6wSPlqTfUVY9aH3Wb12e1OfN6Wxoes5OaAZ03Ae051/X7iC+v+6Uuj0Y6WuouSelKN6q6lm+6ciVxGkWRuJk6P67usrq9J7xVFeG5WwZ0VCjUcV30QRttYzeVV86CC0dVpbN/hxnKD8QDQbEpUwztXm2VmyNeXeD1ta916MUR9dnIHFWv8ESfdOQMH7r0WtrlcQxHnDhnMb3YQFnokPdRhRClKuLVw1PppZyomntoWJ8kxmxW+Xz4Fuu9SON/tft7swzMj0Iiyy56GnDzmbTzc3Sb/ZzcMU3+YVkWrwyRNHqNbsB0MUBUcQ5X536AUas5vi1AwP7ZTpCFZugGQbYFvCxg1VQ1Juo27IdTddGFFugu2moNM2Bx/g8rZBePXd2bBOzLDZpMW0VdVtKbkqd3Wp1VD+it71kZa1uJeYyn5NFcPOe5VKrsyd1UuXQPxRJj4J6SyEWBy6oApW1xs+V2kZyNTgDc4Pjp15+bIRvZ/mfnyWRHc3PE3NogkLtplXBK/8nRC1jOmio7J0fPfHDQlzMn90XZed3rXUywnTHrt27I8FrnGyodrVoRIkLqpUwZIZ9CTMvXqi5l2A4BtwShufINPCooDzvRt5ae2qrftvIM++JleR4P2WoLTAg0WPTgw/f/hbw6XYEsss1jfTbpDtl4MMrk+MZh20TOPMsOV9ZpOaXhdfMWSn+64/kKySkZeyH98/LqRL4dap79es9ACvf2GSePId9D08r5h60T28Re5vdoHVjI0jfwq7rSxrrP5i5Iq6pZoqSNFc5xEPdeYPpSjY0EVEbozFFp2kGbDD/kM9TNsqY7FOM+sNAJx71jp3W9TvtmnbtzAFeExIUdpNEWfFJxYIehDmzD3kyBFGaG7VGut6Hi7cj15MpxErhv+g6bjsyzAlO5PvfkEVJpl6pTQ/i0qewZn0pxgcPAk0Cf1CqdGbxvKmRSQKn73MqoN4D2TmGjm9RFlm/lB+KOuxmBzpg0eUVUK98MCJxLOpm3kgZ1wwNog5kAEZs+IcFqOdN+4tGyOceXx3it1rVyofuXx21svjQQuETuDFiKxuvNHHDLUWqbouGjdjojlBdf4Ut54qqzrsAnDy8BIicPc42RlQrcu2tPaYvpUIc0CaGsBnqh6rc9Q+gDIOdAF3amanS0cXXeek7bquypph2Q3UNV1Xjvx1Yzx2b1F+OwymfwmNTmqXWH0yoqBF3iEANuc0oceZAs72/qu7fJyVShsUqLG9LThxrKrNAnHX7MPpZFU2vZQxug/UtZ7WokVzpkG0+18OhqhrpsMmlMa50gwMxTnv9FjwmcLNQqlzAuEWdsABKmoa6c7DUE3aVH/4ZpXHlXGLdQJBQj9K3q6x6W3NVZ5QUScoyc9UEJb/I2ji8qcSubvxuzZkvrjp7IlOHaZFQgp9PB0cAnj9Vc3K80BEU8eXAtfi8H3KpyZRYIMRy3ZgVo+CHxe2aoQ/a66+B3O8W6i57WRoOOSUDDtTgGrluJwpdpldFTWldyX9uVhpdegYl/Yun2BqvettBodMc7w9oRfiPxYETXgzVFjzcySZLaCdwisMoJy8z0wM0ngOY7N2HKyPhotXMiDjvC6eUVogPDZR16KLGlHV5IwoWUUwDZyeXQZ0CGXbnxXrn0WVtA0imQMAAUZI/kI0wMbgfu0sCPU6z8w828mSwo+tSFsVOfPuNhONneP8H7g89gF2pmbFLGs8y+Fl2piGdeWqLUxnZsUXLqevJOS59/1a0ynPmymZG2740cy92wk8Glm078UonMaxNcbQfJ2VHY6gDbmOV6k2P0Zh4/kQpGicfaHmhiHG5lW3d9tu646oYAWrBdFlc8qCMxf9Z2VbkF1CFOthkeSdTS7RKRXrU+7G292xbSE7WY0Z583PAVu4HIjqQPbdWt/ZET6avaimSq56pchRjuiizLh0kC/P6/2HrQZNU/myQBcW98Mew2a479dEvFh9BwXHfYXQtF/9h7+pGHMN1NJNG5Z8T2cLE4+p79zjkNZv3mGylqRfXxM4jjX3iWT36WQ8eIS4rcC5KPBWig/njGKUY0SozEeDa8py2kKf+eluTc/unGTnyK0/1iPBudeSDTfYPhvtZ6aiVR53k1BNr/OBJt1J31oJSMUNJ7E4PUWfzAzkBjFBOLD1LmWXfLd5K0aNJrR0Ydbd+i/n3JbM/syw23OqOz/gS5Z/Z2j2HHTjnl8PnPFjYrO3GcJnMsHNmdKI8pQy93oOwcV8fgsCtJaUOj831xDvdZe60QzmMyiOKCtxYcdxitWWEQE0vat9bU3HnHd1e/xZkX/nI6sodKGYC0AaRrU+m1hnRQ93SjrlnmaIsugp5u0jHtHGnzeOC4G/qHh1CnciX0URidNmFk3WJtycKdfNdnUC7p7WJC5SvMDEGn+P6QUoidtLHd70WhHbu7FxJP5JAxWeEyzWR6xqpKN3g0RbYQHocVibqM6NAqWW3PPT6XibwXp8bZNy2y3re92oaM4vvTM7OW2RWdTabsDdD7LXd0177BT2vE3UTvdDa8KhNkSZyQ/MyJkhGKePLfG7M4sg/j9YlGD39B1UVlDOntWys0/klSm9hX6h1TII6X+AjyWh2ttr70dzpekvjxPWWsZBKnCzo06r7CLvm5r7Rch6Pf5XodL+CQpONz75KqTxeLek6i2kljEEccqzBdOS7NtiP6Pjd4gglc8g3m7DOhso5J7YQZ4YSW7O5fZNt04Xw37vyks4rUfKgqYdKpwnbSEqph8OpUNUIqS8JsyWRHhEqMXMY+XJQ81Y+8Z9PWxFx57J6eWvh8qquPvU6uXWeXh61Bk7Lk028LCIAh0AHEPmgGPeLhw/aouCl++IS2uuNQ7bJ+G5bJptlFcrYsiYCv/CXNtXOK91EaWTkPE03mMSbT51D/Q9QA48Q8DRxPjbzyOs13aN9nsylQQvuyUWpc3vzXwjW51Vo2lpaKTvftJjRCNyU7bQny81JF6zyEF/7H07N6pNub6Np1A9Ocx4dG+5zUe+nOiR7IPnjN049zj7DNmAQFbhqIPxGquPZUSL4TdRzPQfbGbEe+BmF4N9LufeL2BDB397mV6hM4lToV+kAJd6rSKVGmSjQWWg7OjjJmX5uhs3ISJQDSiDHBZa2FhJ6MGSOVRsNK0Cpr6Tdu2gOVfcFtDKiLVQ32sdWszrxgvZOgwlpz8lRKtomjl4kzzoRKcxj0CLB6BKHYLt/mbdDPQEerVau0OAQo0kY7FXOl4Sry3UEywfQ6RW/eBR/lblaxb/XowDdsZdw0TeEInaI4LbJlZ9DWe53W1ZNaNV6LmRcHdcLEM3lTgqBMnjCWpwiQtqiO5q2z09aBLsjXAA7mglxwMgpkx2HVrNHrl17wb1hUL6j3haRFulrAXZP9Wk80eZzEgLyVjyBgYukK8+lEMNupvP+dHCSzenExEnxfRhapYzPs6zfM2tP2T7FSlOSVdvjVsIxoBGpO9lGa0V1jdpZydp6R15m5qG3fV9a+qOxz8Za95pFUAz3BfvQd0Ei9ZY4e8gCiZEoSpnR3gGKgPyUy3fhNHHQvJjIpVPnerkHXYcm5v2vQckrq/2aiS+y3MmkCXYc8sCIlLW8qddPsffd+UN4Z+4TmnvsDq05t6a9uYhJ0/hlySHUDeIYBncY4zutuCb7Owe+t2+qCjjeloyknuF8L5yo6fUkJHUq519Edd6EaAyXxnB7sTPvtDPz3Zd6nU8He/PsqNTdnlg7WsJlgF/ml99s7vGrBys0pu5uRjx9Xw2uC7Bt6es5HdGhF7YR8z6ZBAFUxL37NuTHGgGN4BAwfqXGZYYx+60s+unmXrdcVJmSkS/4w/mh9JJHjsduIOi+PhQOp3mx6D/s6y/8JJPLl4ynnGWI9ZB9lTbQL8To1X1+9Zm41DMweCvYfKHtdBGpLXcn0b4SmYf7B8rOHFtN1UrlhrqYZdGO3gpVV6YB2oxq/00bPXuBlfAkz+YK+qbV8Td0AlUWsPTB0FElot0ep4HLOIbKGvbIVzs3pMhUgBj42RUj/Im6DTBUUSbkOSU7F61NkEi/MswIFOOEKzaEipP+Iyj2FNU4o4Zw0cMoDpUnXCEWeN08pKYQSjaVoi5YUlezF6jag02+O5uJcovImERnooW/cwD5w4Zgyt3vGn1lSY86CwFMn59aK6rnonwe8JdmeLZNkMbm5d/5I+0YjtTUg2tJuK0d7dvoZyHAjLXT0XNsFM2ltGnlBN14AZ2xOnDfVvRvugV1LpbvvSc9Gqfz2fo+QzKelmAc2lUYH5P5u3w8o2Zcvv6I4OXAiqEwnZ8q+LXm6I7upjdEWnRx/YhmFPBeeiQV0ODNJBKFKpQgFW8kYrbKGf4fsVQyl5dpLV+YTLN+O9zX1OqHAdnccSTpvOap9SA7kKaIvOV733FLMCTaJJx6L1b4GCYq3vAo2My5Z9jice2sN4PbrnuLkEd8pjtdTdKtb7jN84e6cOKP1lg6tWcmeMwOzPloSOXy1ccYSnAzyIFDmyVkxSQj4Xgh0nlNvbFCAwoR65+qi+o2daF/jucRo7trZDv3FdbOd3TM5AjzvRXcDIVj2FrWK6+MSdmcn90m2bEjvXU3yLbtLBMz4gJRd6eA9CfXwpB/oG6ZsJCYgNStKY2i614lP0NWQen8pM77r9N2Y3ZvlIArYVPE3wBboJoG8BBnGzgifJdcINaa+12nkaQxPWSLTOeXRfpFBRXk7GkD4O02xQxUkzCHIZOQF+b0zKIz4NZkq0MkSve/inC+xIYuEPF3BNGKglsnsOngu1SvO0ymr2S3qvUN70AMIirjX3vSkDZA7qAjcPWQwRVr8qcxx878qp0/WuIJ+whToKo90T20lBag7gbHuMbu710XbFv/pqXMyg6i6KkOhVxGcLD7qS75wOucoNXpfoU9p11qW/puNzGJdylNVHb9IXPleaLr3MJ1KAmH8o+iMoafOgwcSXV1ncAsbvLIhtLm8UteHsGCER4gmjMT73pLpkIjSVfCn6N48+HJ9Jyrhd8rdyWIYkyfte9aGW++aamsnBe6Hh2RfA4sR5B6Q6DQPLqD3PdCfVb6LFZOAgppdzINHCN9Tff/arZGM7u6FR+lQmXvjUAtMDq03+Q1M4hwQqB0aPEcnxoQs2ESKxjKrn5ram3S3o0tmVUx8+27BNpgGVMug5agpIy2cuUOCF+keI57dZIzSCsdp8RpQA6owZvuP/UqlgJOT80BY4uYD8cjyq9UG6m5URAY9U3Zm083AuD7qnh/ad36p9GXQt1VkYoM1MA4GaH30PvcAJFyvR1O8t0A2QCazMX7C9+OeAsNTy732fitlCgqE5vIWOru7EreAlFYb8y8RP/8DNocB9m5yawF7hzi7cbcGK2bxwjtr7d1stkglSmR6rHdkzkUnlAbBXIz+sBWL3dXkjRLP+s0RRXxta+K3OjvtfPoovqOGbvHPXXATXXvOXR79WvlxzZyNlcYhL5KpHY5phXzOmLBziuVm3PH0VfqUcfNlZO6E0Vm1zIVI+oSgxzENO1eeAxnzPgwqtEXB7pVZ7ZNSuxEltZvz5cI5TrYq843mi/apAd2NhmEXSUYjpeaH96l4usGR0FlkrzEh8aLnWjN5i7S870s+uVdyWY7tFAe8oFvDkY5QC6i2tjnKEHDNFGgHWsKwzCXgVAKDbXyKGrOa/WzRleTRyY6A+BbIXDCd9eHi6QMxwzknDMKYJx+kU8w+Qdc8RYDN8dyo3Isuy6SboWH7LW4NxHh4hp1CcfxnHnz1H3fH4kSY/Rs7SEsc6GQc6LuUm+tcnoiTwpgZ2YGdbvPeWfNeL2V+S5XJVCe8lIA794zK1EBu3kzrZPfkQQwdvRDFsQjGms9e7PMPia62JoeXKNCOpCmCJkL5wpq/bI0bCxrEX6Y9Z4Z9/KYkbaoiLUjcC71ADQVTdhNe+h2N1IVr1qJH5kipyUBF+qDSkLmG1HuS1rGE9o9nvwrMagZ6mzxjb9v4dpGdNPHUYZLxImG545Jlz6teB6oerQmZhceaXI/UjnZvj60uMjZbnVL0FN5oFm0p8idsj7098pa0VCJ7XY8X1/1ZxY/ZDNpnTUxaoa7z3FNeKxhEG3R2V9Rf8UW3xoMHvNWO6Rvd5VMX7HA966bfMwVOlp5AYMreRbjVOTVDdqeFXiQ6rmndLCqI6Jbc4xwU4+XTKWYkx2OxMOTbB0M8wwxDGcnqZLFg/H93qABhQaiofTruHCrTKOWl8F/RmQftjpF4/U7+MAc9I2cDjez7Vp5jMt/4DT/R52EnuYWaixdv375+S7rVikYIOL3A23uqGnag5YDYgXlmpaT4fzsI4TVkqwAA"
  },
  "styletts2_samples.py": {
    "sha256": "6289d800a2cf4878533bd5adfea2ea321b704ec24bb1ae6223bc5108359ee3f0",
    "gzip_base64": "H4sIAAAAAAAC/71965LbRrPYfz7FfFBSBiQSS3Iv2qVMn5Il2VaOdSlJtiuhWTggMSTxCQRoAFyJ3mzqS37nBZJUKi/ipzl5AL9CunsumMGFu3JVzpZKJAczPT09PT3dPd2DB3872Rf5ySJOT3h6zXaHcpOlpz3Hcd6XYRqFSZZy9r48JPzDh/dj9mO8yGP4xopwu0t4wT7F5YYtw+WGR4x/Dpcli/Z5WMZZynZJmBZ+r/eWYLJTfzQcwH9jlvNltt3yNOKRz94syjBOWbnhLFut4mUcJqzI9vmSs7Bk79/89O7Zi+Ddi59fvn/55vWkx9g6LtmS0NqU5a6YnJxAyWa/8AHoySE5O398daIR9rG2/iVbD54ZIwLMlx+zfcnOlzxaPh4tT09PV5fR5eLy/Hy0DM+vTh9fLqLl8nG4OltenY+vei/TogyThFDO+S4r4jLLD/D1t32ccxhXWfTZDobMt/HvPO8zXux4+HGQrvusWMa7Qx8HmEarOOH93ma/XsfpehUueQCjYEB0tg3L5QYKGQBebk7o/3AfxZnPXhAstt0XJVtwFsXFMrvmebhIOFscelW3TxjQ/Jc4jbJPBSt4yd7+8Ob1i1cv/9OLd8GL929fPP3n4MeX3757+u4/Qi8sLguWxAuNKXv+448+kEvwAyuQXGVZjAM57/7uwAYDHD3zK2pDUck/lwMcGduFRRGuuV9+LtmvAAr/sMmK5zxd8gENiOnf/qfwGp4Dn/R6PxXAWUje1T5JDgyGkCZZGNEgNZPUp3qgeXOZpat4jZTs0ezusjgtkZt+efHy+x8+aG7y2YdNXDCjzh47/iFexd8/haevgY+vocd9uduXPag5PmMff/j9CTD6LstL4HiiBk4UsDy0DHOYEV7yfBuncVHGSxiequKzn7MYmRqwInICmlsYYZ5tKyIwQRTkgTDZbcKTBS/DE1E9ilerfYELC8ZX5lkCa+v9jgMWgFm4g07gK0zlLudRvETsBDNwvSIL4JhVBjjmyH6E04cNdAwAemkGTLILcRyAThITrfk2o5WchAuesCxnCf8cL4H2fAvIFdDv07cvGa1exCEKdzB2WPEvSBBIRBBACmgW7NsX7z70WX00OFic60puiHYZrJwF0D3KgLJpVrJ8n7LvhiCmeL4+mNABMQQADITD28IULLMIEIHR7UFChVAEta9j/qmXwPjS5QFJtuVhsQcghEAsutjBbMSFoCOsrpSH+SDG5Q68gajCqt+EeQQgud8D2kkxGIlBwbJXgxjAHEUxfkO5mAI+yCXUU1ryNc+NSQFgPSlBoY9nb3+iejnfIyJA3OKQwvCA2iBJrnnKPm3gP1wpgG20XwJ1kJjIQKXgrOJEA/ehIgdOBInCB1kKi+lTHpe0umDIKKGf0KKEvgb4Cxl2v4XncSkGGPFVuE9Ktoo/Azbnw4EYMQOhDRIwj7dIOkt4Ax35Iss+fgXw90nBe5+y/GNILPeEBBEICWg8wMYDtY0MkeI511sBrA0oL7maS7kI/d7rTKwRJDBMZTmI+A53EpggBLhFMmcVdUEUlSWW7cKIpgCwRbr6vWebffqxQMYA0lX9FRzBAlFPlkmI4gBw3sBiB9ZP1+VmAPULnl8jKB6tYf2GES96wEYwEbAtxUuACZLm7yBOgPjv9MKGKVjk/ASmrMiiA0mKVXidIf8tDorKsHRw+iVFZeGEgTTP2Pfw4Od3T1/57NlPz5+yNQeJHaIoQHSEoHp7+IA7BbJptgxhAQFb7cSGongdxXtB6woa71NolRcwZ2UGOxqMLYOBF+ycff8tW+9BGAAtYFRhCnPag/WwhcHCulgiYQQasGfkYR6j7BOdkrgsYqhXboAOGhXY6Hc4IWHSU2vm9c+vfoTJgD2CJFhSZLSdomD12VNY1gscGhJoD5Qu2XWcJUJCrIB/QPqBktLrkQANgtW+hAEGAYu3CIIRzmIN9HqqLF+DhAOOlL9BhG1gAOrn34ssVd9hvWzU96xQ33LdtNgvJLV0yUF/LTc5D5HbdEG85QLRHcCFLhWWb7EbelAedsSeovxpeuj1aroPmzLnvjqK06ttddj4NAwfL8LHq8vV+PE5X45OoyUfjy4ux+HF6OzqbHl6Pry84uPQ6b1++uHlz9Dz0w8voOH4bDgcKmx+ePr+hxfvofSGtnRnC6szQXXAmTAnulqdny2uLs9HZ9HobBxF4+HFxenp6PLx5Vl4cXo5PF+MF49Hl6DrPA7PzhaPTy/PTkfLi2i4OruMwpXTF0BfgVxDMbaBnXgdphL6aMmvhuNwFfKzy8vx8ipaLa5WlyH2NuJ8cRbyU34R8cdDqMSvxqMxXy3OsOPLaDgMo8cK+k8lcM/J2x9xOzrZww8Jf3XFz3h0dRmNV8Pz1fDqYgSAw3B8MV5dPo4uLx8vRhcX5xz6Da+uzi/O+OlFdMaXi/EouoIBnQL8216vB8uWEZsFwG+FC/9dT2CdFOWsKPM5+8/sNequU/rw2OAbzZb+6xAk7w6IMyFEd1BJP3uar/eoWr7FX7kLQmeZx7SipkEQZcsg6DPc/MIStuAAZFdRTHXjd+Gn51WDH3iy+05V9URXPohHxJf6cKW2BrQaDIQy5fSRQ/kUGbbPNgBh6vyH98BX2eLvHPb6bKVRBbkFG04qVmqKQ3rCnv34EnaMj5zkLfAwikSHehH9r2GoAoctrGMQGoeAf14moCBc82ANe8fOlRVtRB2hcDrHHpI2auLfOmRHaLP3q2epsPdpUqmYbYRUkpGhrEz0TjpIwgMaJlqv9Xdoa1WgunqjNXlk4jr6YzUF2j9sk84+UgUYEOmrrWo66qpegALY2uJs3NUkAo1tiROnqjrL3d5RY4DvuM8v91E4ed2JpdAaBlGc23RQIPGX6zSsG8frHAdVGKCa3DocQ3J2gdjh7jnYFqr9CkwbA8Kwk4dAdShQ3fjStuZqPt7+XFH3x4ays+M5cDGoTIba02dD3x8PGcAz1nILP4afB9d5uB2sF3d1/JwvQVdNQAHpwx78Od7ut+y8c3otzRZAh2QPVHLv2yxLeJgqfn9Kj6tevwOdg99Jsko5bltLT7WqGWAVWKJKLY+EbyRG3ZSMJq3HoxaMSukdZCt2xqKpEWx0J9pkP3bxiT9S+P/C4/WG5LdSy8JltifbVRhqqCGCFVWZqLNhfzQ/grmJBJqvnTicduMg1OS/jgOKD2VgDoqS74p2cdW52Ph2wclgGBQwdfwvzwLhT7ZKnO7j8iB1EPwzAOqyijiPq0JBpLdowcLM2CYn+yRoB4T+xMEsXAqrRtCnf5RAu7hcbo4PT03QK/gV75KD4Vv4bqicCWShAaWcY8uf8BqAJRi1z8Tp+d2ty+wjT9ubg4Z651RUgAqO1nlxz7kYD+tT8Y4LnQfYEmiSZCgeYe2jqS/pXydN/37LpWEXtw72fKjm5TsyyivRTPY49Gza4GAcCzv8+HIB/MHiWyMrLDfoqCqmrsPTwR5xwC8guz1jLxaPuqB9Cq8HYCIh7hbAt89eBaMLhIjfxmf47bsf3zz9YMGWz+4kF/kqk3gbl50y5upS0eotuk2XPCZ33XUcMvLLlftI+kVQG073odgh3Lp8+Zhmn6A8ID1VaPdUVOn4ol68ElV9oUJNNLpK5AMAtDN99GcWrlnXR6MxQI3VVQ6jqbMvV4NLpY7gn9jiEMyN+Oqjuj2R5TT58mucAqqBqg+IGfXJ2QUVXAeJg7MgtUXvtuoJTPdPwF5TxN2VcCpEACABKYR/bMldNUSgP8gID9UzbKmKwdZRMCuyiBnmeZ7lrvNM+G3Jt40iE51BNDdg39Ds4+4kGaFpZTgVbrQEoKzPrsNkz8mJJrHw45JvC9ezUZAkmyryzrD13KoC4zXGGkpVwtW6RvC+hCX/Id9zpWYcV0O8Ggr4B9jGSHCjIxpAn8FiTjyrPk+qKUXeRx8KiIkvAQrVPXIP1Sbyi/qkJfclvbokzKiZ96XdF/xLegJ726vPIfYlWuiV4kv5RLgIhpHLw37e7Fpx7sp5mQqgZBMjaU4EIOTEG2Smv+W3BoMqLsZ1XBwKHwXIbDSZ0zKF70hYchPgkKnEXHdhehBDZNMpUyYwjocKfSBCXhaofCoTeOp4hIleDbr/GhOqVTJzsFlAlvNc+iruwkDY2cfQoBp/EZcmGjsfagRS1Bfuw4da0FCV0BDVNSGtxANJwGpDEfa/+Vuqm4G086tHpj1flZLdHSi7uyoXVmhAVmhVWlHY6hOMjUAYG2L7mZh0X4M+XZa5G/ZpAJ46t0DCNKhn1uyTq9Gtt/f0ruWGPiKkGM/DiZVlhKR+MOk15Pb7/Q6VQzr+hU/kW9DlxZwjP5j8oTtEtEOfThCFwkK/LIq3dCXPHHGlNs8S0akuT2Ej2VG6B0U+x2VWTXTogyoYoEEarBd9o5gMLrOAbCizAO0Z87c2EwLSoy1gqJ4HleZvPiPVu9mEPAPBtjDLtNVfK28pQnUoIHWoXxsraYeBVHtNXVTNQ5K46PH242IVp7BDisVdW6iSlm0c8BpWcI7mmtJy1Pm0AGfP+pB9bc8B+3rKzkke+yk8G4mvNBv4aCh+G0Sj0hY03onJpx4MpwPBB45n30wZ2DQC8DdQC9nIMIOwzGkQZojNr/G/kaAHiQ3FHIopWmbca6OUbIVtqPu6gagpB50IK04j1GA3gzja2g3I2oXxj9WyGmE1kw/ICsPCy2M0rFnASJs+qxnVSNCxjFiwbDw28v3LGi1HF3VEhEFHszMcdqIr2RaLT48hXDcUoUPfR8A2fgocYHhaw/ByOByK3oX5FdBpOE781Xg4lJRGGy1AQ0s5C4HWx9AyPIbUge9LaHTu3TgIBYoOzSlXMgF7EQiYEkGVukZFCh6warXx4Vs63KTKumq1bNMMrNm1OFWkKnScBipZucHHPMXwgKixrKeVYMIf4za6SA9kjc/Hw/kTYUkLRKBDxvOwwOdxGYfJCUgS9KHB2l1u2uRJJf1oxlp6rsxFs3PXWGQ5L/c5aCTyFId2+UAEI7m470xoF6VDm2rHxRPzpERFTp8J+vk+rfabmbMmC9UZPHNIKSVYYPDCbn89IA0Fn/7w4ulzZw7Wcrijo0yhNUzRogDrFrU48ZWwou+2IBd4gBSKoCV85PHO9djfpm0xVOIvD2Mg8Lt9iueTL6QmWwXzyCgsRauwZDc1UEqlRbGY84T4pY/RMZx8REBd69SwaXtF8Rqt0ak6ivWLTTg+v3CJQuxEA/WEcbw4lBzao9aQhDAnC+fX/Fd0y8KXFKxkf8M/C5CuZajKboAWCjdbWWolxCswwVcxj07iFKMDAA8MidFRKhhkIGMTBKEm7Eahq+iy3ITpWpjQXbxh4SEYxS5CprE1O81AdkWUzY3GAxBCvAGTWK1RtV5SHe3WHsjj2Xox6Yp4ntrWxjx0NZ7Nq6+Cr8knX1NRJBl9sTyR5kd52MG4lV1M/rg4VS5jyc3AaPLYa0vzuxQxAnLC0BbbkSKEYRUlyAZNbui9yJJr7qLS3OvRCSt7xbdZfniVpRgCKJBCwREEKLWCwC14suqLKL4JHuqjawpPtSYIts9IGIECNJHmsOkuSlY+teuL76Kd/CGE2JTJCvQha7jiwyPr3lUdsIdsxK88G3xRZrsAI4pKCVbELSBcFcDgv8DHLkCzTC+qnV5vkwBlqmy9yrUXBSv3G0ajMQ7hPUDbEc/0aouRZvlow79hw30UOkppEBTAIlBhg/AatlXcoepuHsEsP6NCK1nlDbqX8CyRdk3VTh4ryskScX0FGDkUm2KY8Ua3whalGTDQNauWwHNTs8laNwlANuyA32KQbmZrn1oFW+IyyzFHMTWBDIyZsm2culTXZBB0uCETDP3H5+bko15RB2Er1O20wugjFYtDm0LCQ9APADrGC+EeQCFB6BbfAPs4NXYz+KPR/YmgTxdhgTaBFJ+SGIGC5lqwrcXSOk94olkS2ypIBZjERcek5QebKDI8Z3dA5u/ZXkwqo1XxEha/a3ueQIJFqN5N2cyo+Jz6+56XP9Djbw8v04h/dmNheMW4geYo+ty2Ns+yPS5Nb27jgfJHRk65NU3F/Pu0QaMe105NFuACQrdti3sS/yheq1+tBhgRCEmS2q31cSBi8DgaSYZ20K0Ub4OHrEDu7VaqbEFr4rAhpCC/3gqmgWUlevaOA1e+kDsr0YQDbDBKIxRjWYGrGX7Upr3rT4Scan8lgcKwxO93e7GlCOfsvWE1AZAd/fXX7GJ4J5B7EEXPPHs0bensXu1NltHaa9sf/7zkO7XIfIwSJAF0HEuMNu91zKfuuRuEvaOhRA0/u7VC4DxgdSJD+8zUV9KnMEbpOzz3er1GvZbN9gN9c8swB2aaykUMKkMIRE5J4/e64Ahvq8F9kogv6AMkoz30ByL4Mi5YpmKBhJV8QE1/n1Y74T5FDVflEfzy/Pkrv9egu1Z8ijTcFZusJNKR9MGjoBkpO6D+zCd/WR9gN856txeCm8wAZ4Kz4cgooQmT0UEOxmcHYt8U7iUVxwp1kIS3LTqW3GkO6XKTgy73e8cerkJcxZmY3Rj9FXJD0dXaoYiYmruAqFqdMITNeVNTwRGWIJLGQpNLl/Q7G6ledRtVUGtCi0IGIuid2ZqbmopoN6/2f7VxqxZ6I7cbCDXBBl7zdFK9Y5NvWBb4d2uyIq72ijzVqFmLCPDYNwYC9zAk36g4ZgzhFtNLyxNTIsTAyHTEGTXPiCTfiwfVElsmWcGr9WXv7nUJRBt5Y80JkTHplCUYvO7iCLI9xbgIbwieWAfCJnTDSUvUqmnoEG7lHvhjRmaPsH20LMBIWCkMpEp1CLdC/aNo6HpSlArYXuGvQOUDVfWlRVq5QaUHL9cB1c/fvhq/F2V99lxV0yX/HOZ5WLzHJAwAVAHe7kEmKCCv8Ef1rATWKjDmFg8YVD/JAvR4cZBd1STj1xfGr48Wsqr+DNTobCsa9Yxeicqq0mIfJ5Lyoo55zsTIWWoWAI/SWU8VtGJTzXU6M6dQfr6ivk9U0YkRC4oL4zpGuk1r4eWeYbAb6VSIm/HzTszsxfxFaIKFvtwUwTiNguFwOB5irOwRhHVPFuaKosiMwEErHhBaJnnvjtLAJKotJoGhtk8s4+OkPgfmdwWMmTwppJqFM/fk4hbepKkBwZeFvYZmqnKccOOshBqGmbvq0X5nuLEL765aH3meAlIFbH9mZQMZccARxVvDaSMHtMu53gbkqGZOkTvSv0N+SNeR8fwwL7MRbN7nfXbaZ+M5/BzDTyy66LMz+D0aX/aZGUt73OGjfO1WHpKOZpb5fCcAVMTjnYg8PrGO5BCkW0/KX1yPQRc7uPLY8qSWSmBGTd/NJtgFwDYlgGtKD/fhQwOLBssIIA/Y0/fv+uIMC/aUEndPPCQAXQ4zQ3PYoVLcZ4ULg2JpKKsMrSioHKON5FcyBfAxhI1bTX3fcOn0CXXpnFPuNeCeg33K6mAt82ibRkPE4NY5uE73MwsldzbOy4+1b3soWLblgd4prNg8MzKAVgkRwsfBuZ59FE/VZJyITYZJzR5PBHGNmCIRtFlodxCxVSUoMfR5F5A2gqJLarayUYCzaBgDFSz5beakvHTmjfF0IVkbi4RyD73mWSXZMZ0yLgrKYNIe1y1tyiL8xVRs0OGCCTA3gAesk212zTELL/7sOqKJ73gTVsXPQDUjkEuNkqipThQqZe4BKPnGtpxmdKrE8XxiDRPKc5H5aGfpLvaYP4mBlAXWScvk4BuWlDzOmJrz6NMoAuQ81w48A3Q1+1Clmn2NIYIwcJ5GMjhGn7QWPiWT4yluHBWOCI5CcHJq2g8vFJNRXzOoXsWtmfiS+mYgTV9JE4NfNePSbMeB8q7aoRJF/mYcjVzv9XVure/WxdpYpObiNGhnIlVmIIRNowimHYNNrExbkUMpj2jK8KCya5GLk0bKMaYH0+nALscU3UgwgFIjpw2d0bURM1RP/a0SNxLM1FRDXWOTLeL1NgwKqX5ObW3UFU+BW6dDH5QbDGEQJeHnKWyg+SabXhnAlgl00DxIwdlSmz3OmERbqgFrTKumRE2oYxgUwB4RsIlQDvoqjbaPWZ6WuCcPkYTui1az4bzf/txQNborya66K8iwqrZ6ls0jemMDNmYPdR7wIxoBaCUC1zvFXWPJYaoKsRaZAbBwMWk5vc6SvQhJ1YJnK4LnMUdW5BDrQKnFge5dEBg4Vg/WeXSN1RpsVRXU3AFGmBqdOBl7jNd22Kdi10Rls6hx2CgGr2IgyEOvE2H2aYwdOhfDobHl3ipmFOZkFetFy7/LpCQk5NmZeY6W7io7Ez4tW1LfrsFgORcr81F1lUZlbdGdHD4sKXSD6UxfqRXvsuQgrC68XqBPFz2g52ZFGp7biFkDK5ci0R060TsdY1ZS8ik8gGkSmbs2npxM6cPf8jB1w89xofJJ8DAfuwHuNHTgikVXQsubUpayv15Gbk6yvJF8ZvRkDcgVYzHTfE9OJFg5RP3b88MCR+SmO1+OyY6+8M/x9AhYm6B6oBabcDEW41wd0wEIHWFGlX0MsGoo9+ah09s8u8Zs8hDEGtBJBpQZqUDZivnnvg+dqLQOjR6hi3gXv+X0SZQmmtfGdHHmsYcP2djzPHSc88HFMZyqrH7QJDioDks89IcFXQDP6bzMLcc4IamLKFVP3OKiiwv/FU/e466eZ2tQtt3a0ibTbTqChQS2URoAyGJ6SV9XK8xMOQMz6VOcBkKwTEdjrLfJdqrgdDg0doAHeN1H86IGcY5XmF1isEeF71cYS4UhSzLYVgITR0gFBX/SnivvR6kmZ8JkBqG8DYBMDutUng7bFbmQo02iuUI7xkUapPvtTvCtv8zgmwpdFc1cVynSaxdm7xzEOzzw92nx257z37k7hIl9xM6QO888o3zkNXUJStsT8CqdGeWQxanKAyvdt8Bo0liwtBkX0PCkFPMbyg89xcyXeKvWfs3rR6KywEQVYUzifzJ4AAPtAuU/pFB+EIU6z1xg6jiYN4a6rbrXQl5rgWlBFKiJg8UgbKkIEtTBEo8W2QLFqI+XLMj1ZPSIq+TYEqHbNaiBdWK8gBknFfdaxYRSuGBf5k2hOhqSWYzj9GncGHwwm/eZYGTJc1TrayF0EIAxMZjvJA7ERa1HJtp9o4mlI2CrdnjEZCB0MBN1yhRMdBiDHuZiu4Eo9VBmju0TGjqtpDx7ysdxc2fm/+2f/u8//tuff/zXP//4P/PZr85X//qP//2v//gfv8692/nDf4cuK2fWfzL584///ucf/+vPP/5nS52WA7slKLxxRJcQTdksrp8fywH0aZyP2MgTAWN+wUPgXleiKHiqgOYDNpp7817LWVrVUfthmiB/VWs2GM1bKy5g//xoBP/g/PvhDi9zcR3mCCe0wIfIO4EHc8+yIolToNhcOAKQXDgUi4j5ygVJDqk6SJ2FFk2HIoHXn4Fcy/E2FDxTEGAHlPpMV27JcN1Y3oCGuiRFKaEclCo2LSK9fMSSEoyJzDPUeHh9c/PULCS3V2o3qR8KYGU7+NCQjOIBAIdi2OCSOCVdCrsEliWASkup7+kC3GxClebs4ZTgmI+EbJjoZ7PJRM1wm+gSAUh4dmLOgEzDk2FP9hwYhwbVeQHVm1dC7SklHNTz9uiaKX0JFh6AsLfPXsnbuvBynh3esaKnJMvjNQaVqiNfrS7g/MBHuBBso7aaNeajiRkc+UM9CthSLEi0tmzYlMUDbXq1KQR7BIH22Y1jNQDl2foNJjHWg2JR3ZFdQ4H8pi4eodBnMKtE7HT3QU21jdTObGgDsQ9tJdlJV06T8qOvOlHK8oec80WYfvwFVusH+Syv2lR30vmLcPkRJYRsKC6z+1YUGgcrOpBQ9wAlz1D/A7jCya5hwoxYYFyVyToNffW1r/WQYLdPl6VgFxlPi+sXbBBc64aCrsYINFiKjilsoWWguEEZ+LledVYntnmX5oLIXZEYZI9xGi5vakPhbVBLf3VnBGLugZHb1kbLS421niNXVfOqsFnnX/4F9pmvnK+Moq++kkV1UQNdwo4haVBBM878aZcmSW3cc3MHM2mZb2Ivtl4jaovC+KcWGa1dG+WmqOW15xa0RhLMbkRG2URg0McTcMosn0hlDgRKAUqMKFWiUimY6dqz9MrbeR0jMQyMZhjdHX7IhZKkzoPRK6Wu7aP+hQQ1vKebMFmxqdkR7hm9f1ON4w5tY2SpWV+uatyhZkgKoByuqqGL6jBNwu0iClkMUg+ENwLHyl4zXMfWPiRjGJw8m2DDORoNZikWTuaS+fXeO6uc7YKn47Sus/ebGTgNY8KHeijARIdUU6nAXrdxgCuHR9Lgc8WHWogoZNBnBwq/1HbozoJwVxkMOmVYBahKCErS0KBId0Gftnw2mhjLWDR8NJXVHon+gPRGUKpUOmTv1mBE4KcYizh8ontjhSXcvn31Ko1x0ipj+tWRltA4TF+uUWIYqrrMcDKJ379FW/mzkmltnUqSPKBrLpm8nrBKoxJHcHFa3Vdo3HdCGxjPxU2feD0sXUi4x9v08GpUAmxA0ix3L1PV4ktJNmtBaEFLD2dKHLa42fUJsmAEbf5KiTkTLTGLY8dR8583pacKgZ+2Q96GxUcNFi8+LFyUJ3XAng0Yc8XvACzPW4UlTqetSr6TBkmOS+x86v4X/PD8mKJr7QR02ogiC4y25elU1NfeYHcAaA9q9qBx1CNNHWrmL3d7VCMkfPVTUln9RLTEdxVpgAulzgzArmLVTTGb8iFJYjHpdMVGsZyKFJ8T8/BjleCpDmwF6LqdOlQfz58Ktghzm5OqAwQh7KGX2u6wkzfVKAuHFrTOCpjNGxYyLorPYEzL7VjcherRiVeKiaUg393f452rnAQVFdsuUVBZgzRPGD/O0eSVaD8SnWE+xBD/ThvNBUNtQ7ApEoLiVgBb9hGYPT1vesb6io1FHq21juycWjnWZnj2fRa2nRWEtwFNlZDrDmFOM1BC5LKBGYxSWl/wb3x+4TXXUL87TFfNwpSIcHc9kbQ6PZI13Qj25iGmwhVTLai76wKniNzXaSMbtr2R51dOwK7IYcXK3Xn2bXPQTAXGgycF6hFzR3gVSjNfGGpRWa8j3Jxujp3KjGNVeTahcJnJvAKMjxGYppqu0wpZX/g1VSnvJuwJNDRg0/MGcKrUCvwBe4YsTortw0VCl75L6CIKxfRI4/4wIHe5QbrKRdwkiSSp5YVVw+krkkkH66BjmvWxlT48ntb9tb4Ze+Lq1S7h15Z9ey9Jto7xXvigBXpSlFu3gcZxbPWY8Sg4i2WQmgG0upYuz/7uiu7BBiz20NURasjrqSua0sXNGjvMn8E9GHY39KXLLH3PpzNndDNORx5t8y2JECnlRuq06SklJ6n+CDGP4jlgfwP+uhgOieEa+dZtXskaaJUXXyv+pjWv/GH7IdtRw61TBDhvzYvA6HxouysPJ2UmzoOfYPwVK7ZhglEFjbvREOlVSGEq6v6/dtnVKDW2WqVatLa86UbdNIrVpSv9I9WV0VzTGo80afA4tG6USXXnbijYt2afuxrJtSoOm52JWrt3tlPSRDdUBXe2tJkPWtoFR1pWCoeiLv1ob3Hbu5s5QInz9zu0k+t7nXhHh2QYg4csE40qKZd6TCGglGONktlIiQdTSKbAdmR1G8drJNQz6NV18oXQNNEFF25tZXORZEsyW0Au5K6w7ieypjiAvyRVbnwmPzzMBa+7KwQ+igIE0xqffG4mj6vBYgQHBQTdw5OqIudqce42bYx74KqliEFNk8blPo3rzOz7igST1fOsxUW7tbJdS6G8H7SWZ0H3dTbzyg1lqv64psw1+q4pOfXnxq0ujfxz21/S/VhdDFlLa6lvHPUK1ZWHNZSrc/AGtvKSjXq5edtG/VlHcXVfhVOPW7q1OEXxbeVwwbsgyKlUHYZRcnugo91r9zRUFVUQaVdcfN/wSen4HFq909pibkS+WFc0WFcx1HL8Kf5IQTXlCSUH6ChWI3hp7nn1kPMOrGwQVlDT3LsLQ0so1ADTpY3RfrsrjHsOiwzsf4ymFQcHvnRU1y6iELJEBtjJk5kucYKidlJLmvvCIKimAYtAZ+augvr8eHT2+Ozy9OLs8RdEPWhPkQB53FH0/8dJ1HQQwSr6yFu99fYVgoYSX7lwLM1ewejQ5yM98KYi00GDB/hKC7BiA1RUcpEwj0rhb/v4OkzorSUZ2UDGja1gYw42dPlUvE7xZjwD2jYEifZZxxSKCHEZFRFegx0gAnF+24cRYrisgKj0P1TijUWOyBWNcXXOaWWFSuugPjo3arrA+qofYYgZPrGwyI+AMiesG8YD9kbZjzLTQ72JiBWbeAWj0+/ioTc1ibumU3z1Eqw7v3NoZEzKQrJx0cyda6tPFUlnZPe4hFFa5CYM42dre+VJELNi665dM4MbKbqi6O1ITSPzu+FrCr9ytdVK4OxAQRmqZoUGI+4qxeRh7fI62V1fIlpTlm1MW7JXVZemVSn0al/Ee3lmSMMDJu47xnMk0AeX6BoaJPyaJ+KCY1a9aAhEI5R3vwboBDZsEQxQ3bTV0Bdagr1UfAQbtNSf94xDQFGT8p4EbWrmwPxoRpPtInUUX8v44W1c0BvimHtjdHXLrgt2Q519ZXf21fzWe4Ib+r4QL5UDxbaUb1SSMJ2aCiKjT8yATdlNe8imnY9FyXl6x4vY6/D1yct0Jd7c5Hj26ZkRpCPK+uL9ZG51l9hDO1SVfLcebn/dx2JbZPW7372ij78onIj2BRyHv+P5KqBTKzrR1xeKtl0nStdcTqsbNImZ5F2aymkoL3Kt7tPsThAbFHgPs6du8NJRcoEKIRJz7sO0RDgXufPr+0eOsEQ8+8Y9upLNaCtu2jsSt1fjO8yrAx0GX2GGHmux2ZDbH8c2Gor79T7haiT9ZaBeIReJ490nbA1o3FQY3Nb5zLpiTeTVGRctmZHbJBDQTkx3XxLr/SVh3lUAym/R1g/3+F5ECQoKej3j8lvAtbrVFTiSDDaffwZWA92ylhkDuli8QgsTlUjKb6m9MgKmm+72A8DGzZ1VB/46yRauMkwe4qsS5SpqLMTvgBQvCI3WGX0jFj3A5Et6a2WYICMe1H3bBb1lD5ZCjNHfYMvemHjcCvdVCFQCgc3M96sA++MLvOjddHiJrdBDyJVQn/OtuC4Ls2PN67NceZtVqO+6si4JlWvNvB/GItH2I/yPKYt4lbAM7aH5CLKPbelNfZ09YaTQWTnuUtf2miYR7QS19IW+AiuHoYRTdVZNGkEtPEss20ZPSLfA8D6gmDR9EaqZxLwWFbnfaedmqzhTEas8sjpMyYi/Uw7qDbO6q7h2lUAobrcwchvNyl+U3WheXYBgZ45BB7Bj5PZq0qrpwm0JTxavW6SDSWXOnegpPTFy1sWrFaVY2NAd69cSdafFlUYncIQn/XSqQ5Hm5eWqgRn1EFah0HUurc7A9L1vIJVM1Vwe4945+fXJNnK9YZJ0mAZyudyI9eWiHTsxMaV9M2lrzZ51dMM/GafX9vmy0CjtI2CiUM21JxUdtTXawTAz6Zmuq152RASCVVEytRAVIwWU7g/S/bhWvw9rN8Y+soY+wHDrEysrpnlcDqTodNu3u+zlboC5Wupoe9TvqCk8ybWz8I7KXW5rEcLT3kbTp2plk6wTMZGxU68uTpYqijab3/baPd5CPloPDc/NfUlbRURM9PR0DcEWuJQ6Z5X07+jCaNko62gbp8tkH/EiUBcyqGxZccNRC7WaRTHFIE3H9pPaecYq2RebqQ2zsUnJI29LHWpqOLW7+FBIujeWMJ80JDnmC5MUFY+K237VZ4UGzm+gQ2Es/Gf3nPCPE3YNtiqlbHu1N31cq/s1P5D7SiZl011VlbdEqPfXnTfgfeyLa8NFUJ9MWu8KOqi/DqOOAMKr9d+7+1SoIfOsGs2gHCV/xeFPs55SapuXUNHGhfmfbTe31Fzttt8aGnV6rqm+0BLM+p3+aylg5IvWQJEVEdjudcsEiyMsMYfWfF2DveeGnr5moBUdhC6/tdyIi1HFlq9etNQmETbWPxqHJ18kXP6KUHGkfOuUdLQE4Wm1zOpnH6RLBbIaklgvUsOUawEa3LH67RZCvuCIZvP6SRCZEfBEWha+vobOa7vvS3Ful+yyzbX2Zj69RlvY78YGo+r0K/kKCgy+ejkIwf6M5SsH+6xxKYytWqsXG5I63BrEPqxdP2D6sdsdhHa+f2uVegQhxmSAso5RxUL0Fp4dREj3hFd+HqG6toUR7jC/lRe1wNMv1/eMUIf7WSqmd7OmbXZKxvYgEIKglLT6aU5f5RI0jMB2Md/k1K6gLyJbx+k9YU/34gtfamHmh5jDo2SjOJWVrPBt808qv4GZrCbLREg/dWYmrHVuYhao7pC5zjQ1q31XulrrdivpAXVB3OGFfql1etr2d/eFrwR1NhlYaB0JdbHb2c0weU6E0+HoMcUASTmb2LAxxA4e390FNa51cLxVN/Han7S/d+sOggtewSUgw7ilgdWYyT7r4CSShKZd1XUfgoVN424Eo0nfBtg+WjUe+8YEORobIePShNa7FPTdCd1Rq5oO94zzUncAgaThIW4sdCgpd+6ZQG4+a1pkc3MBNz0k8sUFAb0qA/fEReHqi48Gui+M+VPf2+ZL49T5qoUvuWKlclu+wBtTqthHdFbiVWdA5hjTJ9wb1dmtfhOnfGfsjULpVr+X9wkTo7yxRz3xL/49npNIcxq1cEXF+wbhSSah0I4U37CexL+raE0jE1bzkvHGJstf+AW7hPkaxIYqs1Iu45vKRzAZnka3FAYQ3NQ8ArfkWG7utyuh7bhGX33WtiDwXnB66ec0RFCB/NWytNNVJm5RwW8m4FYxgJV8Oq4tkLM0l1HuADySPjIpIyyU7s19zi9Pf1apwvKcjV7pJ0grGM1pmSKp8c20gjr/q/GXGBMnNWiDIJ069F928VThRxaPogreKDwaUSnfBWx7fYR06m6mJs9oZUzvfYJHK9uGGqryI03thU7XGpsFR1pKPc/utsOlbqmlx0JKFwXeEsXlPc3qNbTOpDUDXUjsrsm/T/RnxaLSUJrTEf+douXfyOCxDBoarWnyKDzFtccYTBXj+1XwcC0IyAsSBHjWGwTy2nDrhEis8/eHAuz3F5/j0qVjYalr1K9IRzsFyiY1ZyLsPu/evXk3YbDJLG+dPoXATvFNMUUJlpTxitFGbyNPHBxAw97/A6YmzNlpjwAA"
  },
  "omnivoice_samples.py": {
    "sha256": "e26e798cdf6001ede357d6641fa32998f40676c2962d25a1b82ff4eb7b02eef0",
    "gzip_base64": "H4sIAAAAAAAC/8V925LbSLLYuyL0DzVYbwiUQDQvfaWGs9ZoWjuyZySFLnPC7qERIFAksSIBLgB2q0fuiLWf/eYn2+Hwj+zXrD9gf8GZWRdU4cKmZmePZ2NbBFCXrKysrMyszKzffXW0K/KjeZIe8fSabW/LVZaOHz5wHOf1Jk1+ypKIsyLcbNe8YDdJuWJhyvinMCo9ts15v7hNyxUvEvgYXvNFlm/6a54uodx2Hab+wwcPH7xMizJcr1mSspAVfBvmYcnZG+qJjf3h4AmDrpM8Szc8LZkLzS54bhQYHsGfUW/y8AGTALL+hm2TLTQpmi6zPFpNpyP/3B+Ih3AXJ5l8013PWSblk1VZbovJ0RH8Xu3mfpRtjj6O+osiPNIY8OHbvx2cz/lgfhxF8zAa8+GYj8+O+fnpYr7gYXwMH+D18ZiPHOaUeZgWiAyeF9PpiT/2Bw4romR7y8o/x5uHD15kOXv+4btnbFdwBghkm7CMVkm6hGG/R/jF15sV54i4mH/y2YsXmy1fQslbNucs5REvijC/ffgAOmIANeCtKHjMCH08hXkjJPjs9bZMsjRcsxQgCtfJLyE+Qws8Lpge46OCXZXpjKbs8pOYcRPlGRS8xoKBJAcfRtPvl/xT2V8ka4AozHNq2S8/lfAF5g7+8qJMYHC8n6Xr23+8OZy5fBeVzIGRcI9tkjhe83645LHH1tkNTC8g0mPzPCmTAqg1ioConH+kY8BnnzCJmPWBzOU7rMGcS1wLBs5p6qM82Za+AwW3OUxM2QfagEmmbv1t+Y9A09rgl+O54GscQVHidL+HpnZbQCwPNyzfQa+yNSAsWCZZBO0WgppgdX3kaeGz9ytY82KgLArX0W5NhcpVCE3KSkhl316+eP32ki15ynEcQOHFlvMI+UjMit12u06gGnGUNayOVYYLIsyXvAyAkUCD2cMHGrakpEauOawJiXCfvSwZgJJmJbImXCtbDn9SbO4aqD0GuGLkVXESIUQPH2QLtkujDCkpTFL4mIblLof1AZCFHxFEGDw0/CrTTA17QBiAtpDQyjzZbPBHlj98sA3jGBopM1iEABC0E7J4JyYN6LOgpe2zdzCbSCA532TXWAjGrwe2ztJlHzt6+CBa7VICIsw5i5MinK95/BQwBJiKEhhrCkjoUym2DYF/FFQyynbwIfbZj2G6g+apQAFTAd/m8A1BJJTzP+8AOUm4BswBR96V2VEMDHwJsGYxF/VYliI/kJNGg8NZB+CyNE5oYMVuLpuSXTHBhlLYC0oY6C3tFtkO14YiThzVIs9+4SkTm0QB3Oa5qC25fgENIpqKo2gtB4dAr7K8ZDDuTZLqsdGwcz4PYaOJAEZgljC5yEwBikWy3MGUi6GzLdYsgWr/RcIUGisWwDaQK9gLjhBaeviAlg6TCBLwNHhOn3gOdIYsiX0rOQ8RE89xpXCgaQD24QOkdxgJTkUJNDTfwZLxBOmy5Q42Rhg8ALSAR9gmywRXVc4SpOakvEVIC54n2a6AucpiYtWvxHqAQW13tBBGx+zj97947OJ0oDbuI5o+n11+wt5h6PQeZwM2ynQJgxIFHz7A2ZYAAUHBrEVsDhNwk8TlCusjrxNl+0Rw0ACP5FQA5mCFwRwCD0jZ8PcA3huYmlQtKMBytN4BKlmyQThg5HF2k66zMD7CP1DOo9mupgaIgsQFZH8PH/xLCAvRYC0w2EVYlE9hvtdxH3Z0GBtyISQ+XK9S1AA+AM0Q2n32DLa9Gxin7kLsgkZHT4WEA1SMgNcYbgj0AMWTEgn3zQfsJuaLcLcuxXpGAvmF5xn7I3z86e2zH43dl3Z0oseCtny502tuicyWRk/IgT5glnlYEBkjLTUIBUk+g5mIc6CAnMoAWM9SVjUYhVv4f4oVNcNjovzROpnnID8YGIUF8gFIFQeGi4mFDx+swjxmX09P2B+/pTFFPCHCgYEDq02AyFPiUD77o+TwMDF/BgYE9OoRr1qvk2UyT+gFjBVGmPIwp40cRqEnLAdmkgAVySGTKAhN3Wa7nCEQN0BiRPBvkhR5dgEfIn707M1LwloEvCuSXAXW4XYLQIL0coBsdwQcmB8dKtxVTa52yyV0sghhD4qyjmajk0U8P4FWh+cXp6fn8cnpeDxenJ3Nw9Hp6GJxHMVnJ+OTxXErpEYPR6ZEeTRfZ/OjaxIrj4o8sj8iF18XR6tkuSwC2rOD61FAHCAB0hTfodmgqwQICL89OHEYVT3DA3UCSgbOKGwJGxYEix3swjwIJHtgRLaCLrGUepsvYaUWvHqBMoz8HSeLBZC1fl6Fxcp8Fv+0vvE3vAxBVghbPu3KZK1f/6lAGUI+rDPCiX6GTW6lH7JC/8wreIvb6jWyRTn+LdSEvtTg31BD9KW83WqeCVIEMt9X4YYXW5iIqggtSlHmWQpL7zksa1yYnmZAHvuQkvzz8MG71x/ePr8Mnr/+8ceX79mUOQfrNg8f/Pj6u8sfgpffYbU60evPby9/evnu5etXWOjQNQCVXz17//Kny+Dts/eXUHN0PBiA8vb+9b+/fBV8//oNvIJNDZQn4KtVoZOHD767fPHsww/vg59ev4RRvXz17v3bD89pXF+oJEi8fP/s3feX76D+ZxSkGXO0EE2kUPhCoBZSgTNhzmi8mM/nfDw8PR7OT/loHJ0cL+Lj0/gsOj2+ODnlg/FoNL4IT+NFfHEah/MRIHd4wk8uzhfHp+Hc8To6gv1zGQA7w0742cXoGHabwcXFIB4NT8Lw4nh8croYhfPh2clwFJ7Nz874RRQfj+fRxRwEvsEF4PnkLF5Ewyi+aOlELE5fv8BuTsfDwclgDo2fnZ3ML6Lz8GzEo/F8FJ/Dm7NofD4envERPxmM5sOLi/N4PBydL3i8OD8/Pz5dLDrHomRi7GQcArWdnvPzCz4fhcPBRTg6Gyz48fj8NOLDcMFPzsaDk2hwPj47CU9GxyG8g4eLi3gRAumMOjtBxYmQFZ2Px6Ozcx6exdF8NALEj+MRQB9ejEeL88Uc5uX4/Bwo8BjwNz8bDcPT8Xw8ng+jQXh8EmIHd7hUHj4AlNO2F0i5iceulKsmuMGhEgA7HCww+cg/CWGDHnus/w3+OxHwggaxy1OmGmCPdWn2RLfD+mzYY0dH+kUFh9LGAlKY3BueLFclwLEGceUKupmBbgJMc131rb9ICIDn/oDqVVGCErsJUV/KTaFK2HiAs8E+jDswB4kduA2K0ELu9olrY1PJgiQSCQRKp9Q3+xr3YQVbj5Fcc+vegBjBBqQl3KAtSH2XcBF2wgTEj5/C9Y5f5nmWu853Wo8CYVSP3oaPNi6CjzRyCWRPIRzHiMxxKqHrW9CJUiBgZ6haoMg0BdVmU/ue7jYo22R5AZ+vqjYfw1DqA5qJKlLBwfJDmNtrHBTOqdkVVhUfoHrVh25hUWYo2FWQI2SyYQlZlsdUoiAicnMU511zgD2PfeS303W4mcchSybM7btVV1fJjP3ehAmKJz3ZNoKXIGjUydVEATQzpkxCg+08mbKhReXym0G9JXQTBbiDurjfTWib8wQOJrhvEcm+glmVXZQctzSUUqe0Q/pInkEKBEkN+PgLsOv4qC6pOdeV/Btg8DxAsnArkLF7YEabbeFSx7BiUxQ5g7CIkmT6IlwX3CNDQlpORx5R3Q30mYpPPezv59TBaiB1AhlMnV256J/LVaEmpvBBkl/DHu1qeDwaQ69CCEkyASzHHcq7hUvDVxKOr3d5iQx6i5OtSzyTNd/QF2OMsDWRaQaWzjQI4iwKAg/nE+QT0NAC0K2LYqqbeRvefFdV+J6vty9UUWtMiMhgmWe7LU0H9umHcRxsQHIDLN0GpOEVoF2IUm6jHhVX43UdYehyDitG9jBAO8pD0zcCkRVa7CoVJqCyMAeYNT22gkFOnX/3DkSUbP4nDmotaN3Pf3iJmCtxKaDY+ZTeEJEAewPKz0EVR+XPnuo2AKDbVPVIu4JUE6fD/ZUKUOva643Gx/urxvwat/CqihNtd44aKvxmrvxCPDnaxeHVhGzLM2d/y8K40I+T3Eai6gifXEeUKo4qaaJ3z1iFIQF19NYhIx9Q4EvTxfe/PEVrHVo8lN5NUqJ/zwjIUNbfFKqfxToLjZ4G/kB19LyyHS1BeZ7z8obD9rIxrWp7e7NIbxN+6l/D/tlfzju6Pqm6/o5HCQisoGp7qHI/FRaDSpmf7+Ilxz1vfXsABVpwWNZhgCQkE9fUKaBdYJD5jmtCeZWRIXDdMM6Q+UnYbpda1z9sKaAYuwPx2yROnh6MRWWVc9pp43m22YR9Ze4RVuWmlY37S589EgrBzQotF/mjw6DXBwDt1I+gHDoSdW7QMZK3wggSC3t466nCU/bs3VsyeHOUDXBuePyl5GDatLoHpaD6AfqwjGKk6z1fQ5E31BBRRghrMkTrGgplhyEWrW55hudYEqKKMvXe9G2WgaiXKiX2GX2u4HwPpHsPG5aHXlwh/gt7EJv+fka2Nbh2bX0P/cGhs6IVJK96bTRp7u7GJFWvxWy9J1lRLNS+kI0LjnZ7WANzPMzgLUcIT/X2VlBVoCjvkBlE/kbt9GUfHVg434sF2ZAm93saGw7uaw0E3H5R8m3rtjIe7a+83CVxSFBEYSVw1GAY3QdC2S9WyaLs3HCG93HMW5iiLQeCLG/7C6DZLO/eQe5ZZllBZzZ9FESRb8Pu9mvbIsnxgIbunSLUOUDb+81XvLWiFsAcu3f9obHt/0C6CnIkaPGaDqpCXAyoVpp7/2GMLcbuoNdohZyymLoOnrPBC4dAGI/0z+Ep/pyr3z1jg6QqB3N0Hn7sr5NN0k1xF+dqsG+gsLbkXycho+OgchcL+QC3TQ5DFnh3B95w9oX7C0os6z6JEYdsLxEyrDTcFquM5HA8HdkKM3+2WCRRAt+psQOlDeodtYVij7xz6Fj0Sfl+sek7eewcwWrNOZ73gaa64WUSHekTXXFYVhsED/M1aIZBpVIJrfBjCvIXwlQoNSpZiMK+UGgMLVy8gBZIt0WZoHDNoqCKhrHQg2tKa78AzUjJ6YZRJynEAQ2or6IJmLskKk1rjYE9Lqw1StUS1hr4XYbk9GOoWVp0xv8EOtFG8ln89FH7msj3ZH+QP6EZ2VUgK91VzWC5j/zWq2wpctSg/G8AezWYoSj0CH+1gu70cREGJmgSFViW0JFqYEVn+Oazg1OPVbWG6SC1BEQtd7VuG+haOB9SmmRsEg/60c7AYzWZ0MmEfYa/d3W4JE6mCqQrKDRrgG7MYBOQUDJVV3Pd4B3SNDJXxXD3M+QaTD06hatRjjSvzKF+735sCM1LDFlQ0JxTXejbb5mblmFJAhDeIMhjWkYOYEpiQ74kDjLL9nJu+3jItIq2sfaR1jHzK0dOziwlX0K5f+bgaZv4ouG7JE1Rvd6/Bh7QMjpvx0JjyG0jxJ1nD3WCavnrQUN7Hjk+pMt/6ix1F6wN6jdY2L2DOjKw91vNczceTWwgLA0ktvRZ5rctb6spmJqtiuG0jJx/ijgouO57KERnEp5xPgFMr2A0qI6eauN+mZJDmuLwNH7g8fT5rnvYUoo0Rm5tSer7/Xjv6J/abPQv6xa8DKTUVriPH3+mbUnUkBVQRKkLLzWxBR99vTWaJz7ipAC/68IulaazHEn5PTadGq8DcoxU34zWbGnkTZ5do71W+RYiewDhUhiTkW8ZZmVk6uicRFKatACKgyc1dk2Wajgp+5oN6XBLDA527q/ZwH7xRBXtQ9Fvpmz0+PF41Akw+riq1SB0tmtB7mjpuWXUovqO5yBAAVegwFCjPQ3fIknx0EM3AJhzlIHCATU7IHE0qFRseql17/oHtF4Gy3mt9TQD6XwZqg4MgZnsCQHqXMZL1KXq75SaHUg12/hEym8gld9AKb9mJ1KjDSxF1ChAamrbV+NoS0pxdXw9aRmjMWWKeSx5idZFoko6YmvK0OgM4ieFaE5yGKaP/CSxuB1gVIsdaH+wV+5urvsF0NJnU8J9FDzy2KP+o17Fb0WH1M/nR6rbR0robYGIrwvOHhlYeXSnVBnN++0lYtAP+4ad7CF8w06u/NWQ86Ln6YqTbxjwDXgiPzSpuZorsoluAgC14oC0YkI8lhoA4mvf8GB6uAe4SrduQx5gilTkBoOQRA8jHw32tC5tFLppsapHg9lT8gqWdogI7a95WOD3BL13jxbkKiDsfI2+tf9xXTZyOzClKvQ0A9NNfF2jv/oAYuuMvs67RCcHzFUZkMGsAkC+EG4D6iXIggFa9pD57oFJWt8Igsoc2ICwgTdxJBVQhEoddfhwPhgMEKBGYSShixF83QOUcd5V6TfyfIkaxl5EK+z7XxqwNXg3e8wMV6j9CGkYaiuPihvbncJw82+SFR0xsq/Qq2u7c6S/qr/Yrdfk5e7mDh4run+YTH6On/T+4HhmvZ7aszvhfKdPbtRxpjx9nTA8v6ydWnaBNzXBEx9IcsT3ytDWCQI6wEojHfanLHPoi/r8zYenTD2jk+xHnqfAFcn7GamjOnjSfvAlntXVwIS9VrhdtpKYLoLCyZ4ZNcNS6NSoMOJSnpJjP7rcR8bZkWQIKqigFTASjjTiLFD3Tp0RE1ODR0YhPQ/JR14c9BTaOgT8PcmtqJkaUKJCIP2vFWDmW58OptB7oh3uFiWhOeccBCr0hOr2/SaJUQ/oqBrw/SDref0SsPeC/AwjS4oWKNuQL9lgJ7w6qEr2ugeg7uEYoNqNTlm7f6blKyQ0AOUXU6A5OpBhKy4tBNQOPRWXxCfav/XqCj7MyCgDf4HNJZvdRnribWB7UU8MdzHtEId1Koe4bylIxPCDp9hGPAigWJIyz9aFCE1YhNdZFaQiY1QYT1FxKCzPOAmJUhIkKCj17XV4kyfAaFpQu6g4pxOBLDQxzJS+RdwMnhnLLvTs7lIMIZgSgwaGCcMD9vzz1dV/+nk2e/Lz7D///O4JGucBubKCcG6YVBhC57VZJTBjgyieUMMTS9xV8+Litx6IPGoibPmwMd6F8zrlFElwnRQJTCe7yfL4SCKdCV8jIQmKoAmBDBLGQHPF3r6qVGcRAgITLR5vVqgnipfCJ5EgN2UZmDl0nqMiT5T7WlUXP5s1hTamxuowx/9TlsiPV6KVCVWCtma9HkoF7YigMlOzQ0RirbdaHbQO5wa00LI79JiL1friLXmNjmqKPE7dlpy5yI4ORHDlf/WH//uX//r3v/6Xv//1/8yufnYe/e0v//tvf/kfP896d7PH/waoAsp4Tyd//+t/+/tf/9ff//o/W8q0mXlASI0p3o7cHpPKgVC6JeIAPIWfHg4ZaLPgYQ6Sg4TQE9QFlfuIwlmzE6hV9dNhdxHTWhW76g9n7SXnOQj3xkGK8O8Jt3iY0zrBE/gwM21kiujgvSXi1ldJc62/Up4JsfRcFVsibljK0YYa0Sv6d+wnMgRQNNwu1Tp7rIoDk17zAkMVRUzOWxUdh9a1DUY8L4Q/sWhNLCby+EVWgTGSFFWMHq0Y/zPP0HJasUXSQoT3q4op9FVbFA9Kp4cqpgyXK47ryAgIf4pj0xtKlGeA6JgRgcZVa8g+Jb/EBYcrQkwMcpahvQjFB5xgWG6a45tHY5s5BXlOmSo6mqELJ/zvCatqt7MzVXvPUtatTohfqhpGg6g41yvJUsFhTFqDYbei/MMD9MxFEqwjZTSr1bBX6KzJKKJdWS1Y4C6EewvYXtvKRwCAa6DLMYZdqHVj17yaQOuzntf5HT5PZr321gNBRbIT+WSOGQv1KgnBpXK9Vv6hN+Op2bQxxURiRimrz05K0DwFl4bwWcaDELdicjXWew+L9VgTP4It9to7rmZXcTBS9xU8IMLMC9cmmr6JAUAfdOGZs9lrWqX3ct+Cr3lU0nLb4ATrsi0g11aOqopk67HqaTibWaKiimxWwiIJAwEaaVwM156I+CdpJJow6QeRbisH83IHevgVFcND7SsSLanczJAKn2GsJ3Kt23kYfSQHKYqcFAyTZGHLbcLTUd+mG8kSmGUlGGrkI7gwaOrUTbeo3OM/OEE4iJ7CO1aXuByit4ry2jiym6LN1G6czHVDf2DhDttmj6lVj312rBrOxG4B1imWg9eiuCP7hhfy1505CUAsIuYgQJGUnt06ttUfMxgFnQsxTq8P+E0WCW5mZYyRcBjQAxKwzDuCUdsivo3iiVcYuUtB+1uYnHApYkOxTSuZgfyojGYY3qiijxlF3/rsB+ndSjvgDkRv0e+tsndjmE4MIjmdISBfoH3tJmMaXpjsbI3OmItdKp0EZJwwZkOg6GRTPk3sqHvA1nXCb8jBxWc/kuutzm4AVRJMnSJBpkhXciKRHvMU7I5WpVJ7zKghS1+/ML013XU13iVEWx4BedmhjrQTBfjJdQxnbr03UyWlDwqn4MgXpNOitwq55yX1IAWfKrOM1M5lPhbDJJFnGe4k5FlutN/D8yWuzutFDDD5j9xVagrGhFAUlmBEAJQV1tfiFYKTTJEkU+qX9iY82XUf41uftFHXAdiuhrA/iRiUYrdYJJ/g7fbWPDnDOGRc17LFbscbp1e5n/ycUzwJRpXYpwgyftUvVuHo5NTFxn1qByrJVnr+in+KEwzoguX21VQPvFX1ertLMeZUKV8f1DoRmJSiWwyaFQJ/hxkiRKKcz1bE6J11Ritn4QqrICN31eA9woYsCeS9W7fOFSklxtnN3mBLb09BHSy5r1Cbk217FKE6MWohFE/NszX2qpwYLHyvheu6AZ1dBsEU8K7xBNtuQDFN8J6eDdHxEyxCXOFQlKafTtzNmg4WQb9B0acfBLilBUGvDk01QeJZRrYp0KpoV8KI8gkAZq/jh43PFEkM3+hf80OOMaS56ZXsyKBseC9/WUd92XLJc/gmI6n9JS9/oHcG7wkkj4Rhiqp3NeD93RaFDFeNdD8NzepIOrwdTWKNNjpJY1+4rSQZTBWA/ghFKY7PXWPdVHsKiEgV4lIoYXvA4RtcStiYP8/iW4uTGJ4bKZERdvZCtv0dXyj/LWAwak1+dgK5swVG/Eb1UkdlyNmY6Q0ClQYNNzGlUWNDsDiRI5M6qH6OtPFQ7a3CyL5Lw+swIbtfdd5tjF13anT3O5IIChwUpcAI01I0Fq6ROd+qJEQxSMcg4z21pYwkVxuq2eJNtlvHAI1KwpWsbw0RQ0kWrzKWkVFZCwiRyLBD+rFvTyZNGYnBJVAgpb/AkVVPGCKqyzW9NHVBMbViv32RZxtlQzEkcywAm1cgTciBVvBd/PIjEb+L/Uw1QoUXbwDLB4SKYnqldRCLQ6k+urmUXmrKSKl2oCavrPDToMOpbuaqhUhnXktVRVGtVTUpW1X/4/fB20urPL3RhXqWaK1G4rFO7qH3ntmeQrT3zCrJmsTM20Cyk3tFalzrjZwWPq4g6Ni1MtM5tDQdkZ7ungX6Vh0rtKS2I0JFUVRLxCIbi0zDoldqtwjwZeKa3l2rgYpfgfji1jZRKdxIaVLvk2In7v22Ipro6/+PkFaJ1aB17JPY9gkEKkNBs/TejczbW75B/geUFuugWmo6ZppSNfHKY4ncByYtYdOeXpP3qP64fAyV9C0QBIvEcaVIfEYWStGvmgVhxFSaaSO9nXUW9KXnmOapnwiwm+qh+PXgO3L3dwvpimQ1iFJhuNXsXQQEW8rKjg7GkrR0Rb3qwC+Qif6ArreczKsW2decjrpqp3Gyof2/brhk+/sbzLDWeaOWgPhrfcRjfML90GgUyacq07vvIKryjRTzLvK+SewTTJTfonlMrYxiBNiR6ZDxTeW20fCuuxcceVpeRYCqQ7COUDkLHrmGBfSeKdcThePqxRwpYqhBhLRkqkNSkBc4hII1rLaWxMkLCAloHsJ/24rlm8KRFjmT4OB1zyh+1+EuYR45ikxH6W6zvUUn3HSrzD90/kxrs3t/aPAaEd1rYBCzfE5l/lRcXwKAapVpmGCJGUmLbJqARvwC5cGp9GBCEk23ld8TFOj5aPK/nzzf2kld7Qgb0Rzmh0s54PRW5f2ziWJTmAbH4s+5MDzyMEVAHj8e9WpxQOgqh7W+ZgN/WINQIAj/PsavQPlQ0qotuZGK8A3EfLc2I/BMiTFhhYg8mQiThVtK44Rfp6PBAG3UoXga4hMmtlurjy0cjqZCcTI8JNVJpVpWsXI4+43XdNMA7rwRe4q5zmliF0nZtdBltjsZWpXa7lqt3phCw1SMi01tX58vnzOrrYOXmEj0QXF2W1QqRHihWzXW5F8iSLCLe6X85n7WVTV/P9fCXZB+97pZF/xtMqoucDWolvWh6XVDlnoZIWqZUvQwaoHeLcC3lRAgG18Me/0m/MgDPJF1q9wrwsemcupvlalstU0ctsgkSlK8kyRgfiKv/eZrTdyTmjAG30las99ODCs66bGUpnfL84VAA89VyIFSBU25qaEnCk6unmouUboFLampNz65MQgx3kGOK/wMdYUuC7jl9fIhrXwFVU0Q6K2O7hoRB8pfoO4aSN4DUxPLtbLkQuWx+lgrD60mltRH16JVD1OcBr+spnSwWdUixVidcpJLkbYP6EzS0zqF+G/hj8rwdanKuVqtls6I04pOrqpFMUMO7bxCKa2E7ZDzEnOFKhaIxRTvr9c21s1M+KC0tU6LEpMLqgxhUDC8cTHPFK2TWmI3gw9IvxxkJ3rwvjqWDhQOqCVPj9KrQCbUyXUjEmGL9F9Ts2XbRdt0BO719nuB08CqeDK+WIAmiimbKEgE8QXDpCmErc8CAJq1n+utCq9obMZEG3KZlMd7Ufc79iP6D1fmIEo5hmyBQnh2aS5TWCs8emhwoAhJt2dY0tTnyjHgC3F/wGxywEsNa9WMKQeCaTVL+/22lesaeTyKIANrolWDKGadn4jJ3T8Lsobp04fm6IZLpadmxjN7116TUzxzHlmwWgWFq1mvnu3OEHNqyQqvkLAInp5wNSHvo0Qd6GPqQmNctijaNWLbyYZdKVrb041luxNufpieUDp3aM9FqitfJjqXnSmokxM6jDhFBiud3TyjCpdp9rhO2Sdt41Y/ptIBRH+AL5DwmhAu8DaWKO8fNoKuUKMDdAoVVqMZhYy9QfmzLZE6mvbsDFhE85RelU3N/pE46m51AmY2kZhDH0k0eGtvIKp6NRHNWX5C4ov4MLG8AivC+y2oTQSvqeSceBoMg5Z2FRnXBgvCzFJ7xED/GOhzJMESdH0zXyNUrDLZCu/NgWf5tlHaz8c2ENJthFKriUiTluATGJ4Bk1WlgqWWwtSGtaZpGf3ZRm5TutUiRqylbds64GiL/ESLHuZnbXWfaFHEiuEjtMDHK5uQPzuyOyoAVKInEg1I2Jf0FHPsMeov5kzctbjAiWarhfxLspVz5DE7+yYde5kwWz0S/lACN1BrlLULGSj3GuGUjYFYNGKVF/dcwJw06tRmvLWSyEZrVLLJqL2Sihitl0YnpY5REZvOA1JQ3BYxXjk0o+flY1piZvXa7gtt1N7YVAYtArClTj4srjEQ8XNyljMUA3eFMvgaTq9P1c0irVeGVMFv1QUjOnWWoW6pi0gCOhlxyX4uHdXouqLW5KfiFGUqU98YVlJ0Y6rCpS1bOhRUlzsI3gBaSoM3aGMqtK8yxKyyrXJwhCot5pF6BUoLrPuohISm10TbedIPlLdODhEDkpY5VzdMGel/qtjC2lkSVMtE5lsBlLquIpAfFAbTazo4l0dGuLzlT7wHQRT15RGE26sdlIv3cob8NPWfQ3Pv0aKwzQo+jHvVibfswMcUBjGZr+2uCBCyaV+de+zEY8ceG3lsPLsHTR9SfTAmUSXHDYKyusND9FlYB+K1niemuzolMVWHPhXExkwvFgUvbVkO/+uLsrWXDGVE2Rh6cQNIVmv43xNVIE7WNJ+IocdMncWJOLoAbaX4AffBjvqSn3T3Y0RU2LKRHBTMwOD+QzfnOSG7VHNNFrfrbL0ztSmVV0Csmv5mty6T7frWpNJ76G3S5aLRSnlAcJOGM26D7mCAFHDUOiv4tX0mmsbJzrm5P91HN0ZNNO40bZtBAyJXsmLMhElts+KbLL8NVKYwt+KcmFgMQz/lg/AInt+WMkV7ty0JMChq+jpCFM/Imkp93Qq53O5EaDv1ArvKoG40JOjIkTZQoml8UGkZebancBFlWzIsYhAqelbs1E4lUsaieZgCSkCVRtYxx4u19HVdTtOCKegMY2v94jaNVnmGe4wrw3alRVANgvKR6/Ko3cqJ0SXsimo8nfVUAbsaWWZkzW+sKd3LMheOukFP1xbu0HR+hsigLLoT9ll9v4PmPxvt3zH6x0gh3xB9986tfuO112jMr3phCYdqitVoqjTA6Gv+VKaEl1f/4E1DYpiSN1V3BNmJNapRojRcPTUlFp2txa2LJeSWhMda6Lc5pN1sbCUIUBcCTCmS0NWxJFQJI796tuea1OrxWgQoUZXz2MXFRZ2/mGTYfhuCvAKhnjStACSXpJHpJiivjOhdxRnZ3wgSGQtifvpGh+4h/Mprvrr6gQbjUmZqjw1PUUP0mHo+Ph7azxj7b0XVEIalDC2RfHGKLUFB+H0ewL/4/wZuxBjbL6vwWuCkQVIuBqzTditFFWdgJGnHEJ9nDImDXX3LLv/DkL2bAcklILGjPwwmzMQ7KkNgS4vdGj0m6IT6AIuUvKZAXgiAEyLSpoC0pLKfy0FWMURSe54KA7gw2Ke3rqNBc7Ta32YJsJtFo4vWySnqa3S8p9ZCHxPSPMEcndD/ralBRClTtgbbQZuGQw0EVcSVNADX6naiTLbbiTfput5DJQokzmPzaLSOR6MniUwFtVob9RICWlipF2TvGUqfjG2e2Dk036kMmqC24g2lE5neWlwsUqlannUzHbZdeCr+2gypsUxR+AIQ/7e//HfAujCh9YwMm9V5FwyxZnjGwIzggNMkM8GVeVnD/emtDCbakCgG1noycl4tmJUAq8XErpNg3ZfX0+qCQv9RUNy6djIWlQ8L59i2PVYR4hh5eTLYH7uuEm5pNwTRMN77AdIkTD/N0CopxVFiH28Dgx0I77HDadMbrp25rfKrrB0feeaBV1Nb6YhIaiZZ685z0PTbak3FRxNTd9Ns6CEyk5wRCNOWPa5FoqkJfz/KvA/qntMoAVmFvcgok4uhOsuYGmnXxRT5ePRAGTcw7SNQz6OCxVmksu4ZADO6VIxAU0taeBJVbgrTDj+8yudOmVTXIQWw6fNmwbDssvvn1ujXaPTKofWrpxaNSRR+0m5OMhe7silVy1splnEC6tlH+OuKOKOC0k17IidIkH00s0/beeKs8AlKkofOQjLpl63dydSJdIWJJB5K9NXMY3sNLIfQ2lNusFb8A0xpskC+Om24HWDghrqpzat/CzCcohDWMPu+NksEJVIOkHQSNLdaDqWWodZEBLl/Gc81pwQxjST0yt91uyBaN+GfpqWUTMGWtRV93ncFXfclj+bq4i4dZCs/HeWeDBXEJT8WMs3LiWokAVKQ7sHHAo6ncd9r3/SMS4Zq1usWN6A2U6og8eaXmbe/gcpEXGugZtptbcc2SYsWzHetlbSdHvcPuTLFq1mvFVjT+CuhrF61d9FY6ZNuHlBr4K72rC92anDo2oauj41R4Zq07d+2L4zHApHGu30raHNZNN6TkEyJLeFbsbC+leKeWBI4kTMbN3EGq52+M1IZRgJ1c4tRhS6f99EBKVThpErawhR/61ujLF5Rr7um6+qVzVWk15JqvHh0zQRf8jC40ivr5+AqGyDwX6XEBNX5WZs1ZheHzqSRWtEwJCRFoIOCOhwpa5YoNJHAFmgEEz1Ft5Yq51i03VkHnKZ5hNKh0sCtMQsnhufC8KKv1iWVXJ6pzjnexovpLlbAj56iAyPdmIKnCiDYimw+0hRgeDTIS3+mFlb7bHx+TJvYSP1joUhW+rrL2Gl5GtO1xKICOR8aiReVIUmBZSMFxAs9ywaKlhpF6D8IGzHGcCts+aKasPx0YniL6T+lD6I0EoGyQKcrLqriEtwjEwgPg917yhLYPns4DpEqQDWLW0hhTyUliwtkPLCRPk571dATen+Ry57YwV19s0OXTVG6+embHozOjHSiBKxnwKDWlIhSP0Rd0TcpWCKxdl2m2xECkgN6TedP42vNt6LBXVwtXzAlT0xr0gQTzVEQPbHRqdFJ9bbBf3G9uCL2W/aKzkhO7QjMwXfyNIp25R4yA2zW7X2RSP1OYYzoX2QC5tXVE7VugVmkUsaGtcFz5KgLkLQpS4SVXq+xw4iLtQx3Ox2y4yPnxY1MHjCaMyehU6SN8acUPyypVlLLlP56TLirF7m6yBDIKg0otA7VVhkIUsTb0L6GQpwdchA9XetNbfBWibZTTXlo0bMl1CspeCoy3i+r14jdFterJsUtK9hOtVwM27H26iM33BktxMpp2LRX0qEjcYlKWMRKje1ER+MI3Aj7eiDce6lZ6THdcpzSGjqgPNCmlks2ZZ+zXbCn7Z7Znf7eeyN0iKV/1eFW+YWBOuhO/9V+H8v7onD2rExanc/QWXxtXoYmiFHm57bPjrUo+NTIonUDIpWM0LKXZRNr+9IaamXRCtoSiuLBWmNtPvDatI7greYSkscsC7zCU64RIv/WgzHFLqzjMJVVRicECfTVMU2e9EddSiTsNyZHpeGdWkl5DZHazu4titnvzEvORKbfqZn21/jclg5cbiQtX4yKbWnC5Upq+WJUbKQPF7Uar6072eg+q6kUh+nBBKWxok0VVAJbqiJC9WwWAc6uDCV0Z6VxZIDJpvU3K//0Ud1/Rixj4ZLa0ZxZpFyB2LQCPUtcKeU6wCR1Joae6W1GrhZN3XmhdMjPCR7rTAbj+C7AbPbBZzNVfnLn34TXTj1roKAww9cCjft0+a/OnyozHJKzloSkKQC8ALngkmpcqhSqAixZRecDEM367Pkqy9DiCkoRjJ+Z16BqQVj5mGvTdwa8CGlEemzL9P+PmduiHONrO6k1pUV8ot0fqqUvLR242J0q1tQ5yHKhWumwXIjdD9Q8l4Ce2mPw6CrhaZWyZ2IkJzqSLg3oEE3WBUekb5w6+Neh6zHmIZo3rbsLGY41AOx+dK2gEBYkKc6dehLDaHWNqjInpCRMiomo59ij04lp3WMBIFJ5PoY9EzSESO/tOV9iAuc8AOK6CfNYwKsht2Kr6heNSFdg4yph2x+4hUarEwNxsmmui5a7TUhYEffS0Upy8U9LirV068MqirONv6/UAXp3s3d5iq9BwJPwrg4O0FpqUYEgSYV52ZalUCRbyQvl8AlF2m/H0ad2Afkbe9URXjUTtRXZMWAt0tAxULs8hEiU4iGpSyaIe1oVRzNue4kWua27oO6/u4jQVO0xXw1mV46MeYEmLMy2ho105B7sdQ9SRJ7a8b51dLVPoYiBKT6a4kkllbyHL3swN8eIEnJcmg697mI49mJ6RfhQmJjtKy+8jYGfG7UsD+S91dERGiqKSahSluyropyjdbUqXcm+aopkoJr6eV9xk6BlLfp9Xz2YUFEcfuwrS865ahR1L97OwXSQlkhwRxsn+bc3d4AvV/BsNidXvNgQAqVBBElJBAiCO9Km15Smew3HxNpCpSwUrqlIUTihey6ZVIOi9sHZZef8o9J4pE+zyGJQ05cWefYLfKvcmNctl+3VeTOmf6nwImSAALYZvCumVIY7OTxPEUgbntq70YHYfXuOv5rad4Dswck9vrQ6zEwI6iBKiftBqNvO4bdf6FZHjM55MewuiwmWVXHyMfpqqia+5rY/29tIS0IBalRmFehYTl+Gt/2Eh5fWZ+iMoemirpDj+S95yejk1qm8PUdc7VHXxw9Y+dXeYZ7W17eujnvhOmQIuRhjXoLy7fb8aLvrQmAlnqg0u3oyw4Juk4M5keZgdGDZ3spLxPfQeyWnoPwv08DX9QUcpTwwawsLmU32U6cNM8D4C8+zwt3XorItVgPqGgQaNtWKV+jwaPNuL//FNovm9m6J8Xv4FAAP1Ieue2koF4lARbso3FiZ9ZPUA9He7gWt4EBmK+MtukxYrYwoWbSdy9bCPDogolsWKBKbfP6WUexaYV8tDe9ZAhK71iniAfyxtRv0H5PQeXZ8nX7fwcyaS+5Xzmr9fPwLZvVt5bYmbQntMys2P7zMuhbN0oEnPKdSUXbKy6SW7VqUrF1ohtmuW1qWvvWUmNzQTUU+WzSeu+Zb2BA3mCcfDZwlWWbaFLuFf5PjzoMmTN1+z9s/2x4rdnPiK86LH14/e99K5+kCU/hAB/ir1n77FGNBEZZUHDLBdHUtVhFvVCRTC7xderCCx9+lMPkfXZVPsd3iu4+C/uXZT0wl62Nxxgt1S1q0OkA2I08ttBrMC9dEQv9eHOCxGkaiGrU8NmxHr+jlmykb+IPhPx8lLyg+TzsiaywIMGB//2Y6/H0rOrJC5+3T8FjWmJY6hqlN4UZtlO0gf+7mdA4eQTrCAcxcUTK1htvr7VGRnEJENBZ2GGPTi6bBHNAnpvFybxO/jdeQbi+kgxqjMZOqDoPjV7k/NQDQrZiL4ai+2Pe1lPO12OS1XyKRHUaa0iXAe6oaWk4FyN4Q132tCTEJWvgCeamjvbvm670Loct7Do3QuJZbron+hwzRBwh30of227Dgl/QTGULHHcztxvNFCGszdrqKilnGkgvnM8kV9KbnqwTdxn3NtYPB32TsxAjNfOPAA9d18zLan2W2NHfvSYG6GkCNt36U+OsOD/+xkQpHyYXzLsTQLHEAlN5VqWPF+nTlXVtWY5jo53PdW1cz1btavNZAOP/DxqXmjgzbQYChAEGgjNu29V5sRO9uUfi5/JRg+hWMG7BduN3KfcqzNi2PDpleZeULdOYzXhnnTq13hiukUImKwjxy054WtwVIYzG87O2BFNfJ/wN7q3BJzqcAAA=="
  },
  "supertonic_samples.py": {
    "sha256": "fb4b25d31867306b9966682c83472d5e5c91c97c1627b5ee3085457874fe31d1",
    "gzip_base64": "H4sIAAAAAAAC/7V9+3PbRtLg7/wrJsh9taRNQnzoSS2zp9hK4m8T22fZ2auTVSgQGFCISACLh2xGp//9unsemAFASt7dS1VkEpjp6enp6dd0D7//7qAq8oNlnBzw5J5l2/I2TWY9x3GuSj8J/XWacHZVZTwv0yQO2IwV/iZb85x9ictb5rOwyv0yTpNRmqy3Q5bzqvCXa87iJOI5TwLOsrWfuL3eewLNZu5k8vIcXhelv16zpNpk0C1Nkq95lZTxhi8WE3cKrYasSKskjOI1h49BnG1ZlObs1ftPbu8n/PDp9YUGY/QfrbLqOqhCfwh/kuRGwptSW+6HLI3M5ue9quCsvOVs45fBbZysBOTJ1D19yYAELKhev33LzpjsUAAgeM7iIl37JQ8Z0C3O02TDk9LtvU6/JOsURkGIQYqkKjlLM56wd2/f/m/2hcer27JgSw6T4QgzgSHnPcZuIxaqzoUkOB+lRTHyc8Drnh8UehlGM/YZujA2GuX8Pi6A/sz3o+CYz6aHk2P/6Ozw8Hi8xC8zHgWHZ+FJxA8DfhweHkOfdRr461EY56wG6flFwcui94pW/LYss2J+cLCCRa6WLkzk4CmcXGgr6HXLgzuWVmVvws9Ozs742fEh9w+DceiHJ0HIz85my+OjYBacTmb+yXJ2MmN9pFbJv5YsAkqWHKDEQOdNluZA4QFwz+VXYjsklOBRE3XBkoULLIIEyVJ2kPnl7UGZGvhpiomJ0vQ7mkk6QLv7NAbu/W2KBPOBLWAJFQhC1bkAHKIoDmJY+fWWAeFWLOSbFPhM7AlWcJwMQClTmF5RElcUGefBrd44rgMAYV/0esDaOG1sE/LIr9aCnsDfsF7Qnf3J85T9DK1+/3Dxm8veZdjfX0P/ELgAhkG2n48ZcHTRi9IqFzz3QTCu4Gt4hexSiP0bwCee3wMi97BReT6Sr5kPe9dngZ9Bw9u04L2i2iByS75Ov8CAG//r6D73N6PVkm3ipEKC4ehAfgTIR5ulyz7ewhe2jjcxEDRI70FqENxi2EvSknF4sGVhDmPnBwGu+9fy4Eua3xWZD5OR80YSCdSBMLncNvyfVQwjMf41W8MClD1YVmj+ZVQlACyOYh4KdJZVuOLlOdtwv6hyXAiQF0Q/6FvyHMiHYssXZP+7v1qtOSLHl2l6BxsSuIpdwAyBKrC6IHYiHwRSKBDK8vQ+DmFWQBiQCDzP0/ycVQlwk+Bctsr97LYH2x/WOs0LkDFb3PMggGg8hT5wD8w/ileAYogiToPG0TPc4/xLDzpK7pCsg2/COADIQ4YE1bsIeC6FzkMW4Wpt4PN6iMjfp/TcZW9gp6NcTmA4sXsAAWATkG3Bugq5GCcl7gZ6Xb3++18KoPomvYdvIEAVT4YMZWBS9jI/DIFCQz0llvnIhkNi4RCEYA5sEhclbERYOBwSmrvsSmwGIGCC7MBg63AQwzzsAboIFIaAzbPx7ziRGACAnEZs85wHpct+w8kVNIpm7U0FW20JnAeyFPpHcV6U570gXUOb0s/LKqP2kq5iCsEWeZOh9KCVK3jmA5X5euuy942GcdGT/BQKwsNqAYsIVGFcsUl8wD+H+VZrP2e0AWHFBM+TEgPJEjfoHHKeDWHXrTeg8oCP0wpZBhYnw21V4o6CLqRNSMpmaQxrdgvcm6Q92BtC447+cfE7C0AcxSS1aMnPQTYUZbphJNUKtYOQpv46B7W4JW0FbIXKuyi3sA3+++rdW5h8lQRlJXS8/wVpFNxWyZ0k+S1Qi3HYY7AxQoIbVrB3N7CoPkhTmBNhTq17fwC6iFMAHHJXnNPcRTdaDu7fISVLWFfJ3CRKJM1sdnXRROn1QF1smOdFVQmL4XlSZQA4WBVCuej11LN8BesBkkx+B6rdruOl+ir+gQduVcZr9fSPIk3UZ+C7W/U5LdSnXAMsqiVgD8TV74qt/ojrL7AN/dIP1qhllIarH4kWqJQAEfX2PY5LL8pthuSTzy+Sba939e7Th1eX3ofL399cvXn3li2Y81yt6/T+cfnm518+Xlm9n2tGqN7eLxdXv1xeQdcHUo6OEk6eFk4umlvOnDnBjC/PJoeTw/Aoik79E386O1sCyDM+O+QnfLqEh6eTw3F4OjuBFsvJ4cnkdDkJjmdns+lRuHSGYgiUcZ5k7Br4CdhU4RH3T4PZZHJ2cnzGT/3jYHJ4ehwc8jA49mfL4HQGNDg+OZ5OgtPTyXK5PDk5XJ6djadRpIDfc8TZk5LGwP70dOYHp8en3B9PT454NAaaTo6mhwD6MJocL2fBeHZyDDicBLPp2I+Ol0ez6Gh5cjqGJ8eTQA+Q2oiPT49CfnIchqf8NDwCDMPjwD8+Pp0eH48n0fHk6Gx2Np2cRmdjfjQ5i07GHCY2O+bhFJb3cOkD3MderwcmAyMG94DTiz78uZ+D7i3Ka5CpN+z/srdo2C3onwEb/aA3hPvWB6sWVe6cEKSHObTUDS7yVYXm7Xt606dW+B/s3CCPyQpZeF6YBp43RCMd6Aby3iOWXmgoH/wvr+sOv/B19pNqShAHxuguSH6cBw1bD+iAkUF60hniZuAL3BtDdguwFg6KK2U4gXB59esbBtqqgm2GlkcO2hQFzuac3fFtgTYSaGpYhwJUCS8cAwVSoQsTkw3IFzQUPP4V1GMBBou3ytMq69cdbIwdYSM6T7wfoXdjzmU3DRxh2XZN/H1MgrHW1rXNCy5UjnYhKguwyJ3n0bi2kLuGUy4OyuIEDMRbUIBovflCuCNbH5BAJ1XjkS4pDpwn1tiRBjeMKNdw4fw2dfQcSY2wpQ8mNbDrORPaExXmgakpnzlDGmpEqHVN8VIZMYZONLowYjXDn1S4Pz1H9CTMKfLkmRij1xAqXMGs88saysQdK8x/AfdSGKSRD+iB3gfrD+1hYWkEPtpWhgEJ+DwDa4CUFWpwMDmMoY/VyD+hzbPiCVe+D/Y5r4cClKrEjyIQrzCR500avQyyNuAvNJUSlLaUwsQQRgKj2XhcPxSo/YiBBLLp/lnBELhHhA1zzv4Om99PDv7bz/wEnRXleAG2k+lYDriXNuUtmk/d1Dnc2zPppuj+lTC4wO42ne0fTbiIJu8FWaX3F3xGBwF9put5DNT6enOOplsGy1WQt+OhjfDjr5fea7AWXl1ePVeW7HHLjPX0yearlcWPabqGhVEu7gW9bi/2T/664M3lvgjukvTLmqzS8tYXftG7Dx9rpxb+RwuRrSqw8cHvBDsY3dw1H0kLTniIAY/RS3mSCyz9ZDvBz2LZk+PT5hxe8wDMjzX77UcmgYUYUimkFmPk3ZMTKjzq53Aq9M+qsi3WFRr4re+04yrO4JkiipqP0GvqZNGxIV7JPBaOJ7Y/ZyJqwRIRijg8nIzH7Jc/nyGbyM8cbYodsnHsjltSvQCti1GZJS+/cPKlyKHBCOYzuRp5a4SuCw1sL7FAoC2X3JZcepUWcSJcIAZsykE4C1zIozrXHlBh+ehIqzCmKCt4EOVzVn4FunmE9vMuEj1v1gnaa+v4T9gm4Kv9+7v3Y161Nu8H4UDiAOAWLsF0Y2P37BQ9ax+UbMjev/oNHcgMXaFdJPqmHWuEsnYQ6KjmIbUvf/5xiEu2SYuSHZ0bO5EJ0cZKGAsk3NNro0IaFEAHBJ5HS1sC7h3gi38/At8UJwbgg1sKACyuHSCkNzmGR/Rpeoiffvr13cVH58ZQErKVGIH7OUbLvNo6Fh4Hytuk9jtE4zgS7V1hss+by18AFPSvXTQmi77Z1kWV6qGF3Cc3D5Z64VRlNDpVskgOgEI8LugUADZ0X0EG/MHzHNRDGuShKF1fexIiWCTtV4YuEhl36fIP0HzOoMHiiPOD+OiC91PO5XM6mpAfAYwcypOdHjUU2NtIKoAC1qxGF9wx+i6bWzOUPfZMJXI+SaiEOXo3c/ZQUBirL7sPHo2pIK7QaijcI8RXIeLGJd8U/QbhAAtoju0eHNwFyCjSDXu0W3bgpnBiD/Dnu/yRQn2waj7OuETtOrBASCIuFMGvodtNEx2JeCF8WgqSqzXR5hsOgq+fh6GAiJRBPB+VgbDkLKnW6waOJga7h4FWBmv6ct/29cb2rkowPlEGqj29f88PBu0x8D9ABcTigiRXn/AaUKQeYLTa8zVgJWmF7emYJSm/CTC0fxZcEqLfABnYi8wF6vfYMUTB90MzqE0whxhNHnStCy4a9eoG2GCQN4kYgRiFZqf5pM0YJhHICX6aUcp8242H2pa0B/SWcA2qtan0NeBZyfq/4+tLnMCQfYT29HGA5xzQ4ttn3dgeIF8AjDl7LewpeLJzunpGIoZH4Y8bGZTqWcxUQ6NWzwXZhianBdLGU037L15o0UvtUHM1dVpbm+G3Qk9QxNEWC+OxxpTezXvdmucKfKo42sJCwHqut6RzKIZAkIHERmxoKA9kMLBixbccjRWNToeeSh6iKsKHIozj4UGnfLULJXlqivxqnY/69VmbctmRBzACg/u176AL66jogFO7wg4YVR6Zsp7w3A1qtNU2GEp+WeZEyCFBH+DEywFOpeMl+yub7FWID9jsUZ8D+SwDS5scC4DKVzxvkK+BLcCfjnfTqhGVqIcpGUht+DwdN+Cj1w5A98CkFgpQkiYJX5EnVANCmuERhBsXUZyAohZ8R3EhIhQ2GLvTI/bXBatf4bdDd9/ImTm0cokIEPLDIesDM+Z8Iw6uQ3GGGW8yH/jjnxWIiHI7eBJLctK8TTHQzKme7CeLcu40fgIoYfYtVEJ/ykM/S+FAxNKUMl/jw32rb7p+bbIRZub670AIPTIvXAp6LIvGwx8w+LQbBenOdRFFITICAPRg8gxkkP3RA/JWS5M4rPkSSXO0f1/onACJGkoJcJ6ObkByKc+psTlk7sJ3C0YRKbm2eLLmRmB7UVpOP3cwsNL/23z+OXw5+JszNPsOduMkgSt0OsJcz8RGSFQMZnl1MMtDQJ7w+HbgYIkpx/ATVVpEMxIlMhToMJSCUH9dHAHNzs1cAccGuj/IRjPYpLDaMPE6/YHyAfAQBGep4dm0oNlJB9vbLPdv1Ubga78w20nqznFfsIk3Ho/xf/bDos2Wxnv8/xtxBJl9b+mIljPflOYUafAwbqXZ4xRx04LXaADPJmfTvVgZcTNNtTEuDwJ1XdEdg2G1Bt7patajCJNt0dCfhu86FGZKr228yhiBaXMZtiwp76TRdr86NkattbK0ex5sQA2j8j/sQen1MuD2Gr4EOg6W8zT4FybnC3NtKdCQc8p5WeUJ8Yg8sF3iyYSfbz3Me0lW5W3/S5qHc3Jc8IxW+2aO4xiJGBj2kskeQxEWo7SKdAP7nF0mq3Vc3IJaWWLeikiDOEBxH/vrwsW8CYQIQ8RZxtFxwjHdnB70nc/O52o6noT09+wvg5vHmv1BJBccU/9AHl9/9zdsMj3+XM3G4+nnKorGE/w7iW5e/g9nqAcwyCcJMFXwVBOXJ2GBCWl9xzVNRXMKGDNRzUHa8bwvfEZbFm5y14hLykdF+1n7UdjumeVp1HpYtNv90X5UlK1H9+0xeRm0n7mr1rPY5a1nvrtpI2w/q51oufpAQ+TIfkO1glq9vhj9H3/0581nd/DSXLtWtM1aEr2bBHh7n1iLLb9MWlw0PJ8Tpx3S3xmxUUBstKS/foOZKA7AxnL/CKOdAoX4hzbOUKjWufAgYBPpJAi9lf7OeUYgg5KhDQ9bmOdiM2K+YywyriLMGhD7SqTxqnTO4iBYU3ab3kwwKxrU9kxyPwZca1e877yi6H49pOikhIbSQLbBRtkDYnMO9sH+qLL/QGzzTVaqswxxumHmgizY9Y1WJZjmRmmKuJvEWGBolOs44VYkEIWEdJJFe9GubyuQZNtf84SkGJqxYn44DD7BEQhMQ6K25hI5F2Cq3IEdi/EMdDro8JqIJ0A+0D+mshDLRuBt6CBbgCVIugL6YIyOrdct5NrREbDRwjhE9bxQgF6yepovxaaCcTqDTXVvSY/u8AtsK/+u9UYMN8RJwOAalHjwkk2a4Sd8/FeNWzHYMxlczOtOVMgwRuhEHPo21DTiSbXBM34uRriew5A3Axy7W5lRjGTaGudmV1yuxq+bTs/A///PHCadY93sXfIdc2isZnE9mthwxJ51fRB5Sdh3mONi7qSFrc1raneKFtBgfmPKXQFPCk3t7fb9KozTOWYSDplhtJLoHLJNvAaZwYMUpOJchGVJnkJzLUmvqgyP4AoGIi2UOxRRFfnlaQXiK483G0oNlrm99Flm92IOsJahIq2RKkIwUJlkPTmFAkP6C3jg+oWf5/5WID5kIZ3WwXNCbjYduEGabaVIEgnNC0xP7+e4rH3TLn9hTY8dsAkY2QMSFH0x4oAdHLCpFsUC3A+2fN9kMEB/7B6xEcO/LxDJIC368A9IUMqqQ/8XvmbxUMAYDAYwDwriGpgbNhIOfj2ntjfsxYKGab4e0eu5fn89nysekksuGqrEQJFW66Fbaa26DDnI5cVsZnnES03nJNlo0csKSHdNffBs7ZoULXW6qfXqRZbJgOZqnS7B3kDw5wxz/JIKqY6ZeMVtmoLjC6uAmUrlrZ/QeS4l5/1HmQFWoz8Zsxfwj5wmLPN0rLIZYH6eLMNZiJngeoBfif9gTEaygWxfBJiILdiJEq7oWPrAhDMQMsykoM5sVkOR+QL9zdnAGhL09vINDfvWMcB4dZ60MzfhDxvNCS40oX+bL/0I0zQtUHvpIPqrZNPi1p8eHfcxW3lOScrEJ8AXgh3CeIUVLwuVaO3K9oKaVH2CXV3MZu87+dKh0wmUEf5mbp1VLtdpcIdiOwZ0+2t/swz9uWxJJ8WwytNDjALAP7CFl47T0HsCF7fKUNr2CZ7lkcn3t/yr+ARIijni0bSnapL6GCQ3pqrlIDpePon0a2eFGRnMGb0SRiv1AZwcMJpH5Dni218uL147arfK+q2FkUDu5lXSl1CHIntTJEtg8hKluotcHvkQrTb6OAA7MYRXylys/TY5yHcL1sgYb1qUsljosh28ijCBhrL3VFqpBvvQAPp4zlagyh9Ug0eXvVJVYWTKZSJlVb13mxGoMM7LbZskutVuOoNTX1Z0BDEagQgJ+Bq2vviKf7PtAR5h4ynP1rmpN4RB4/pZB63rVB9Fc5lqIimvKE4T2GW176axhR8VdlCtHnoMCTCmKJlQUT6jxhLzF9ClqCujDCor/m0SGZP78NjUqnzAU4MAWX5Dp1meqsGyEsM0xCET5XaIuIP/1MTVzEfDGGdT+J1yPnjHuVQHgZxXIkRKlZFdSc4aGzHkJg0rktSNiYnnYmqIQx//1JSQKLn8K8xftO2LfyxRIR6BePifumqjDyD/5InYgT16xMjRe7/2ZeKGdk71Ny9GowprOPSTjV/c1Y9U4mz9ROTGqay8uT6IFyVYMDHM4xePpaEHU6lkLYYoJxMHDfOO3P+hKp/yhGVhe8+AQq3lfxYHaSFH4xRGptxKXXwVxsUfVJaExKcnaUTc6K9WOQaCuRVbdYVep5irDAi/eV3QySPKBsy8BpM2Tvw1cL/PVH4m0krUWyFXYU0kpQ+KAkiXQK6yytvwjSccRlGiZ4TehzINRXv3YvQRdEnzLYDzV7J20VUT74peL2T0uhXkunZevf90qVbgvVwAKfVxW5jkfipucKGOFnG2VNSpCjXjonlCKxDzhAO0wHXsGwhLr92Zw96dDK4nwu2BrxTTNSYm4yzkmoqDBAPWnug7JjM9FcWXh+6wFnJ1FmoMMLktwtQkrxs/QSuMgyBJQI8Dh0i4FOTHcDbL03QjKsmNcliVRWtt9dq57DjFaa+sHYt7aDl+jlqYEOwsc5GG7aYW7zpzc/odrcVbrGdNQo+Kj/lqi6VFd1ewvS+KD4oaTkdnqpIHYif3nr9epZ4IzWHvXy4/fXhz9fHNqye60UEuMIQu3aWqpq5OYeqhg+ahMS4SGTxhxmGPSVcPhF1Gs2kXyMf668AwbTu3nXh/IwUjZbYAlTzFa7V8lE+keySWSYYVKepVx+ZrxYUnAPeckvqoswscZ4CUuZPKeFow047pFhJkQtgCRmzI613Mx/aJm/r8SmAKZqBCp95JoAhI5oEz5HxKNLpaieia5EIkAhE5MBVIAH08r6f4oD49GpZdPaXvWjLTHP8lvBWb84tf1DsZ74MAcd1Zfy0PUSmvGw9II7wTYumDwQkKAottnX0GhhwYbX6hvC+TVazWFsT+uwQFCqbeJ6Uopj6nwhlVsxzgaKjhvoCMwfK3kSqDSVaCaUSgWErniHkeBsw9D5htHYmT7C6VbObMtN3hxivjSgtsAI96ugkO4yYZec32Q5n4RCdS1ot6rRc7zQgRLLHKQO3zgq5TXnvNv8faajBizdNteacHLPLb39+8fnOhLt4AdzG4g3WyQwd+0gCYrpEjiD9Ux2WVhAhvuUUboEpyLm7u0HX+1hUejVgeGOF0fgoURf8B+qJPGK7XhdMRXYVWrtmmD44AFZRuF047WXDHZlZHrAgMRYl/D0yP+f6mUGmP/ZRzsV9/yQItNZTL3nzD3SpO90Dk5Xdfp3LOwlTOUwyDGwhvPsA2aM99ueVY2F+mQAFYbbc9xKDBy+LijkUz+c2FvZuu77lxUoFTory4hdUV/BcqyrXhBtHKzlTv6+7QoSwLF985g+clrRNIMxAprCo10rXjc+fm2jFaODeN3uLQq4j/3NUZqyS9uhUAAOMrNssezMldwxzW2E1mwIE9jBFdL/KRb52b3g56S7cjjDctNCTAukWNwnNHtQRJv0m1YZMSwyZOA5Q4faojGrLZ+GQK5u7hYefB0/4d49CdC37oZ3hg1+VfW9cjqXs8xAFpGwGbjRskxVAZnWtQAuNchdgMfhP5j2YSpiV/H02qmSCBGFbDJ+ngkJche8utSufF5sy172SRQN5xZFYcqFt9Fo2Imk5erTfpoKWI0OFLca+qbu4nGAj08Hv1ro8RIHNXVqKBsK9BeIgdaoDGMl4P449KXhjFyhSksMWCWcqMcQ5M+9C9HgX4xhY3R6i/dAijurk5RaKTMWwfI+79BujBTVvf0l1FUoFoaBe/X7z59QKrN3+9ePvz1ZPHr5+MO2wQYIWG2YOG/10O1p7K/GEPu4Yxj2fTjGQzqrQrYSWL3Jyib7dxY4xkeWnmgbHjyYxitUjya6sDz9sdJnar2ohBC05iopUg3iLjvvvw0bu6/F+fLt9+fHPxa2N9VHhj0fDsxEZFYG9UNE7Or8mUMq8ZIQG2NPcFojasDeuFbXsNWgfVnZu+I99DtR3WcYLEnkh3qc/TjpGVeCnzsRsW2GtRnIhhob+A5KzKFAQHSAZtroN/neMxEYZ3ZU224WZQYlXhNoC+ScIYelf+Gq0ETFKPV3Trgb5bKecbXyTElSrkVF9GpIbGaJbbUIPCZ5MllR76Dug6NHeoJk3I6f4O4/337B9+vqH6Uft+JukxyvSY+E9dCU/3SIQiNrtpXkRU4xeHwB0Y6lKGipaG/WuHxqwyF8v1rvXWbBoLar2vd17McmOH1vE/dG2H7MFRgUrwvgkVR4ieMMMTJS2K3DAbyktZEFd4h/88GnaD9nxEcM0TtyyRNCNXuj5RrAONphLbH3AzQjYdURc66cJ4CsWrltuS43TGO4IuIlzk6XRcjDt0Oda02PP6Rrb6srKiK4yht5EinH7QCGnUhw8YPZNiwlh90el6fHNtZAx0Tn7HxGlpe8+a99u0URSv3543Eoqtqvn6XrVmehrNyQhlebB3taRRCJoTv24Ewm6eAqhjzM8DBzYpurDWRhl0jYHn1rdo3qt4pgJfs+eecGcDog5peEQ9RXsFUlvJT0Za2+mKz2Iy3VDRBYb2SFRBx/YOkqZoR2hL9e8PvkHh7ArhCWLD+iiJriUx8qKu3Tcvx6Nd9wWlKOayN6/dq1VDKxMT6SlvcEOfHwfAMwxQFymeIpgJ+uqYwN4JuLfOKX1exxDe/v7br6zka4AI2s3KwtTir6VDtAS0T8Ecx/nAsQAZy7/qK2hkr5HMQVOHfvo2UV/k/yl1glfFRWiThDqtQtkG4qyIqjPAL5k1rADhRSmpk2QuijnweyctL2soIQ1U+oXq0UproUN//14BrdWSdR2W0EVK/YhxYHXEh8cByL1mCAVgYmZBRhEmxFBO7UXTQ/w3AiaR87tAkjUoL89lQ9g2Gg8rDroXoceukIbmFYwwygChPj2sD+T0+aKhLdVRii08jLq2br1KDgMVd9/h9VLOH77zSFEYCW8B3uu4wSBypMm0zqWkiOi8iV2da6pYT+SIxUkzd1em7TZrz/dYQAQAjR97Ljc204HZ02a6b7CF/vP2EC10k5UNNI3UJng6YAfGYlENXnMPQCtrD+giLgVOl3shPKxi6lMT+AZoYInK+Fmboy4KlmQzrpKCLQAfwTF0mu6AuAUWD3RVxSvayOJKUSkm6lukf5DXeBD/0a4f1aaVlOikTWwbXhyJe0Q0PGBBar5oxdp03p2mS1IeHw66QKmQlgkYaNXwyMTOXrRja3RoiFhQHtWgAxf2shXMG7GJXmxDTHQg3RSs9jfkNk/jpuawc8DWiE32UgD+yias1h3AasZA/0Y0+rVa+zIdSc1jLDldOIDXFINc4Xj5Tc6jii7jxTu4Ez8IKiKnVHr7osRaSqm8Wi2m+jT5YS1thrSnRDKpSo0ttLIbNC1vglqL7mKbAIsjKaUAx/fzOidkyPJkJU8YQZ6tMPBJX+20MjK+sloq1AXymyWn3Nm2YGtfjfk8944uaVfftWDDEK0l2eBBQ7TVHRtOnyng1CQN60/bGEAKt6C75vPQE8mT3ZYGjWTluuxPpJVaA16lWMqAMCedUDpSR+s0L/JvRLopSWRD2VDd+82+3qjvsBWquxwthX6j917Dy7CROu8k3aesWk8fujdfksbF1msYWsPutoK3NktopzlwR9M97LMHdidL7eggl9BQsDsagnzIsSXSG1oaCwkPulZvF4bkq0ko9KXdsOHOtJR8xz74j9rEyvD0DAG66GD5tjHa22dSd4D9Fw5SaitaJFQqY9owndl9TD+qII9Xcj4yLpQ05vTQgdHjrpOVDqJ/z96pPEI6zyfPLwcbm64453UkD+0ceXc5XSSOt+23dZSqZOiZRk9cCN1Vp8nhvemYqk7uA+kyutFelNnqS+lwPFHdi+EVukVMOXxRTDcmuU31Uxd0ACmvx0Mm95CdMHgzZDuO0dpXJOiyAWnmeYJAClS/IxVxqPITjQoS2Us/sWtaJfYNYMCbRjewWgyw2mIxnklU5aXy3CMSIyGkgn0mVlrrYgq/MeJiYbVvkh7G6ckTro34lRIXA9PAJvp+cIlXlq630kSI72NxoEV3MayCsG+MaOE3aJQE1KBwhlZTJIuEbE3aeK7VpSNlHd4HBzaOukROUPJLDp6Ch0daRm7/kC4pnzcitY3QRclxzn6+pTJBmBqe/3v0IxlfCZYrPsOiOm65yfT1yLKbK4Yml1ATmg7ew2qTFX1EASvfMG7j+UDtWF4AStssKRfToQgJeYmfyEnhWJ8xAbx5Jm8kRqV0BrnGWh2Ny5BmoGiCJwvPuFJbs7Z5oVDzJiF5vbRxmVDErJuFdM09ZXzZtwvtTjIYAd851ghm4WizoPSpBMo4QQndVU6qKs1EoZQLMjFEhzJ3Pl9h5TB2GVjDTehiBNENLxAxgwkdR4+6yKC08jgnY9eFrljmWYr0qBEIbowYwEsCLmsO6LOuWe/Om1L386sfGKLKk0jszu/ZB3FEy1L1+ypilzElyQO8TVmG3BpHOyllMgGHYpZdrbvcvfdHtB6C2qW8gVb+lyVgANMk/SfsyZ8OxxPNda5QrpT4srmDv7Dx0P4pZK0I/wr866V3MltdGMpJHAGVrfPwGgyefxu5/6q1cegdRzYMlwYp2oUPPwGtL+mdWu13whRQv0EhOs7ZgwWPykfSFDcDCFpe3DLrMluZwy9+2cOjf5H7wRJxAetIJBLrAztOyX3QQGT5UQpCA0J3b7yk0Ryjp/zJJ4cUyYILObRLkT3aJ+Kt8Fz3DFsPYtYByOwhCdTgH5VuqjVDx7UkFK6vdai41wKTbev4h6iZtC9remFrFlE02VGcQKU7m36HFSJqzte+uLwECYMi2h77BaMicvkWdb6VQWsM8kwDxVKIHQpWcVtTv1mn/Q6ZinhWYGa7GIF+R5RFeaqqCZo26qKMtjJJxmzc/DmOVmtP5AOh/yUWXaTnGO2MhJG6scwiUoxSZ48YHeV9ns3znzu+pcCzvqQx6rhlURTEkbKq7wm0Lxi992XO6KB1DGT4TOIGvznpEHPyINHFfoB3JN6Nl8bCondnLHPdxlhvpF39zVw7tbVFQTCSTT0xUfHzjac3rG6qn5gVlpo11TXa8xYHdzfXYFsMf7ADebF9bJ5HtMxdZTTX0e8NL2/TkM647R8GGOm4OO3RYV13Vbs9I+n2qN+FwpMy5dqMMAUx1LXfxiZ58ULyoZ0KYPEiFasDVtdPVVRIfslcm2OazGFQpCGNOvpYjjJ1sZ50lV+oH8OyBmo63HLaRspiG5JwP02v1pNE/jbQqDP2zvSxlVmUaXmsXxmn7Q7+9hiuyI2qARaaywp8RzqUqzbE3PDUO3l47h5Hj6xg/Va7R9lM/+JTMTC4KFpXxa1ZftmND6XI1LjUmvYFaa65O4XRN2A5mr8Nhpcnq60/d2eI37m+ROWhVkyPT+JjeFKWPTPUCse+WExdBe5RcknT1RRHbRUYnIWnwpLULZFpDFJfvpA3klKAEXSrjEeKjqHwWMbGOaMOj7TPoRNAhOpP687GVdYa3ksz2y7K9RXORpMDE3HddumjF+x878jk4NkYPigA5LqN8BU+H3U0MGM8rSOYyPmc/1xHj64fYLDHG/aAPyxQw5gfIw/8FzBgjezjwYOB7CPyxcpmP3mHx8JpPGuygXGkq28hqWPBYvEMI5kuAV0YV4a+FF3sc9vmVaLdTtyVuKtz5yV8wuLbYe9HpsH/oK9SmY9n4aOHGDzgn0f3i39v1ffsMP53OwCZ/s0jbYATCXZZ0oR1zANx+YuRAUVnIUnmAmXDdKPuAffgeR9RHbSPoGXtnbZF6zthxD5q3YFu9BK3OpjateOWcUJTHTXpJApbJ7ejz43jKguIsuDq46VMnSbVB0kGBBFtXDRjY3RFSYIaK6Hp0hiDp2xkDXAo47femt9jhcai43YRmSUqr92Q3+ybMWqolPCJfLgsyO4nCGj3N3UCno7WDZo3k5nn3KJFfdJtgDVq37ROQredkPgBRKM7nvwrwW1D1sgaNWVMHQikRmI5ZA4QHZlro2/R1n90V2PlrxcPNfLwkPCE5vjP3D39r51h773BOAq/0fY1dl8k4m9mDEwupml1MvlbEgtaUwDhyQfGgiQRsh0AxE81PLumA9+50qiiLtJBFaPQA0yvSYDHRDFHy3q28Zo0tmwdVqySdZzc9eWvuNrBj30lEJfKAsEf1MT0P57bJxT272gZiex7Aor18aQwAa6FbXWjNvlTNi/9AtSc9ECHBUk3qguXzS43GHQlvGqWNZyOPZGPzhEFlxr2qbmuu5vr8Yzmu/wb3dkSPNDX+t5Vb6wcHdounlL66FrR1fitHk97J7b1bJzjPdPWE5YKGCfJlY8/6iSKKpJHnWWBqsUOf9WLCCanYWNYhwN4I2KMpaWYhel5lCbteRi39jyZK239zoDg96stWIkb0Mhln0LcAxVpEb8i8IbCjvJnBN5dyQ/mbwuYG2ZoXtfyiko4ZYXOjl8fULS4/PDh3Qf1mwIwQ+DhRbFFCxZ2nPELDi2cJwNx8AIde/8PkSoXScp+AAA="
  },
  "audio_quality_check.py": {
    "sha256": "fc4f16414bac0556db53dd4a378c13c50032472d7d094f6d9c1614d809369003",
    "gzip_base64": "H4sIAAAAAAAC/7U77ZLbuJH/9RQIUxWTtsT5yO5WVl465djjxLex1+exN3WlKCyMCEkYUySXIGcs+6bqXuNe757kuhsACX5IHiebqakZiWh0N/oL3Q3wt785qVV5ciWzE5HdsGJfbfPs9xPP894KnszyLN2zZ2/es9VWrD4ots5L9u7dJSvFKi8TmW0U41nC8qKSecZTJrNEFAL+ZBV7evk2nExeZqriacqyelfsCVjldZasZSpC9gLQzWZclVNWbQWrhKqEg83iWkkBdEoxnzBWyAKoaJxrDvDl7HYrVSHKKDoLz8MztqpKnqmUV+I8ir4JvwtPGb+BwW/Ds/AUMdAaGa8Tmce/1DyV1T6m9YXA4WxWil9qYETNrhWA2W8hfZvN8roq6kqPmcl2CNYxmXThJfDNUqkqlq9ZfnUtVpVit7LaGvIoBhJKJT5WU1aktWqWP7nOr2KZTNkuT0Q6ZUoI+ELiBmGVTKKU5XoPWmBrKdJEgUABH4mKZeIGYGCqhLEknFzCSLYSJ4Cq4ierVBYFTrzlZabVCHP4Kq9VJVdsK+pS4ic1ZVleses62eyAGjC3nsisEmkqN/JK4uoZ6DDjVV3yNBMKeAC9s1uwDibKMi815lW+KzigzDP4vuGoQFjJxDUXNKlNJj+J8oFipMJVKYtKM1CUOQoQfneyQhtRRf5BZERHhZM3dbaqao5yO1lxJYim3GR5KZLHaHpXotSWusozwL1CSCOnHEnflog2CyfvwAxbTvCjkDcg0iyHz2tRogxnqCxkaVdUIXv2/vlT1PNWJrAOIgISmNQKZqHngLj+ELJXqMNZmnN0GhQZGOyMtI/iWnOZ1iVMKEUFwmkVUea3sLzJ0zcv52xbVYWan5xswHzqqxBEenL5X5fv3j59fdJ1BPTeyWQN/LE4XtegGhHHTO6KvESpw4pJUmoysc/KDWhHCfsdTdd+3vFqaz/nyn4qG1i1bx5Wctc8rjO5giUnvOKakwLwpPLKsvEG0dJAtSdDNM+fZvvJZJKINZh0JisRa+X5NzytxRyHAzZ7wtYgyYr9N3udZxQVGIM1/yhE0QYPR4vAGJA4Ad2v0WlWgu1Azsgb+4/Ln17PFF+LEIWGiOSaETHUaYsef0A5dZnRQ3qmWWOR5kazGEwcSAMAGFGKoVR6Tb5+HjCRgqkSOr1ksmYfrWvOVFXSSjF4LODLslnmey1aDa2DCdr8SdE6AZhruYPY9El/wyjerM8OgRNFrprCZsD3Xr/48Zk3paAUhIh8naeJH4SlKFK+AoD/+5//hXHvgddZbylCWGECodkvvcU//v63ePnI/+P8gfkYPPSmDvkpyI1vVASz3r9++eyn5xeBI4cYgDYZBh2/cTySypRt9wVGQCVVK6ZErkhMUzSRVlavQOC7emfiUSIrAIS9A02ABJcI8BswDgp2U9xYCgjR/AqjsqXfSE58xEEKwijoFOSnNdYwGEzNk5ZFIyDw4ykEn7TeQdyJGMRi36IL2CN2NqVHGi89oGmrHHYSAF8sTpfsYTMfd2FwaBA4zzbCR+TBkibgiOyNtAZM6BZyidgiJpsZ1+0MQ2Iw6XS5uMZJ12NkgPsepR7Ws2btDkyHI0IOivA7ww4Im7EzAKN/IJ9GemaE/SYyWjEwwfQLmJYk9kNADqkekFaonLLraWsjfZV21Dlli6VRZ52RPj97sBsICsLenJ2CJ4HlidJ9oOorsMmqbp7dEQoI8WCbEneQ61aUEGEk7TzX9Lcvm6gnG7MTOqKPxsTcVRWuGK0Gh/DjzNhoKzjIRrJaHGLqy/RQ1l2arQ/yAvOEoXV8Hjwhz4dNRYDQOlL0puOwjffGmI58hGl6jePQrWc34NdHwK0qAKynlQMTtKIAvKOxIfDdiFG2qkA7W/RsCOQb9XR2TKcibRR4VHX/DrVZ7/j3qexXVxdu4/+cltpQMKYhNos6SlHiV5Z1E3m+Qti/um+MS+9Xdwgnyo7I+rqVdSvVEusoJXyN0JQ0EVP1ztdYQ8r7lB90UqFW6I78KK+hWbCc7n7RgrelzxDebigtNIFortBm6cPocFxCQdyAsJNeCgK+br+0aWmL6OFDvdgBakipkHAjLwegk4OipdnEFbPLE5tVPmZu3vrlmk1XSGA0DiWsSMsCRywpLKpaQULStzY60EVfIjkQogpL3OiSoFdnQh4oVlssN5VCuqGhd2dSVFuhxZBNVspXfFdA5U0FClTq9I0kPsf6D3AXbe1yIFU9P2U7xd6+umRKl+rAAyxgCnqBtUO9yD+0ZbvRBomHJ9eQk0MOQqW9W8c4fLAf3CBdcgkq/hkN9wLtwfcuCZQR6A5Wxq4EK3IlK6h+TZJ/y28EWH5WhFzxsuR7u+opSzCYRDBCldB33wSWA5wTZonc4b7hMGBw4b/FfErGthzOgZzunNoLoBl6qra8EJi9Dh+66cpweU+x26IXtkXSGRCE6n3P1iXfCdPEygTi3eXYr9hCnQx+YJau6za9+KaKQ+J6eJUKnunR2y3Yma8hpsTflJ0aJEhKu7NZerMeGk/qUjtB1AE9cdVIgNu8wGSZf9SJdw1B39X0Q3Yanp6fmnhERoQlhA+WWoIpYpLt4AdrxeewiQPaIKDMXT9psndIRzsTEFCz/EstBS5moStgWL/6paT/O5CIT3JZELa5qvICapiH7DwAKmCNp/jFn317CisEblvCyFGOTT7N+9KE1Q26oeUWSC6nbSVO9QjuOFDCqVhzBfMFRBGBMvH1o0ds8YJDcFsG3dTZTqHupF57v/YnURrSRKmbJ6ExjqPBkSEqsxy7Z/u0Ui0prM6meukLoqTzLKhoglFuGiGAsKm5BHYIdTvgRCl2rIfkdAWks1bGhhOUA42go55CyQI2pOlBGJWpg7dhwnfNCJik6cFXUaRHUcfeNemJU2lhcJc59lkWk/GcxiNZxErATpHgdqRX0uGkmzB4IHUHfiiqHrj1TWeOlu/4stvZd52SeFwSri1qzp+wU2NFAPiDKx6C1S6BO0LTfEKXg4CAwflKaccLAhudYNOgbk8DSqhiCIKfRJl3J7EnoP/w+++/xyXpICU/mTiXYQPNREKN6wwEMMCnYUbmQ6Ykk7jZryITVxfzGRR/v7PfzuYmHuJ2ZjYcYA/+4Tau2YSd76NUkQ1yKS83QlWxndKXiY7KXfJTTQDDc0D5j2vNT9iZToOsLaZ5hhRM3CV/XQytYqmjGAyieq3lBkhqzeu0ihqvKnfK5XMYNh8+PLcKbHr0kW0kUMQxunA3VQ1no4rXwMRmp/aajdkaxRNc4enp2REskAYoeQWWbbOPGLJJRR1/SrcanFZGZEHfHcGIgLGVjg6aMQrtsTknEM0ZAQJBDEw6SWLBsbnu0O1oH1f07THiCN1YgZGMnvuY2bWifFYfdE7I9VmROe2AagATuJY6anK4nR3TiuAgP8zxqphOgbxDlYMTUTBIjcUXTycwMOxu0e1wk8jMu8mSA6KLingkxNlHDjAGnRgZkVWdIFf4wBkHaXSG4bvLjbY6K/S1yeoBzox0yoeu8brQzZgDb61gVLNYP7lW4s7T22Zssm5n+WakW5vRXjgCbIdczNofWksfIaFB+rVMBxgzAW/eRBMH1njOQDxYnLq705G9B6P0uOVY+jzj6R5KeiyrdI0C0s/BDVKuaxEsWW4hU8lvFRBJ81uGXpD86cXlY5tHQyYmyo0U4+VbU0tZVyFj1R97VRfy8gl8wByy+sZx9Le5PizBqgr2B1XGdHBqajL8jrVarcwpQgq5bc039lDBVGggoDV9mLSnL92Cbdk7b8rwwFBBcMcjPcsLHiWs0prysTqrlT5RoCLEHBA2Z35NwYYt9DbEOymtwdnNaO2inUhDWRBt9HgSFwI7a61VUfpt7ghkAALXZHEEOgmmU3XQuN+sAYHMKdVnzzxFn9af7lyUYV0kyFY/RQV5Rx7GPZBEr8HUaiRqP3ZB2hI+GnaIer2NEYimszYy1raCRgY7jdPeeCvJogSf7K547f2nvhOgL0ywzzqDf8TO7k4+Y8elUdvdnH1GuW1E5T9oLwM8mLIHP1DTASwHz0yfPAjueoJbp7XaRu/KepSrqtz3Gve2PDmkYFNMDwGIOa9lzgumdOA2HzTfhhX3BUSIfWO6mRCJsjupufaAq0N0aBnB/TjWDOGx5L/KisS7DKWEkMhTOucc50SzS5xGdGZtOVm4YlniyajK0xvhd6fr+p8KoQgCC0DxxG9x2saJR3ng7889PFG65XsVnyek3y42MJcuWURalQ7CAbx1y167quULA18wQkZ7I5Fo4zPZNMTuvrKaWHu44LXJEuI+vLMv56M9YL3wJkZortQHnUu02HQyNcAw7JkfdJWR8HSPiNr/UUJflsFNYJ3D5EY+oQ1oV8I/OF1Hzo5ap0eB7U4W2Q/o0Yc76vbnSvBdjMVZ9O1xwErsCtx0YLfCAuY4MGY2kvJI+C1KcSPBeGL0sIh6L8dn3/AEbRvkex9o2gFQM6CnXaF6IbH/80WFxd0deOyHyjsNTQ0qo+n5cWU66A+e0tzv1GZwOtLvfHRvzRjKIUF9wYoOtEbGEQLMfdFRtJ5bIYR00e1+M+kqBUxd3Av8/kK7p/CQ/FdJ7l4SJKz3F19HGJgX43T885XTizK/4rqEHWfJAfgK1u7uDYmeQ7dwwG0aM9JXmGBksQzuhWn5ZdaOs3SYjHNcBLsL88LrXGaW04U242WIWULhBwcCAbl4YIEOknI25sFVJ5NdaHJTh6vgXviOB/I2nXYuWB6dQrugiXKRu8p7TDPZP/Z0IA/v1HyHSZHXRCPbLRSzzpZ8HFVz9y+Jm80Rt1Tcj0P7JPjarUJ8XAmwjgv6h40orvRx6vw+mnFFgrUfyoNWRCiitfcZc0GfvgVhHGdYlMdYJtCjOyczvScjDvEB4a8nSklY57q00/YcTZBAY0ZbLpJmC4TP3duDMGbqfLqNGvNyo3z4czNvb0GaO5/m3INqdHt7NXyNnbCCr0y2Rw/x2KABeFpuajTgNzTS+koitCuAKKM4TvJVHE/Rw3e8goXEq5QrqFEtlrf89nk74S8iLV5YUOfYQlMPeZLgOois7/Uul+MVS8z/31AtgEOyFG7mfwCJcwf9n0LRtl7o0rpHFxpx8bAj5qWIK5gND7ewtMh7W2ftpVp9obm9tP03fePYiSyYxn9ZDJQNe21f3LviIFlhm8kHZlnPdSeKrOH0gk50dR3HRIY9F6g56yoH5chVkybTpc+V07o+QK3aYrmmrISlPjLQZL85OjPJbzO86j0r87xyNaRnoV2DWZrJPVtv+sr4JDQ8dE/vzUShq1sLcuDU3jgXYjPOteOwr33ZrWC98wG7fT5/yy4FUl3joTnP9q2liAxqa2GudUO8uREZrgpf88AOP9iO+IhDUF39+c17FRK6XIEN3MgyzxYe3quPf355+fJPf72In1/8/PLZxaWu/LwB7E+v3sSv37+K3/3l7cXT55dtZezIMBjMevXjX+85y1xON++wQI1buI+bF1pwSK0nnaYk3pkFPw3RHpRGbYdiGkAyVB/5Ilvl2CeLvLpaz/4wU3LjBcHkaEcEikxUYnDs8kP3hRayE7xSQy8a0GxjKk2Z6nZLnT3KWMeUeYmkjiZuJM3xszVZAJ8fbkbRlX/9ukJsXlewYjSxhN6UmAw6hbq/EHWghtkOcaDXMBhLoBZdCchHitobu/e7g5gKsY8aMvjSxhhQUcfGMCLXSoaQYCp4feuDKFU0cv/NRogYI0TUGF3ncdCItPN47HrWMGdpFYgeg0zuva/NHw4ZgVdn/AYSCTQBsIAvJxEdrLpruqYrWg6ix+ZoAGN38/qLfusMEDXk75Bg0/8M3Mb54HCgdZFDlo3CtdsCdsKw+2/DJtlk1DmO092nOWtT2hjfIrD370hf2d7vNdBAYCbtoqoB++9SZzumtz6aIbv9rjkba5B7jVAMAH12xvUmOx91CpqsPYIW1PEJGnQdQt8T7biEhwvAw4BOLTB2Fc8qs33VLEv6Lwp2XhAzWQyFJoFTHps7cM0NqSQXutGInZ8rgNs27+I19/SacKCzJR1s9dYT7j4ksvT1F9M2AhJALs4/OIaFja+85OVeJ5FdVKR5NHV/MIRP2SMoIYGCDa4NrhDfNRM64jdyo2kJbC9otWbHzBS+uMXVSkrTDCOpZVV0jn3iNL8F8pkewldGvL9jKtTbRNxcLFfNCzwNN9PBwkxmY7z0kt/ADq2PLtBi76ygtddVOfhmf/lNL3zUWXU+cgrJCLiLDRbkJXGMqUkce/PJYO/Qu9rlHixid/FRVj5lMWZ3NPHMbze9Kb0J+TqvXuDebB4Rc/jC13OB91DpaTAS++zaCaCNYrAWwBmpvYICP4GHwWHuzoLJ/wMuQdBMWjsAAA=="
  }
}
SCRIPTS_DIR = ROOT / 'scripts' / '6facce3527a8'
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
for filename, record in EMBEDDED_SOURCES.items():
    data = gzip.decompress(base64.b64decode(record['gzip_base64']))
    if hashlib.sha256(data).hexdigest() != record['sha256']:
        raise RuntimeError('Embedded source checksum mismatch: ' + filename)
    compile(data.decode('utf-8'), filename, 'exec')
    destination = SCRIPTS_DIR / filename
    destination.write_bytes(data)
atomic_json(SCRIPTS_DIR / 'source_manifest.json', {name: {'sha256': record['sha256']} for name, record in EMBEDDED_SOURCES.items()})
print('Extracted verified adapters:', SCRIPTS_DIR)


## 4. Isolated environments, pinned sources and pretrained weights

Setup is sequential and uses **no GPU allocations**. Each model has its own
Python 3.11 environment and package versions. The notebook kernel's Torch is not
replaced. Shared UV caches/hardlinks reduce duplicate runtime storage.
All source commits, weight revisions and package versions are logged. Model
failures appear in the final report; other models can continue. Runtime CUDA
compatibility is confirmed only by the later monitored inference jobs.

Progress is printed per model; detailed install output is retained in setup logs.


In [ ]:
import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from pathlib import Path

MODELS = ("kokoro", "f5", "emotivoice", "styletts2", "omnivoice", "supertonic")
RUNTIME = Path("/tmp/tts_benchmark_runtime")
ENV_DIR = RUNTIME / "envs"
UV_CACHE = RUNTIME / "uv_cache"
HF_CACHE = RUNTIME / "huggingface"
ASSETS = RUNTIME / "assets"
REPOS = RUNTIME / "repos"
SETUP_LOGS = ROOT / "logs" / "setup"
for directory in (ENV_DIR, UV_CACHE, HF_CACHE, ASSETS, REPOS, SETUP_LOGS):
    directory.mkdir(parents=True, exist_ok=True)

BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "CUDA_VISIBLE_DEVICES": "", "HF_HOME": str(HF_CACHE),
    "HF_HUB_CACHE": str(HF_CACHE / "hub"), "UV_CACHE_DIR": str(UV_CACHE),
    "UV_PYTHON_INSTALL_DIR": str(RUNTIME / "python"),
    "GIT_LFS_SKIP_SMUDGE": "1", "TOKENIZERS_PARALLELISM": "false",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PYTHONUNBUFFERED": "1",
    "NLTK_DATA": str(RUNTIME / "nltk_data"),
    "HF_HUB_DOWNLOAD_TIMEOUT": "120", "HF_HUB_ETAG_TIMEOUT": "60",
})
# Subprocesses get their own environments and therefore cannot contaminate
# the notebook kernel or accidentally initialize CUDA during installation.
os.environ["NLTK_DATA"] = BASE_ENV["NLTK_DATA"]

def setup_command(command, log, *, cwd=None, env=None):
    command = [str(value) for value in command]
    with Path(log).open("a", encoding="utf-8") as stream:
        stream.write("\nCOMMAND " + json.dumps(command) + "\n")
        stream.flush()
        started = time.perf_counter()
        process = subprocess.run(command, cwd=cwd, env=env or BASE_ENV,
                                 stdout=stream, stderr=subprocess.STDOUT, text=True)
        stream.write(f"EXIT {process.returncode}; elapsed={time.perf_counter()-started:.3f}s\n")
    if process.returncode:
        tail = Path(log).read_text(encoding="utf-8", errors="replace")[-5000:]
        raise RuntimeError(f"Command failed ({process.returncode}); see {log}\n{tail}")

def setup_json(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                    allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

global_log = SETUP_LOGS / "bootstrap.log"
UV = [sys.executable, "-m", "uv"]
BOOTSTRAP_ERROR = None
try:
    if not globals().get("HARDWARE_READY", True):
        raise RuntimeError("Dual-GPU hardware preflight failed; installation and downloads were skipped.")
    if importlib.util.find_spec("uv") is None:
        setup_command([sys.executable, "-m", "pip", "install", "uv==0.6.17"], global_log)
    setup_command(UV + ["python", "install", "3.11"], global_log)
except Exception as exc:
    BOOTSTRAP_ERROR = str(exc)
    print("Python runtime bootstrap FAILED:", BOOTSTRAP_ERROR)

# Kaggle normally already supplies these. espeak-ng is required by StyleTTS2
# and is also the English fallback in Kokoro; the other packages support I/O.
missing_commands = [name for name in ("git", "ffmpeg", "espeak-ng")
                    if shutil.which(name) is None]
SYSTEM_STATUS = {"missing_before_setup": missing_commands}
if missing_commands and not globals().get("HARDWARE_READY", True):
    SYSTEM_STATUS.update(status="skipped", error="Hardware preflight failed; no system packages installed.")
elif missing_commands:
    try:
        apt = ["apt-get"] if os.geteuid() == 0 else ["sudo", "apt-get"]
        setup_command(apt + ["update", "-qq"], global_log)
        setup_command(apt + ["install", "-y", "-qq", "git", "ffmpeg",
                             "espeak-ng", "libespeak-ng1", "libsndfile1"], global_log)
        SYSTEM_STATUS["status"] = "ready"
    except Exception as exc:
        SYSTEM_STATUS.update(status="failed", error=str(exc))
else:
    SYSTEM_STATUS["status"] = "ready"

SOURCES = {
    "kokoro": ("https://github.com/hexgrad/kokoro.git", "dfb907a02bba8152ca444717ca5d78747ccb4bec"),
    "f5": ("https://github.com/SWivid/F5-TTS.git", "283252563dbf91be625e0c27926acfaac449186c"),
    "emotivoice": ("https://github.com/netease-youdao/EmotiVoice.git", "59f0f36de4db12825f4705dd4e0780d79dd6bb01"),
    "styletts2": ("https://github.com/yl4579/StyleTTS2.git", "5cedc71c333f8d8b8551ca59378bdcc7af4c9529"),
    "omnivoice": ("https://github.com/k2-fsa/OmniVoice.git", "08be0b4ccbac3e13e374e86fbfead4b4cac343e2"),
    "supertonic": ("https://github.com/supertone-oss-archive/supertonic.git", "1e9799e964ea4c0dad7cde993b65c3c813a7b373"),
}
COMMON = ["numpy==1.26.4", "scipy==1.14.1", "soundfile==0.13.1", "tqdm==4.67.1",
          "nvidia-ml-py==12.570.86"]
DEPENDENCIES = {
    "kokoro": ["transformers==5.3.0", "misaki[en]==0.9.4", "loguru==0.7.3",
               "spacy==3.8.7", "thinc==8.3.4", "spacy-curated-transformers==0.3.0",
               "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"],
    "f5": ["vocos==0.1.0", "pypinyin==0.54.0", "rjieba==0.1.1", "librosa==0.10.2.post1",
           "einops==0.8.1", "x_transformers==2.16.2", "torchdiffeq==0.2.5",
           "safetensors==0.5.3", "huggingface_hub==0.29.3", "wandb==0.19.8",
           "accelerate==1.6.0", "ema_pytorch==0.7.7", "datasets==3.5.0"],
    "emotivoice": ["transformers==4.26.1", "numba==0.61.2", "librosa==0.10.2.post1",
                   "yacs==0.1.8", "g2p_en==2.1.0", "jieba==0.42.1", "pypinyin==0.54.0",
                   "pypinyin_dict==0.9.0", "cn2an==0.5.23", "nltk==3.9.1",
                   "PyYAML==6.0.2", "huggingface_hub==0.29.3"],
    "styletts2": ["PyYAML==6.0.2", "munch==4.0.0", "transformers==4.44.2",
                  "einops==0.8.1", "einops-exts==0.0.4", "librosa==0.10.2.post1",
                  "phonemizer==3.3.0", "nltk==3.9.1", "matplotlib==3.10.1",
                  "huggingface_hub==0.29.3"],
    "omnivoice": ["transformers==5.3.0", "accelerate==1.12.0", "pydub==0.25.1",
                  "librosa==0.10.2.post1", "num2words==0.5.14", "WeTextProcessing==1.0.3"],
    "supertonic": ["onnxruntime-gpu[cuda,cudnn]==1.23.2", "huggingface_hub==0.29.3"],
}

def checkout_model(model, log):
    url, commit = SOURCES[model]
    destination = REPOS / model
    if not (destination / ".git").exists():
        setup_command(["git", "init", str(destination)], log)
        setup_command(["git", "-C", str(destination), "remote", "add", "origin", url], log)
    setup_command(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit], log)
    setup_command(["git", "-C", str(destination), "checkout", "--detach", commit], log)
    actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                     text=True, env=BASE_ENV).strip()
    if actual != commit:
        raise RuntimeError(f"Source revision mismatch: {actual} != {commit}")
    return destination

def download_hf(python, model, log):
    destination = ASSETS / model
    destination.mkdir(parents=True, exist_ok=True)
    # Download before concurrent inference, so benchmark timing excludes network
    # and workers do not race on partial weight files.
    specifications = {
        "kokoro": [("hexgrad/Kokoro-82M", "f3ff3571791e39611d31c381e3a41a3af07b4987",
                    ["config.json", "kokoro-v1_0.pth", "voices/*.pt"], ".")],
        "f5": [("SWivid/F5-TTS", "84e5a410d9cead4de2f847e7c9369a6440bdfaca",
                ["F5TTS_v1_Base/model_1250000.safetensors", "F5TTS_v1_Base/vocab.txt"], "."),
               ("charactr/vocos-mel-24khz", "0feb3fdd929bcd6649e0e7c5a688cf7dd012ef21",
                ["config.yaml", "pytorch_model.bin"], "vocos")],
        "emotivoice": [("WangZeJun/simbert-base-chinese", "8096d1f9a7f7c2d9dbd7974511c4bc4a1755932b",
                        ["*.json", "*.txt", "*.bin", "*.safetensors"], "simbert")],
        "styletts2": [("yl4579/StyleTTS2-LibriTTS", "3aa7ba7f8f275ec13dce21682a61494c35089e2a",
                      ["Models/LibriTTS/config.yml", "Models/LibriTTS/epochs_2nd_00020.pth"], ".")],
        "omnivoice": [("k2-fsa/OmniVoice", "c5fdb5ccb189668d56333f77ba2629f4cd7535f4",
                      ["*.json", "*.safetensors", "*.model", "*.txt", "audio_tokenizer/**"], ".")],
        "supertonic": [("supertone-oss-archive/supertonic-3", "aafc6e32416a594460b32413efc49d7fe4ce6d46",
                       ["onnx/**", "voice_styles/**"], ".")],
    }
    script = """import json, sys
from huggingface_hub import snapshot_download
for repo, revision, patterns, subdir in json.loads(sys.argv[1]):
    snapshot_download(repo_id=repo, revision=revision, allow_patterns=patterns,
                      local_dir=str(__import__('pathlib').Path(sys.argv[2])/subdir), max_workers=1)
"""
    setup_command([python, "-c", script, json.dumps(specifications[model]), str(destination)], log)
    return destination

def download_emoti_checkpoints(log):
    # Git stores signed SHA-256 LFS pointers. Download only the two required
    # inference binaries, and validate bytes against their pinned Git pointers.
    commit = "d1332ad9fee7020df5f358fdbe8d257dbf0d1a7c"
    destination = ASSETS / "emotivoice" / "outputs"
    if not (destination / ".git").is_dir():
        setup_command(["git", "init", str(destination)], log)
        setup_command(["git", "-C", str(destination), "remote", "add", "origin",
                       "https://www.modelscope.cn/syq163/outputs.git"], log)
    setup_command(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit], log)
    paths = ("prompt_tts_open_source_joint/ckpt/g_00140000", "style_encoder/ckpt/checkpoint_163431")
    records = []
    for relative in paths:
        pointer = subprocess.check_output(["git", "-C", str(destination), "show", f"{commit}:{relative}"],
                                          env=BASE_ENV, text=True)
        oid = re.search(r"^oid sha256:([0-9a-f]{64})$", pointer, re.M)
        size = re.search(r"^size (\d+)$", pointer, re.M)
        if not oid or not size:
            raise RuntimeError(f"Expected an authenticated Git LFS pointer for {relative}")
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        expected_size, expected_sha = int(size.group(1)), oid.group(1)
        def valid_file(path):
            if not path.is_file() or path.stat().st_size != expected_size:
                return False
            digest = hashlib.sha256()
            with path.open("rb") as stream:
                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                    digest.update(chunk)
            return digest.hexdigest() == expected_sha
        if not valid_file(target):
            query = urllib.parse.urlencode({"Revision": commit, "FilePath": relative})
            url = "https://www.modelscope.cn/api/v1/models/syq163/outputs/repo?" + query
            temporary = target.with_name(target.name + ".part")
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as stream:
                shutil.copyfileobj(response, stream, length=8 * 1024 * 1024)
            if not valid_file(temporary):
                raise RuntimeError(f"Checkpoint SHA-256/size mismatch for {relative}; no inference will run.")
            temporary.replace(target)
        records.append({"file": str(target), "revision": commit,
                        "sha256": expected_sha, "size": expected_size})
    setup_json(ROOT / "emotivoice_checkpoint_provenance.json", records)
    return {"generator_checkpoint": str(destination / paths[0]),
            "style_checkpoint": str(destination / paths[1])}

def resources_for(model, repo, assets):
    resources = {"repo": str(repo)}
    if model == "kokoro":
        resources = {"checkpoint": str(assets / "kokoro-v1_0.pth"),
                     "model_config": str(assets / "config.json"),
                     "cache_dir": str(HF_CACHE / "hub")}
    elif model == "f5":
        resources.update(checkpoint=str(assets / "F5TTS_v1_Base/model_1250000.safetensors"),
                         vocab=str(assets / "F5TTS_v1_Base/vocab.txt"), vocos_dir=str(assets / "vocos"))
    elif model == "emotivoice":
        resources.update(bert_path=str(assets / "simbert"), local_files_only=True)
    elif model == "styletts2":
        resources.update(checkpoint=str(assets / "Models/LibriTTS/epochs_2nd_00020.pth"),
                         model_config=str(assets / "Models/LibriTTS/config.yml"))
    elif model == "omnivoice":
        resources = {"local_model": str(assets), "local_files_only": True}
    elif model == "supertonic":
        resources.update(assets_dir=str(assets))
    return resources

requested = CONFIG.get("install_models", list(MODELS))
if not isinstance(requested, list) or any(model not in MODELS for model in requested):
    raise ValueError("CONFIG['install_models'] must contain known model names.")
SETUP_RESULTS = {}
report_path = ROOT / "setup_report.json"
for model in MODELS:
    if model not in requested:
        SETUP_RESULTS[model] = {"status": "not_requested", "error": "Disabled in install_models"}
        continue
    log = SETUP_LOGS / f"{model}.log"
    python = ENV_DIR / model / "bin" / "python"
    entry = {"status": "installing", "python": str(python), "resources": {}, "log": str(log)}
    SETUP_RESULTS[model] = entry
    started = time.perf_counter()
    print(f"[{model}] installing isolated Python, dependencies, source, and weights...")
    try:
        if BOOTSTRAP_ERROR:
            raise RuntimeError("Python runtime bootstrap failed: " + BOOTSTRAP_ERROR)
        if model in ("styletts2", "kokoro") and shutil.which("espeak-ng") is None:
            raise RuntimeError("espeak-ng is unavailable. See bootstrap.log for the system dependency failure.")
        if shutil.which("git") is None:
            raise RuntimeError("git is unavailable; pinned source verification cannot proceed.")
        if not python.is_file():
            setup_command(UV + ["venv", "--python", "3.11", "--seed", str(ENV_DIR / model)], log)
        # Hardlink packages shared between isolated environments. All runtime
        # trees are under /tmp on one filesystem; zip contains outputs only.
        torch_version = "2.8.0" if model in ("omnivoice", "supertonic") else "2.5.1"
        variant = "cu128" if model in ("omnivoice", "supertonic") else "cu121"
        override = CONFIG.get("torch_cuda_variant")
        if isinstance(override, dict):
            variant = override.get(model, variant)
        elif override not in (None, "auto", ""):
            variant = override
        supported = {"2.5.1": {"cu118", "cu121", "cu124"}, "2.8.0": {"cu126", "cu128", "cu129"}}
        if variant not in supported[torch_version]:
            raise ValueError(f"Torch {torch_version} has no supported wheel variant {variant!r} in this setup.")
        if model == "supertonic" and variant != "cu128":
            raise ValueError("Supertonic ORT1.23.2 requires the configured CUDA12.8/cuDNN9 runtime; retain cu128.")
        if model == "supertonic" and CONFIG.get("strict_cuda_driver", False):
            versions = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                text=True, env=BASE_ENV).strip().splitlines()
            if not versions or any(tuple(map(int, version.strip().split("."))) < (570, 26)
                                   for version in versions):
                raise RuntimeError("Strict driver policy requires NVIDIA Linux driver>=570.26 for CUDA12.8.")
        constraints = RUNTIME / f"{model}_constraints.txt"
        constraints.write_text(f"torch=={torch_version}\ntorchaudio=={torch_version}\n"
                               "numpy==1.26.4\nscipy==1.14.1\n", encoding="utf-8")
        install = UV + ["pip", "install", "--python", str(python), "--link-mode", "hardlink"]
        reinstall = ["--reinstall"] if CONFIG.get("force_setup", False) else []
        setup_command(install + reinstall + [f"torch=={torch_version}", f"torchaudio=={torch_version}",
                                 "--index-url", f"https://download.pytorch.org/whl/{variant}"], log)
        setup_command(install + ["--constraint", str(constraints)] + COMMON + DEPENDENCIES[model], log)
        repo = checkout_model(model, log)
        if model in ("kokoro", "omnivoice"):
            url, commit = SOURCES[model]
            setup_command(install + reinstall + ["--no-deps", f"git+{url}@{commit}"], log)
        if model == "emotivoice":
            setup_command([python, "-m", "nltk.downloader", "averaged_perceptron_tagger_eng",
                           "averaged_perceptron_tagger", "cmudict", "punkt", "punkt_tab"], log)
        assets = download_hf(python, model, log)
        resources = resources_for(model, repo, assets)
        if model == "emotivoice":
            resources.update(download_emoti_checkpoints(log))
        entry.update(status="ready", resources=resources, source_revision=SOURCES[model][1],
                     torch_version=torch_version, torch_cuda_variant=variant,
                     elapsed_seconds=time.perf_counter()-started)
        # Exact installed package versions are useful when reproducing failures.
        with (SETUP_LOGS / f"{model}_packages.txt").open("w", encoding="utf-8") as stream:
            subprocess.run(UV + ["pip", "freeze", "--python", str(python)], env=BASE_ENV,
                           stdout=stream, stderr=subprocess.STDOUT, check=True, text=True)
        print(f"[{model}] ready ({entry['elapsed_seconds']:.1f}s); details: {log}")
    except Exception as exc:
        entry.update(status="failed", error=str(exc), elapsed_seconds=time.perf_counter()-started)
        print(f"[{model}] FAILED: {exc}")
    setup_json(report_path, {"models": SETUP_RESULTS, "system": SYSTEM_STATUS,
                             "runtime_directory": str(RUNTIME), "cuda_allocations_during_setup": False})
print("Setup statuses:", {model: item["status"] for model, item in SETUP_RESULTS.items()})


In [ ]:
# CPU-only import/API checks. CUDA context/session initialization is performed
# later by monitored benchmark jobs, not hidden in dependency setup.
PREFLIGHT_RESULTS = {}
API_CHECKS = {
    "kokoro": "from kokoro import KModel, KPipeline; from misaki import en; en.G2P(trf=False, british=False)",
    "f5": "from f5_tts.model import CFM, DiT; from f5_tts.model.utils import get_tokenizer; from vocos import Vocos",
    "emotivoice": "from frontend_en import G2p; from frontend import g2p_cn_en; from models.prompt_tts_modified.jets import JETSGenerator; from models.prompt_tts_modified.simbert import StyleEncoder; G2p()",
    "styletts2": "from models import build_model; from Utils.PLBERT.util import CustomAlbert; from Modules.diffusion.sampler import DiffusionSampler; from phonemizer.backend import EspeakBackend; EspeakBackend('en-us')",
    "omnivoice": "from omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel",
    "supertonic": "import onnxruntime as ort; assert 'CUDAExecutionProvider' in ort.get_available_providers()",
}
SCRIPT_NAMES = {"f5": "f5_samples.py", "styletts2": "styletts2_samples.py",
                "emotivoice": "emotivoice_samples.py", "kokoro": "kokoro_samples.py",
                "omnivoice": "omnivoice_samples.py", "supertonic": "supertonic_samples.py"}
for model, item in SETUP_RESULTS.items():
    if item["status"] != "ready":
        PREFLIGHT_RESULTS[model] = {"status": "unavailable", "error": item.get("error")}
        continue
    log = SETUP_LOGS / f"{model}_preflight.log"
    try:
        repo = REPOS / model
        sys_path = str(repo / "src") if model == "f5" else str(repo)
        script = ("import os,sys,torch; assert os.environ.get('CUDA_VISIBLE_DEVICES')==''; "
                  f"sys.path.insert(0,{sys_path!r}); os.chdir({str(repo)!r}); " + API_CHECKS[model] +
                  "; print('API_IMPORTS_OK'); print('Torch:',torch.__version__,'CUDA runtime:',torch.version.cuda)")
        setup_command([item["python"], "-c", script], log)
        setup_command([item["python"], str(SCRIPTS_DIR / SCRIPT_NAMES[model]), "--help"], log)
        PREFLIGHT_RESULTS[model] = {"status": "passed", "log": str(log)}
        print(f"[{model}] CPU API preflight passed")
    except Exception as exc:
        PREFLIGHT_RESULTS[model] = {"status": "failed", "error": str(exc), "log": str(log)}
        item.update(status="failed", error="API preflight failed: " + str(exc))
        print(f"[{model}] API preflight FAILED: {exc}")
setup_json(ROOT / "preflight_report.json", PREFLIGHT_RESULTS)
setup_json(ROOT / "setup_report.json", {"models": SETUP_RESULTS, "system": SYSTEM_STATUS,
           "runtime_directory": str(RUNTIME), "cuda_allocations_during_setup": False})


## 5. Reference voice

The CPU bootstrap is a prerequisite for reference-based models; paired GPU
experiments begin after it finishes. No content-dependent trimming is used.
The reference and transcript are included in the ZIP when configured.


In [ ]:
# F5/StyleTTS2 need reference audio. Supply your own exact transcript for voice
# cloning, or generate this disclosed synthetic reference with Kokoro on CPU.
REFERENCE_AUDIO = CONFIG.get("reference_audio")
REFERENCE_TEXT = CONFIG.get("reference_text")
if CONFIG.get("speaker_mode") == "default":
    REFERENCE_AUDIO = REFERENCE_TEXT = None
REFERENCE_STATUS = {"status": "pending"}
if REFERENCE_AUDIO:
    try:
        REFERENCE_AUDIO = str(Path(REFERENCE_AUDIO).expanduser().resolve())
        if not Path(REFERENCE_AUDIO).is_file():
            raise ValueError("Set an existing reference_audio WAV.")
        if REFERENCE_TEXT is not None and (not isinstance(REFERENCE_TEXT, str) or not REFERENCE_TEXT.strip()):
            raise ValueError("reference_text must be the exact nonempty transcript or None.")
        checker_python = next((item["python"] for item in SETUP_RESULTS.values()
                               if item.get("status") == "ready"), sys.executable)
        checker = """import json,sys,numpy as np,soundfile as sf
wave,rate=sf.read(sys.argv[1],dtype='float32',always_2d=True)
assert wave.shape[1]==1, 'Reference must be mono.'
duration=wave.shape[0]/rate
assert 1<=duration<=8, f'Reference duration {duration:.3f}s outside 1-8s.'
assert np.isfinite(wave).all(), 'Reference contains nonfinite samples.'
assert np.max(np.abs(wave))>1e-5, 'Reference is silent.'
print(json.dumps({'duration_seconds':duration,'sample_rate':rate,'channels':1}))
"""
        checked = subprocess.run([checker_python, "-c", checker, REFERENCE_AUDIO], env=BASE_ENV,
                                 text=True, capture_output=True, check=True)
        facts = json.loads(checked.stdout.strip())
        REFERENCE_STATUS.update(status="provided", path=REFERENCE_AUDIO, transcript=REFERENCE_TEXT, **facts)
    except Exception as exc:
        REFERENCE_AUDIO = None
        details = (getattr(exc, "stderr", None) or str(exc)).strip()
        REFERENCE_STATUS.update(status="failed", error=details)
        print("Reference validation FAILED:", details)
elif CONFIG.get("speaker_mode") == "reference":
    REFERENCE_STATUS.update(status="failed", error="speaker_mode='reference' requires your reference_audio WAV; add its transcript for F5/OmniVoice.")
    print(REFERENCE_STATUS["error"])
else:
    REFERENCE_TEXT = "The morning sun brings gentle warmth to the quiet garden."
    item = SETUP_RESULTS.get("kokoro", {})
    if item.get("status") != "ready":
        REFERENCE_STATUS.update(status="failed", error="Kokoro unavailable; set reference_audio and reference_text.")
        print(REFERENCE_STATUS["error"])
    else:
        try:
            ref_root = ROOT / "reference"
            ref_root.mkdir(parents=True, exist_ok=True)
            parameters = {**item["resources"], "text": REFERENCE_TEXT,
                          "voice": str(ASSETS / "kokoro" / "voices" / "am_fenrir.pt"),
                          "language": "a", "device": "cpu", "n": 1, "seed": 42, "speed": 1.0,
                          "sample_rate": 24000, "output_dir": str(ref_root / "kokoro_runs")}
            settings = ref_root / "bootstrap_config.json"
            setup_json(settings, parameters)
            setup_command([item["python"], str(SCRIPTS_DIR / "kokoro_samples.py"),
                           "--config", str(settings)], SETUP_LOGS / "reference_bootstrap.log")
            candidates = sorted((ref_root / "kokoro_runs").glob("**/*.wav"), key=lambda path: path.stat().st_mtime)
            if not candidates:
                raise RuntimeError("Kokoro created no reference WAV.")
            reference = ref_root / "reference_kokoro_am_fenrir.wav"
            shutil.copy2(candidates[-1], reference)
            # WAV PCM frame/rate header gives total duration without loading a
            # model or removing content-dependent silence.
            import wave
            with wave.open(str(reference), "rb") as audio:
                duration = audio.getnframes() / audio.getframerate()
            if not 1 <= duration <= 8:
                raise RuntimeError(f"Synthetic reference duration {duration:.3f}s outside1–8s; supply a clean reference.")
            REFERENCE_AUDIO = str(reference)
            REFERENCE_STATUS.update(status="synthetic", path=REFERENCE_AUDIO,
                                    transcript=REFERENCE_TEXT, duration_seconds=duration,
                                    generator="Kokoro am_fenrir, CPU; complete untrimmed male waveform")
            print(f"Synthetic reference ready: {REFERENCE_AUDIO} ({duration:.3f}s)")
        except Exception as exc:
            REFERENCE_AUDIO = None
            REFERENCE_STATUS.update(status="failed", error=str(exc))
            print(f"Reference bootstrap FAILED: {exc}")
setup_json(ROOT / "reference_report.json", REFERENCE_STATUS)


## 6. Paired GPU experiments

Each job loads its model once and produces all requested takes. Both slots
display the worker's actual progress line, plus sampled VRAM. Jobs are named by
model, profile, word count, text/config hashes and seed. Full carriage-return
progress logs are saved. Failed/over-budget jobs and any partial recordings are
retained. If you interrupt, run the report cell afterward to package the results.


In [ ]:
SCRIPT_NAMES = {"kokoro": "kokoro_samples.py", "f5": "f5_samples.py", "emotivoice": "emotivoice_samples.py",
                "styletts2": "styletts2_samples.py", "omnivoice": "omnivoice_samples.py", "supertonic": "supertonic_samples.py"}
JOB_RESULTS = []
SAMPLE_RESULTS = []
TIMELINE = []

def slug(value, maximum=48):
    value = re.sub(r"[^a-zA-Z0-9_-]+", "-", str(value)).strip("-")
    return (value or "unnamed")[:maximum]

def profile_jobs():
    jobs = []
    speaker_mode = CONFIG.get("speaker_mode", "auto")
    if speaker_mode not in {"auto", "reference", "default"}:
        raise ValueError("speaker_mode must be auto, reference, or default")
    for case_index, case in enumerate(TEXT_CASES):
        text = case["text"].strip()
        count = len(text.split())
        if not 10 <= count <= 500:
            raise ValueError(f"Case {case['name']!r} has {count} words, outside 10-500")
        text_hash = hashlib.sha256(text.encode()).hexdigest()[:8]
        for model in CONFIG["install_models"]:
            if model not in SCRIPT_NAMES:
                raise ValueError(f"Unknown model {model}")
            for profile_index, profile in enumerate(MODEL_PROFILES.get(model, [])):
                if not profile.get("enabled", True):
                    continue
                parameters = dict(profile.get("parameters", {}))
                forbidden = set(parameters) & {"config", "text", "text_file", "device", "n", "seed", "output_dir", "max_vram_gb", "estimate_only", "self_test"}
                if forbidden:
                    raise ValueError(f"Profile {model}/{profile['name']} has supervisor-controlled keys: {sorted(forbidden)}")
                count_takes = profile.get("n", CONFIG["n"])
                seed = profile.get("seed", CONFIG["seed"])
                if type(count_takes) is not int or count_takes < 1 or type(seed) is not int or not 0 <= seed <= 2**63 - count_takes:
                    raise ValueError("Profile n and seed must be positive/nonnegative integers within signed 64-bit range")
                # Incorporate case ordinal/config identity: duplicate names cannot overwrite runs.
                config_hash = hashlib.sha256(json.dumps(profile, sort_keys=True).encode()).hexdigest()[:8]
                job_id = f"{model}__{slug(profile['name'])}__w{count:03d}__c{case_index:02d}p{profile_index:02d}__{text_hash}_{config_hash}"
                job_dir = RUN_DIR / "jobs" / job_id
                job_dir.mkdir(parents=True, exist_ok=False)
                text_path = job_dir / "text.txt"
                text_path.write_text(text, encoding="utf-8")
                settings = {"text_file": str(text_path), "n": count_takes, "seed": seed,
                            "device": "cuda:0", "sample_rate": CONFIG["sample_rate"], "pause_ms": CONFIG["pause_ms"],
                            "max_vram_gb": CONFIG["max_vram_gb"], "output_dir": str(job_dir / "raw")}
                setup = SETUP_RESULTS.get(model, {"status": "failed", "error": "Model setup not requested/completed"})
                resources = dict(setup.get("resources", {}))
                settings.update({k: str(v) if isinstance(v, Path) else v for k, v in resources.items() if v is not None})
                settings.update({k: v for k, v in parameters.items() if v is not None})
                if model == "kokoro":
                    voice_dir = Path(resources.get("checkpoint", ".")).parent / "voices"
                    voices = [voice.strip() for voice in settings.get("voice", "am_fenrir").split(",")]
                    settings["voice"] = ",".join(str(voice_dir / f"{voice}.pt") if (voice_dir / f"{voice}.pt").is_file() else voice for voice in voices)
                error = None
                blocked_status = "setup_failed"
                speaker_source = "preset"
                if setup.get("status") != "ready":
                    error = setup.get("error", "Model environment is unavailable")
                if model in {"f5", "styletts2"}:
                    speaker_source = "user_reference" if parameters.get("reference_audio") or (CONFIG.get("reference_audio") and speaker_mode != "default") else "synthetic_male_reference"
                    settings.setdefault("reference_audio", str(REFERENCE_AUDIO) if REFERENCE_AUDIO else None)
                    if model == "f5":
                        settings.setdefault("reference_text", REFERENCE_TEXT)
                    if not settings.get("reference_audio") or (model == "f5" and not settings.get("reference_text")):
                        error = "Reference WAV/transcript preparation failed; see reference_report.json. F5 requires an exact transcript."
                if model == "omnivoice":
                    reference_choice = profile.get("reference", "auto")
                    if type(reference_choice) is not bool and reference_choice != "auto":
                        raise ValueError("OmniVoice profile reference must be True, False, or 'auto'")
                    cached_prompt = bool(settings.get("prompt_cache") and Path(settings["prompt_cache"]).is_file())
                    use_reference = bool(settings.get("ref_audio")) or cached_prompt or reference_choice is True or speaker_mode == "reference" or (reference_choice == "auto" and speaker_mode == "auto" and bool(CONFIG.get("reference_audio")))
                    if use_reference:
                        if not cached_prompt:
                            settings.setdefault("ref_audio", str(REFERENCE_AUDIO) if REFERENCE_AUDIO else None)
                            settings.setdefault("ref_text", REFERENCE_TEXT)
                        # Preset voice attributes must not conflict with the supplied person's voice.
                        clone_parameters = profile.get("clone_parameters", {})
                        forbidden_clone = set(clone_parameters) & {"config", "text", "text_file", "device", "n", "seed", "output_dir", "max_vram_gb", "estimate_only", "self_test"}
                        if forbidden_clone:
                            raise ValueError(f"clone_parameters contains supervisor-controlled keys: {sorted(forbidden_clone)}")
                        if "instruct" not in clone_parameters:
                            settings.pop("instruct", None)
                        settings.update(clone_parameters)
                        speaker_source = "cached_reference_prompt" if cached_prompt else "user_reference" if parameters.get("ref_audio") or (CONFIG.get("reference_audio") and speaker_mode != "default") else "synthetic_male_reference"
                        if not cached_prompt and (not settings.get("ref_audio") or not settings.get("ref_text")):
                            error = "Reference WAV/transcript preparation failed; see reference_report.json. OmniVoice requires an exact transcript in this adapter."
                    else:
                        speaker_source = "voice_design"
                if speaker_mode == "reference" and model not in {"f5", "styletts2", "omnivoice"}:
                    error = "This model supports preset speakers rather than arbitrary reference-WAV cloning."
                    blocked_status = "reference_unsupported"
                if model in {"emotivoice", "supertonic"}:
                    settings["allow_unverified_cuda_budget"] = True
                settings = {k: v for k, v in settings.items() if v is not None}
                config_path = job_dir / "config.json"
                atomic_json(config_path, settings)
                job = {"job_id": job_id, "model": model, "profile": profile["name"], "case": case["name"],
                       "word_count": count, "text_sha256": hashlib.sha256(text.encode()).hexdigest(),
                       "n": count_takes, "seed": seed, "job_dir": str(job_dir), "settings": settings,
                       "speaker_source": speaker_source, "blocked_status": blocked_status,
                       "config_path": str(config_path), "python": setup.get("python"), "blocked_reason": error}
                atomic_json(job_dir / "job.json", job)
                jobs.append(job)
    return jobs

def process_tree(pid):
    ids = {pid}
    if pid:
        try:
            ids.update(child.pid for child in psutil.Process(pid).children(recursive=True))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return ids - {None}

def nvml_process_usage(handle, tracked):
    allocations = {}
    available = False
    for getter_name in ("nvmlDeviceGetComputeRunningProcesses", "nvmlDeviceGetGraphicsRunningProcesses"):
        try:
            for proc in getattr(pynvml, getter_name)(handle):
                if proc.pid in tracked:
                    value = proc.usedGpuMemory
                    if isinstance(value, int) and 0 <= value < 2**60:
                        allocations[proc.pid] = max(value, allocations.get(proc.pid, 0))
                        available = True
        except pynvml.NVMLError:
            continue
    return (sum(allocations.values()) if available else None), sorted(allocations)

class JobMonitor:
    """Monitor external allocations from before model loading through process exit."""
    def __init__(self, job, slot):
        self.job, self.slot, self.pid = job, slot, None
        self.handle = GPU_HANDLES[slot]
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.process_ids = set()
        self.process_births = {}
        self.device_peak = 0
        self.process_peak = None
        self.last_device = 0
        self.last_process = None
        self.violation = None
        self.error = None
        self.poll_count = 0
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self.observe, daemon=True)

    def start(self):
        self.thread.start()

    def observe(self):
        path = Path(self.job["job_dir"]) / "vram_trace.csv"
        budget = int(CONFIG["max_vram_gb"] * 1e9)
        interval = CONFIG["nvml_interval_ms"] / 1000
        fields = ["elapsed_seconds", "device_used_bytes", "process_tree_used_bytes", "gpu_utilization_percent", "memory_utilization_percent", "worker_pid", "tracked_gpu_pids"]
        try:
            with path.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=fields)
                writer.writeheader()
                while not self.stop_event.is_set():
                    tick = time.perf_counter()
                    if self.pid:
                        self.process_ids.update(process_tree(self.pid))
                        for pid in self.process_ids:
                            if pid not in self.process_births:
                                try:
                                    self.process_births[pid] = psutil.Process(pid).create_time()
                                except (psutil.NoSuchProcess, psutil.AccessDenied):
                                    pass
                    info = pynvml.nvmlDeviceGetMemoryInfo(self.handle)
                    usage, gpu_pids = nvml_process_usage(self.handle, self.process_ids)
                    utilization = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
                    self.last_device, self.last_process = info.used, usage
                    self.device_peak = max(self.device_peak, info.used)
                    if usage is not None:
                        self.process_peak = max(self.process_peak or 0, usage)
                    self.poll_count += 1
                    writer.writerow({"elapsed_seconds": f"{tick-self.started:.6f}", "device_used_bytes": info.used,
                        "process_tree_used_bytes": usage, "gpu_utilization_percent": utilization.gpu,
                        "memory_utilization_percent": utilization.memory, "worker_pid": self.pid,
                        "tracked_gpu_pids": ";".join(map(str, gpu_pids))})
                    observed = usage if CONFIG["budget_scope"] == "process_tree" and usage is not None else info.used
                    scope = "process_tree" if CONFIG["budget_scope"] == "process_tree" and usage is not None else "device_total"
                    if observed > budget and self.violation is None:
                        self.violation = {"observed_bytes": observed, "scope": scope,
                                          "budget_bytes": budget, "elapsed_seconds": tick-self.started}
                    if self.poll_count % 100 == 0:
                        stream.flush()
                    self.stop_event.wait(max(0.001, interval-(time.perf_counter()-tick)))
        except Exception as exc:
            self.error = f"{type(exc).__name__}: {exc}"

    def finish(self):
        self.stop_event.set()
        self.thread.join(timeout=5)
        if self.thread.is_alive():
            self.error = "Monitor thread did not finish"
        result = {"nvml_sampled_device_peak_bytes": self.device_peak,
                  "nvml_sampled_process_tree_peak_bytes": self.process_peak,
                  "requested_interval_ms": CONFIG["nvml_interval_ms"], "samples": self.poll_count,
                  "budget_violation": self.violation, "monitor_error": self.error,
                  "hard_instantaneous_peak_certified": False}
        atomic_json(Path(self.job["job_dir"]) / "memory_monitor.json", result)
        return result

class Worker:
    def __init__(self, job, slot):
        import codecs
        self.job, self.slot = job, slot
        self.started = time.perf_counter()
        self.latest_line = "Starting interpreter and loading model"
        self.tail = deque(maxlen=40)
        self.abort_reason = None
        self.monitor = JobMonitor(job, slot)
        self.monitor.start()
        env = globals().get("BASE_ENV", os.environ).copy()
        env.update({"CUDA_VISIBLE_DEVICES": GPU_INVENTORY[slot]["uuid"], "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
                    "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": str(CONFIG["cpu_threads_per_job"]),
                    "MKL_NUM_THREADS": str(CONFIG["cpu_threads_per_job"]), "TOKENIZERS_PARALLELISM": "false"})
        espeak_library = SETUP_RESULTS[self.job["model"]].get("espeak_library") or env.get("PHONEMIZER_ESPEAK_LIBRARY")
        if espeak_library:
            env["PHONEMIZER_ESPEAK_LIBRARY"] = espeak_library
        else:
            env.pop("PHONEMIZER_ESPEAK_LIBRARY", None)
        command = [job["python"], str(SCRIPTS_DIR / SCRIPT_NAMES[job["model"]]), "--config", job["config_path"]]
        atomic_json(Path(job["job_dir"]) / "command.json", {"argv": command, "gpu_uuid": env["CUDA_VISIBLE_DEVICES"],
                    "logical_device": "cuda:0", "cpu_threads": CONFIG["cpu_threads_per_job"]})
        try:
            self.process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                bufsize=0, start_new_session=True, cwd=str(ROOT))
            self.monitor.pid = self.process.pid
            self.monitor.process_ids.add(self.process.pid)
        except Exception:
            self.monitor.finish()
            raise

        def read_output():
            decoder = codecs.getincrementaldecoder("utf-8")("replace")
            remainder = ""
            try:
                with (Path(job["job_dir"]) / "generation.log").open("w", encoding="utf-8") as log:
                    while True:
                        data = os.read(self.process.stdout.fileno(), 65536)
                        if not data:
                            break
                        decoded = decoder.decode(data)
                        log.write(decoded)
                        log.flush()
                        fragments = re.split(r"[\r\n]", remainder + decoded)
                        remainder = fragments.pop()
                        for fragment in fragments:
                            clean = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", fragment).strip()
                            if clean:
                                self.latest_line = clean[-500:]
                                self.tail.append(clean)
                    final = remainder + decoder.decode(b"", final=True)
                    if final.strip():
                        self.latest_line = final.strip()[-500:]
                        self.tail.append(final.strip())
            except Exception as exc:
                self.latest_line = "Log reader error: " + str(exc)
        self.reader = threading.Thread(target=read_output, daemon=True)
        self.reader.start()
        TIMELINE.append({"event": "start", "job_id": job["job_id"], "model": job["model"],
                         "gpu_slot": slot, "gpu_uuid": GPU_INVENTORY[slot]["uuid"], "timestamp": time.time()})

    def terminate(self, reason):
        self.abort_reason = reason
        self.monitor.process_ids.update(process_tree(self.process.pid))
        try:
            os.killpg(self.process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            self.process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(self.process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            self.process.wait(timeout=10)
        for pid in self.monitor.process_ids - {self.process.pid}:
            try:
                descendant = psutil.Process(pid)
                if descendant.create_time() == self.monitor.process_births.get(pid):
                    descendant.kill()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass

    def finish(self):
        self.process.wait()
        # A parent can leave a child holding pipe/GPU handles. Kill only tracked
        # descendants whose creation time still matches, avoiding reused PIDs.
        for pid in self.monitor.process_ids - {self.process.pid}:
            try:
                descendant = psutil.Process(pid)
                if descendant.create_time() == self.monitor.process_births.get(pid):
                    descendant.kill()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        self.reader.join(timeout=10)
        exit_code = self.process.returncode
        status = "completed" if exit_code == 0 and not self.abort_reason else "failed"
        if self.abort_reason:
            status = "budget_exceeded" if self.abort_reason.startswith("VRAM") else "failed"
        # Reap remaining descendants even after a nominally successful parent exit.
        deadline = time.perf_counter() + CONFIG["gpu_release_timeout_seconds"]
        busy = []
        while time.perf_counter() < deadline:
            _, busy = nvml_process_usage(GPU_HANDLES[self.slot], self.monitor.process_ids)
            if not busy:
                break
            time.sleep(0.05)
        if busy:
            status = "gpu_release_failed"
        memory = self.monitor.finish()
        if memory["budget_violation"] and CONFIG["enforce_observed_budget"]:
            status = "budget_exceeded" if not busy else status
            self.abort_reason = self.abort_reason or "VRAM exceeded observed budget: " + json.dumps(memory["budget_violation"])
        if memory["monitor_error"]:
            status = "monitor_failed"
        result = {**self.job, "gpu_slot": self.slot, "gpu_uuid": GPU_INVENTORY[self.slot]["uuid"],
                  "status": status, "exit_code": exit_code, "process_wall_seconds": time.perf_counter()-self.started,
                  "error": self.abort_reason or ("\n".join(self.tail)[-5000:] if exit_code else None),
                  "gpu_release_failed_pids": busy, **memory}
        atomic_json(Path(self.job["job_dir"]) / "supervisor_result.json", result)
        TIMELINE.append({"event": "finish", "job_id": self.job["job_id"], "model": self.job["model"],
                         "gpu_slot": self.slot, "timestamp": time.time(), "status": status})
        return result

def preferred_job_index(pending, active_models):
    for index, job in enumerate(pending):
        if job["model"] not in active_models:
            return index
    return 0 if pending else None

def live_html(active, completed, total):
    rows = []
    for slot in range(2):
        worker = active.get(slot)
        if worker:
            label = html.escape(worker.job["job_id"])
            line = html.escape(worker.latest_line)
            memory = worker.monitor.last_device / 1e9
            process_memory = worker.monitor.last_process
            process_text = "unavailable" if process_memory is None else f"{process_memory/1e9:.3f} GB"
            rows.append(f"<tr><td>GPU {slot}</td><td>{label}<br><pre>{line}</pre></td><td>{memory:.3f} GB device<br>{process_text} process</td></tr>")
        else:
            rows.append(f"<tr><td>GPU {slot}</td><td>Idle</td><td></td></tr>")
    return f"<p>{completed}/{total} jobs finished. One isolated worker per GPU; distinct models preferred.</p><table><tr><th>Slot</th><th>Model progress</th><th>Live NVML usage</th></tr>{''.join(rows)}</table>"

def run_benchmark(jobs):
    pending = []
    for job in jobs:
        reason = job["blocked_reason"] or (hardware_error if not HARDWARE_READY else None)
        if reason:
            result = {**job, "status": job.get("blocked_status", "setup_failed"), "error": reason, "exit_code": None, "gpu_slot": None,
                      "gpu_uuid": None, "process_wall_seconds": None, "hard_instantaneous_peak_certified": False}
            JOB_RESULTS.append(result)
            atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
        else:
            pending.append(job)
    active = {}
    disabled_slots = set()
    progress = tqdm(total=len(jobs), initial=len(JOB_RESULTS), desc="Benchmark jobs", unit="job")
    live = display(HTML(live_html(active, len(JOB_RESULTS), len(jobs))), display_id=True)
    try:
        while pending or active:
            for slot in range(2):
                if slot in active or slot in disabled_slots or not pending:
                    continue
                index = preferred_job_index(pending, {worker.job["model"] for worker in active.values()})
                job = pending.pop(index)
                try:
                    active[slot] = Worker(job, slot)
                except Exception as exc:
                    result = {**job, "status": "launch_failed", "error": f"{type(exc).__name__}: {exc}",
                              "exit_code": None, "gpu_slot": slot, "gpu_uuid": GPU_INVENTORY[slot]["uuid"]}
                    JOB_RESULTS.append(result)
                    atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
                    progress.update(1)
            for slot, worker in list(active.items()):
                elapsed = time.perf_counter()-worker.started
                if worker.process.poll() is None:
                    if worker.monitor.error:
                        worker.terminate("NVML monitor failed: " + worker.monitor.error)
                    elif worker.monitor.violation and CONFIG["enforce_observed_budget"]:
                        worker.terminate("VRAM exceeded observed budget: " + json.dumps(worker.monitor.violation))
                    elif elapsed > CONFIG["job_timeout_seconds"]:
                        worker.terminate("Job timeout")
                if worker.process.poll() is not None:
                    result = worker.finish()
                    JOB_RESULTS.append(result)
                    progress.update(1)
                    if result.get("gpu_release_failed_pids"):
                        disabled_slots.add(slot)
                    del active[slot]
            if len(disabled_slots) == 2 and pending:
                for job in pending:
                    result = {**job, "status": "gpu_unavailable", "error": "Both GPUs retain tracked allocations", "exit_code": None}
                    JOB_RESULTS.append(result)
                    atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
                    progress.update(1)
                pending.clear()
            live.update(HTML(live_html(active, len(JOB_RESULTS), len(jobs))))
            time.sleep(0.2)
    except BaseException as supervisor_error:
        for worker in active.values():
            worker.terminate("Interrupted by user" if isinstance(supervisor_error, KeyboardInterrupt) else "Supervisor error: " + str(supervisor_error))
            JOB_RESULTS.append(worker.finish())
        for job in pending:
            result = {**job, "status": "cancelled", "error": "Supervisor interrupted: " + str(supervisor_error), "exit_code": None}
            JOB_RESULTS.append(result)
            atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
        print("Supervisor stopped:", type(supervisor_error).__name__, str(supervisor_error))
        print("Run the report/ZIP cell to package all completed and partial outputs.")
    finally:
        progress.close()
        atomic_json(RUN_DIR / "supervisor_results.json", JOB_RESULTS)
        atomic_json(RUN_DIR / "concurrency_timeline.json", TIMELINE)
        live.update(HTML(live_html({}, len(JOB_RESULTS), len(jobs))))

jobs = profile_jobs()
atomic_json(RUN_DIR / "experiment_plan.json", {"config": CONFIG, "cases": TEXT_CASES, "profiles": MODEL_PROFILES, "jobs": jobs})
print(f"{len(jobs)} model/profile/text jobs; {sum(j['n'] for j in jobs)} planned WAVs")
if KERNEL_READY:
    run_benchmark(jobs)
else:
    JOB_RESULTS = [{**job, "status": "kernel_failed", "error": hardware_error, "exit_code": None} for job in jobs]
    atomic_json(RUN_DIR / "supervisor_results.json", JOB_RESULTS)


## 7. Reports, listening players and ZIP download

`jobs.csv` includes every scheduled job, including setup/runtime failures and
duration plans available before a failure. `samples.csv` independently compares
predicted and actual duration/rate/sample count for each created WAV. Planner
timing scopes differ by model and are labeled explicitly. VRAM CSV traces and
native manifests remain available per job.

`quality.csv`/`quality.json` include acoustic gaps/clipping statistics, independent
Whisper transcripts, word edits, and ASR word error rate. Checks run on **CPU**
after generation and do not enter model VRAM or estimator latency measurements.
ASR errors are screening evidence: pronunciation, punctuation and recognition
errors can contribute. Listen to flagged samples before deciding a word is missing.
Disable ASR with `asr_quality_check=False` to retain acoustic checks without its
additional download/runtime, or disable both with `quality_checks=False`.

Download the ZIP from the link or Kaggle's **Output** panel. After extracting,
open `index.html` to compare the report and listen. Dependencies and pretrained
weights are excluded from the ZIP. A full kernel/VM shutdown can interrupt
packaging; completed files remain under `/kaggle/working/tts_benchmark/runs`
for as long as Kaggle retains that filesystem.


In [ ]:
PLANNER_SCOPE = {
    "kokoro": "First loaded voice-conditioned G2P/duration pass; not a warmed latency guarantee",
    "f5": "Allocated frame plan including CFM minimum-length clamp; no waveform generation",
    "emotivoice": "Warmed speaker/style-conditioned duration preflight",
    "styletts2": "BERT/style diffusion/duration preflight for all requested takes",
    "omnivoice": "Upstream rule token allocation; separate cold preflight/import/reference timing",
    "supertonic": "Warmed voice-conditioned duration predictor; startup probes excluded",
}

def read_json_safely(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None

def find_manifest(job):
    raw = Path(job["job_dir"]) / "raw"
    preferred = {"kokoro": ["manifest.json", "preview.json"], "f5": ["manifest.json"],
                 "styletts2": ["manifest.json"], "emotivoice": ["emotivoice_manifest.json"],
                 "supertonic": ["supertonic_manifest.json"], "omnivoice": ["manifest.json", "estimate.json"]}
    for filename in preferred[job["model"]]:
        for path in sorted(raw.rglob(filename)):
            data = read_json_safely(path)
            if data is not None:
                return path, data
    return None, {}

def native_schema(job, data):
    model = job["model"]
    samples = data.get("samples", data.get("outputs", data.get("runs", [])))
    planner = cold = None
    prediction = None
    export_rate = data.get("export_sample_rate", data.get("export_rate"))
    predictions = []
    if model == "kokoro":
        prediction = data.get("predicted_export_samples")
        planner, cold = data.get("preflight_seconds"), data.get("startup_seconds")
    elif model == "f5":
        preview = data.get("preview", {})
        prediction = preview.get("exported_samples")
        planner, cold = preview.get("preflight_seconds"), preview.get("startup_seconds")
        export_rate = data.get("settings", {}).get("sample_rate")
    elif model == "styletts2":
        planner, cold = data.get("preflight_seconds"), data.get("startup_seconds")
        export_rate = data.get("settings", {}).get("sample_rate")
        predictions = [{"seed": p.get("seed"), "predicted_samples": p.get("exported_samples"), "predicted_seconds": p.get("seconds")} for p in data.get("preview", [])]
    elif model in {"emotivoice", "supertonic"}:
        prediction = data.get("predicted_samples")
        planner, cold = data.get("warm_preview_seconds"), data.get("startup_seconds")
    elif model == "omnivoice":
        plan = data.get("plan", {})
        prediction = plan.get("predicted_export_samples")
        planner = plan.get("planner_ms", 0) / 1000 if "planner_ms" in plan else None
        cold = plan.get("cold_preflight_ms", 0) / 1000 if "cold_preflight_ms" in plan else None
        export_rate = plan.get("sample_rate")
    export_rate = export_rate or job.get("settings", {}).get("sample_rate")
    if not export_rate:
        export_rate = {"emotivoice": 16000, "supertonic": 44100}.get(model, 24000)
    if prediction is not None:
        predictions = [{"seed": job["seed"] + index, "predicted_samples": prediction, "predicted_seconds": prediction/export_rate} for index in range(job["n"])]
    rows = []
    for index, record in enumerate(samples):
        seed = record.get("seed", job["seed"] + index)
        predicted = record.get("predicted_samples", prediction)
        if model == "styletts2":
            previews = data.get("preview", [])
            match = next((p for p in previews if p.get("seed") == seed), None)
            if match:
                predicted = match.get("exported_samples")
        rows.append({"seed": seed, "path": record.get("path", record.get("file")),
                     "predicted_samples": predicted, "generation_seconds": record.get("generation_seconds"),
                     "native_record": record})
    return {"rows": rows, "prediction": prediction, "predictions": predictions, "export_rate": export_rate,
            "planner_seconds": planner, "startup_seconds": cold}

def allocator_peaks(data):
    allocated, reserved = [], []
    def visit(value):
        if isinstance(value, dict):
            for key, item in value.items():
                if isinstance(item, (int, float)) and not isinstance(item, bool):
                    if key in {"torch_peak_allocated_bytes", "peak_torch_allocated_bytes"}:
                        allocated.append(item)
                    elif key in {"torch_peak_reserved_bytes", "peak_torch_reserved_bytes"}:
                        reserved.append(item)
                visit(item)
        elif isinstance(value, list):
            for item in value:
                visit(item)
    visit(data)
    return max(allocated, default=None), max(reserved, default=None)

def collect_reports():
    audio_dir = PACKAGE_DIR / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    job_rows, sample_rows = [], []
    for job in JOB_RESULTS:
        manifest_path, manifest = find_manifest(job)
        schema = native_schema(job, manifest)
        allocated, reserved = allocator_peaks(manifest)
        listed = {Path(row["path"]).resolve(): row for row in schema["rows"] if row["path"]}
        found = sorted((Path(job["job_dir"]) / "raw").rglob("*.wav"))
        observed_errors = []
        final_count = 0
        for index, source in enumerate(found):
            native = listed.get(source.resolve())
            seed = native["seed"] if native else None
            predicted = native["predicted_samples"] if native else schema["prediction"]
            filename = f"{job['job_id']}__seed{seed if seed is not None else 'unknown'}__take{index+1:03d}"
            row = {"job_id": job["job_id"], "model": job["model"], "profile": job["profile"], "case": job["case"],
                   "word_count": job["word_count"], "seed": seed, "take": index+1,
                   "speaker_source": job.get("speaker_source"),
                   "job_status": job["status"], "original_path": str(source), "predicted_samples": predicted,
                   "planner_seconds": schema["planner_seconds"], "planner_scope": PLANNER_SCOPE[job["model"]],
                   "generation_seconds": native["generation_seconds"] if native else None,
                   "gpu_slot": job.get("gpu_slot"), "gpu_uuid": job.get("gpu_uuid"),
                   "nvml_sampled_process_tree_peak_bytes": job.get("nvml_sampled_process_tree_peak_bytes"),
                   "nvml_sampled_device_peak_bytes": job.get("nvml_sampled_device_peak_bytes"),
                   "torch_peak_allocated_bytes": allocated, "torch_peak_reserved_bytes": reserved,
                   "actual_samples": None, "sample_rate": None, "channels": None,
                   "planned_sample_rate": schema["export_rate"], "sample_rate_match": None,
                   "actual_seconds": None, "predicted_seconds": predicted / schema["export_rate"] if predicted is not None else None,
                   "relative_duration_error": None, "duration_error_percent": None,
                   "exact_sample_count_match": None, "under_one_percent": None}
            try:
                info = sf.info(source)
                actual = info.frames
                actual_seconds = actual / info.samplerate
                planned_rate = schema["export_rate"]
                predicted_seconds = predicted / planned_rate if predicted is not None else None
                error = abs(actual_seconds-predicted_seconds) / actual_seconds if actual_seconds > 0 and predicted_seconds is not None else None
                is_temporary = "temporary" in source.name or bool(re.search(r"\.(?:tmp|part)\.wav$", source.name, re.I))
                valid = native is not None and error is not None and error < 0.01 and actual == predicted and info.samplerate == planned_rate and info.channels == 1 and not is_temporary
                partial = not valid
                destination = audio_dir / (("PARTIAL__" if partial else "") + filename + f"__sr{info.samplerate}.wav")
                shutil.copy2(source, destination)
                row.update({"audio_file": str(destination.relative_to(PACKAGE_DIR)), "actual_samples": actual,
                            "sample_rate": info.samplerate, "channels": info.channels,
                            "planned_sample_rate": planned_rate, "sample_rate_match": info.samplerate == planned_rate,
                            "actual_seconds": actual_seconds, "predicted_seconds": predicted_seconds,
                            "relative_duration_error": error, "duration_error_percent": error*100 if error is not None else None,
                            "exact_sample_count_match": actual == predicted if predicted is not None else None,
                            "under_one_percent": error < 0.01 if error is not None else None,
                            "audio_status": "partial" if partial else "verified", "error": None})
                if error is not None:
                    observed_errors.append(error)
                if valid:
                    final_count += 1
            except Exception as exc:
                destination = audio_dir / ("UNREADABLE__" + filename + ".wav")
                shutil.copy2(source, destination)
                row.update({"audio_file": str(destination.relative_to(PACKAGE_DIR)), "audio_status": "unreadable", "error": str(exc)})
            sample_rows.append(row)
        if job["status"] == "completed" and final_count != job["n"]:
            job["status"] = "report_validation_failed"
            job["error"] = f"Expected {job['n']} exact exports, independently verified {final_count}"
        for row in sample_rows:
            if row["job_id"] == job["job_id"]:
                row["job_status"] = job["status"]
        job_rows.append({"job_id": job["job_id"], "model": job["model"], "profile": job["profile"], "case": job["case"],
            "speaker_source": job.get("speaker_source"),
            "word_count": job["word_count"], "requested_samples": job["n"], "created_wavs": len(found), "verified_wavs": final_count,
            "status": job["status"], "error": job.get("error"), "gpu_slot": job.get("gpu_slot"), "gpu_uuid": job.get("gpu_uuid"),
            "exit_code": job.get("exit_code"), "process_wall_seconds": job.get("process_wall_seconds"),
            "planner_seconds": schema["planner_seconds"], "startup_seconds": schema["startup_seconds"],
            "planner_seconds_per_take_amortized": schema["planner_seconds"]/job["n"] if schema["planner_seconds"] is not None and job["model"] == "styletts2" else None,
            "planner_scope": PLANNER_SCOPE[job["model"]], "max_duration_error_percent": max(observed_errors)*100 if observed_errors else None,
            "planned_samples_json": json.dumps(schema["predictions"]), "planned_sample_rate": schema["export_rate"],
            "nvml_sampled_process_tree_peak_bytes": job.get("nvml_sampled_process_tree_peak_bytes"),
            "nvml_sampled_device_peak_bytes": job.get("nvml_sampled_device_peak_bytes"),
            "torch_peak_allocated_bytes": allocated, "torch_peak_reserved_bytes": reserved,
            "budget_violation": json.dumps(job["budget_violation"]) if job.get("budget_violation") else None,
            "monitor_error": job.get("monitor_error"), "trace_samples": job.get("samples"),
            "monitor_interval_ms": CONFIG["nvml_interval_ms"], "instantaneous_peak_certified": False,
            "native_manifest": str(manifest_path) if manifest_path else None})
    return job_rows, sample_rows

def run_quality_checks(sample_rows):
    """Independent CPU diagnostics; errors never prevent audio/report packaging."""
    quality_dir = RUN_DIR / "quality"
    quality_dir.mkdir(parents=True, exist_ok=True)
    output = quality_dir / "quality_report.json"
    if not CONFIG.get("quality_checks", False) or not sample_rows:
        result = {"status": "disabled" if sample_rows else "no_audio", "rows": []}
        atomic_json(output, result)
        return result
    jobs_by_id = {job["job_id"]: job for job in JOB_RESULTS}
    requests = []
    for row in sample_rows:
        job = jobs_by_id[row["job_id"]]
        text = (Path(job["job_dir"]) / "text.txt").read_text(encoding="utf-8")
        requests.append({"job_id": row["job_id"], "model": row["model"], "seed": row["seed"],
                         "audio_file": str((PACKAGE_DIR / row["audio_file"]).resolve()), "text": text})
    request_path = quality_dir / "requests.json"
    atomic_json(request_path, requests)
    log_path = quality_dir / "quality_check.log"
    setup_error = None
    try:
        python = ENV_DIR / "quality_checks" / "bin" / "python"
        if not python.is_file():
            setup_command(UV + ["venv", "--python", "3.11", "--seed", str(python.parent.parent)], log_path)
        install = UV + ["pip", "install", "--python", str(python), "--link-mode", "hardlink"]
        setup_command(install + ["numpy==1.26.4", "soundfile==0.13.1"], log_path)
        use_asr = bool(CONFIG.get("asr_quality_check", False))
        model = CONFIG.get("asr_model", "base.en")
        if use_asr:
            try:
                setup_command(install + ["faster-whisper==1.2.1", "ctranslate2==4.6.0",
                              "onnxruntime==1.23.2", "huggingface_hub==0.29.3", "av==15.1.0"], log_path)
                if model == "base.en":
                    model = str(ASSETS / "quality_asr" / "base.en")
                    downloader = "from huggingface_hub import snapshot_download; import sys; snapshot_download('Systran/faster-whisper-base.en', revision='3d3d5dee26484f91867d81cb899cfcf72b96be6c',local_dir=sys.argv[1],max_workers=1)"
                    setup_command([python, "-c", downloader, model], log_path)
            except Exception as exc:
                use_asr = False
                setup_error = str(exc)
                print("ASR setup failed; continuing with waveform diagnostics:", exc)
        command = [str(python), str(SCRIPTS_DIR / "audio_quality_check.py"), "--requests-json", str(request_path),
                   "--output-json", str(output), "--model", str(model), "--language", CONFIG.get("asr_language", "en"),
                   "--threads", str(CONFIG["cpu_threads_per_job"])]
        if use_asr:
            command.append("--asr")
        print(f"Checking {len(requests)} WAVs on CPU; ASR={use_asr}. Details: {log_path}")
        # Explicit CUDA hiding and CPU/int8 ASR keep diagnostics out of TTS VRAM/timing measurements.
        with log_path.open("a", encoding="utf-8") as log:
            subprocess.run(command, env=BASE_ENV, stdout=log, stderr=subprocess.STDOUT, check=True,
                           timeout=CONFIG.get("quality_timeout_seconds", 7200))
        result = read_json_safely(output)
        if not isinstance(result, dict):
            raise RuntimeError("Quality helper returned no valid report")
        result["asr_setup_error"] = setup_error
        result["asr_completed_files"] = sum(row.get("asr_status") == "completed" for row in result.get("rows", []))
        result["asr_failed_files"] = sum(row.get("asr_status") == "failed" for row in result.get("rows", []))
        if CONFIG.get("asr_model", "base.en") == "base.en" and use_asr:
            result["asr_weight_revision"] = "3d3d5dee26484f91867d81cb899cfcf72b96be6c"
        atomic_json(output, result)
    except Exception as exc:
        result = {"status": "failed", "error": str(exc), "rows": [], "asr_setup_error": setup_error}
        atomic_json(output, result)
        print("Quality diagnostics failed; WAVs and timing reports will still be packaged:", exc)
    lookup = {(r.get("job_id"), str(Path(r.get("audio_file", "")).resolve())): r for r in result.get("rows", [])}
    for row in sample_rows:
        diagnostics = lookup.get((row["job_id"], str((PACKAGE_DIR / row["audio_file"]).resolve())))
        if diagnostics:
            row["quality_status"] = diagnostics.get("status")
            for field in ("asr_status", "asr_error", "longest_interior_silence_seconds", "clipped_sample_fraction", "word_error_rate", "deletions", "insertions", "substitutions", "transcript"):
                row[field] = diagnostics.get(field)
    return result

def package_results(job_rows, sample_rows):
    reports = PACKAGE_DIR / "reports"
    reports.mkdir(parents=True, exist_ok=True)
    quality_report = run_quality_checks(sample_rows)
    atomic_json(reports / "quality.json", quality_report)
    pd.DataFrame(quality_report.get("rows", [])).to_csv(reports / "quality.csv", index=False)
    shutil.copytree(RUN_DIR / "quality", PACKAGE_DIR / "quality_details", dirs_exist_ok=True)
    jobs_frame, samples_frame = pd.DataFrame(job_rows), pd.DataFrame(sample_rows)
    jobs_frame.to_csv(reports / "jobs.csv", index=False)
    samples_frame.to_csv(reports / "samples.csv", index=False)
    atomic_json(reports / "jobs.json", job_rows)
    atomic_json(reports / "samples.json", sample_rows)
    atomic_json(reports / "experiment.json", {"config": CONFIG, "cases": TEXT_CASES, "profiles": MODEL_PROFILES,
                "hardware": GPU_INVENTORY, "setup": SETUP_RESULTS, "preflight": globals().get("PREFLIGHT_RESULTS", {}),
                "runtime_validation": "This run's data only; failures are included. Sampled NVML does not certify instantaneous peaks."})
    sources = PACKAGE_DIR / "source"
    sources.mkdir(exist_ok=True)
    for path in SCRIPTS_DIR.glob("*.py"):
        shutil.copy2(path, sources / path.name)
    if (SCRIPTS_DIR / "source_manifest.json").exists():
        shutil.copy2(SCRIPTS_DIR / "source_manifest.json", sources / "source_manifest.json")
    for name in ("hardware.json", "supervisor_results.json", "concurrency_timeline.json", "experiment_plan.json", "reference_report.json"):
        source = RUN_DIR / name
        if source.exists():
            shutil.copy2(source, reports / name)
    for source in (ROOT / "setup_report.json", ROOT / "preflight_report.json", ROOT / "reference_report.json", ROOT / "emotivoice_checkpoint_provenance.json"):
        if source.exists():
            shutil.copy2(source, reports / source.name)
    reference_root = ROOT / "reference"
    if reference_root.exists():
        for source in reference_root.rglob("*.json"):
            destination = reports / "reference_provenance" / source.relative_to(reference_root)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    logs_dir = ROOT / "logs" / "setup"
    if logs_dir.exists():
        shutil.copytree(logs_dir, PACKAGE_DIR / "setup_logs", dirs_exist_ok=True)
    for job in JOB_RESULTS:
        base = Path(job["job_dir"])
        target = PACKAGE_DIR / "job_details" / job["job_id"]
        target.mkdir(parents=True, exist_ok=True)
        for source in base.rglob("*"):
            if source.is_file() and source.suffix.lower() in {".json", ".txt", ".log", ".csv", ".pt"}:
                # .pt here is the small StyleTTS2 cached duration plan, never model checkpoints.
                relative = source.relative_to(base)
                destination = target / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
    if CONFIG["include_reference_in_zip"] and globals().get("REFERENCE_AUDIO"):
        reference = Path(REFERENCE_AUDIO)
        if reference.exists():
            target = PACKAGE_DIR / "reference"
            target.mkdir(exist_ok=True)
            shutil.copy2(reference, target / reference.name)
            (target / "transcript.txt").write_text(REFERENCE_TEXT or "", encoding="utf-8")
    # A static report links to every named clip; suitable after unzipping.
    clips = []
    for row in sample_rows:
        href = html.escape(row["audio_file"], quote=True)
        label = html.escape(f"{row['model']} / {row['profile']} / {row['case']} / seed {row.get('seed')} / {row['audio_status']} / speaker: {row.get('speaker_source')}")
        clips.append(f'<p>{label}</p><audio controls preload="none" src="{href}"></audio><a href="{href}">WAV</a>')
    table = jobs_frame.to_html(index=False, escape=True) if not jobs_frame.empty else "<p>No jobs completed.</p>"
    figures_html = ""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        if job_rows:
            y = list(range(len(job_rows)))
            figure, axis = plt.subplots(figsize=(12, max(4, len(y)*0.42)))
            labels = [f"{r['model']}/{r['profile']}/{r['case']} GPU{r.get('gpu_slot')}" for r in job_rows]
            for field, offset, label in (("nvml_sampled_device_peak_bytes", -.24, "NVML device sampled"),
                                         ("nvml_sampled_process_tree_peak_bytes", 0, "NVML process tree sampled"),
                                         ("torch_peak_allocated_bytes", .24, "PyTorch allocator peak")):
                values = [r[field]/1e9 if r.get(field) is not None else float("nan") for r in job_rows]
                axis.barh([i+offset for i in y], values, height=.22, label=label)
            axis.axvline(CONFIG["max_vram_gb"], color="red", linestyle="--", label="Configured VRAM target")
            axis.set_yticks(y, labels)
            axis.invert_yaxis()
            axis.set_xlabel("GPU memory (decimal GB); absent measurements are blank")
            axis.legend(loc="best")
            figure.tight_layout()
            figure.savefig(reports / "vram_peaks.png", dpi=150)
            plt.close(figure)
            figures_html += '<h2>Memory measurements</h2><img style="max-width:100%" src="reports/vram_peaks.png">'
        timed = [r for r in job_rows if r.get("planner_seconds") is not None and r["planner_seconds"] > 0]
        if timed:
            figure, axis = plt.subplots(figsize=(10, 5))
            for model in sorted({r["model"] for r in timed}):
                rows = sorted([r for r in timed if r["model"] == model], key=lambda r:r["word_count"])
                axis.plot([r["word_count"] for r in rows], [r["planner_seconds"]*1000 for r in rows], "o", label=model)
            axis.set_yscale("log")
            axis.set_xlabel("Words")
            axis.set_ylabel("Measured planning time (ms, log scale)")
            axis.set_title("Model-specific timing scopes differ; StyleTTS2 includes all requested takes")
            axis.legend()
            axis.grid(True, alpha=.3)
            figure.tight_layout()
            figure.savefig(reports / "planning_latency.png", dpi=150)
            plt.close(figure)
            figures_html += '<h2>Planning latency</h2><img style="max-width:100%" src="reports/planning_latency.png">'
    except Exception as exc:
        atomic_json(reports / "plot_failure.json", {"error": str(exc)})
    quality_frame = pd.DataFrame(quality_report.get("rows", []))
    quality_columns = ["job_id", "model", "seed", "status", "asr_status", "asr_error",
                       "word_error_rate", "deletions", "insertions", "substitutions",
                       "longest_interior_silence_seconds", "clipped_sample_fraction", "asr_transcript"]
    quality_table = (quality_frame.reindex(columns=[name for name in quality_columns if name in quality_frame.columns])
                     .to_html(index=False, escape=True) if not quality_frame.empty
                     else html.escape(str(quality_report.get("status"))))
    report_html = ('<!doctype html><meta charset="utf-8"><title>TTS benchmark</title>'
        '<style>body{font-family:system-ui;margin:24px}table{border-collapse:collapse;font-size:12px}td,th{padding:6px;border:1px solid #ddd}audio{width:500px;max-width:100%}</style>'
        '<h1>TTS timing and GPU memory benchmark</h1><p>NVML peaks are sampled; allocator peaks exclude driver/library memory. Failed and partial jobs are retained.</p>'
        + figures_html + table + '<h2>Word completeness and acoustic diagnostics</h2><p>ASR disagreements are screening results, not proof of a synthesis error. Silence can be intentional; inspect the transcript and listen.</p>'
        + quality_table + '<p>Full word alignments, silence spans, and ASR timestamps are in reports/quality.json.</p>'
        + '<h2>Recordings</h2>' + ''.join(clips))
    (PACKAGE_DIR / "index.html").write_text(report_html, encoding="utf-8")
    (PACKAGE_DIR / "README.txt").write_text(
        "Open index.html to compare reports and play recordings.\n"
        "reports/jobs.csv: every planned job, including setup/runtime/budget failures.\n"
        "reports/samples.csv: independent WAV header duration checks and canonical filenames.\n"
        "reports/quality.csv and quality.json: CPU acoustic checks, ASR transcripts and word edits; ASR is fallible.\n"
        "job_details/: exact config, text, native manifests, progress logs and NVML time series.\n"
        "source/: embedded standalone adapters and SHA256 provenance.\n"
        "PARTIAL/UNREADABLE filenames are not verified final samples.\n"
        "Audio length accuracy does not establish text intelligibility or natural speaking time.\n"
        "Dependency environments and model weights are excluded from this archive.\n", encoding="utf-8")
    archive = ROOT.parent / f"tts_benchmark_{RUN_ID}.zip"
    files = [path for path in PACKAGE_DIR.rglob("*") if path.is_file()]
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as zipped:
        for source in tqdm(files, desc="Packaging WAVs and reports", unit="file"):
            zipped.write(source, source.relative_to(PACKAGE_DIR))
    return jobs_frame, samples_frame, archive

if KERNEL_READY:
    job_rows, sample_rows = collect_reports()
    JOBS_TABLE, SAMPLES_TABLE, ZIP_PATH = package_results(job_rows, sample_rows)
    print(f"ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size/1e6:.1f} MB)")
    display(FileLink(str(ZIP_PATH)))
    display(FileLink(str(PACKAGE_DIR / "index.html")))
    if not JOBS_TABLE.empty:
        display(JOBS_TABLE.reindex(columns=["model", "profile", "word_count", "status", "verified_wavs", "planner_seconds", "max_duration_error_percent", "nvml_sampled_process_tree_peak_bytes", "nvml_sampled_device_peak_bytes"]))
    if not SAMPLES_TABLE.empty:
        display(SAMPLES_TABLE.reindex(columns=["model", "profile", "word_count", "seed", "speaker_source", "predicted_seconds", "actual_seconds", "duration_error_percent", "longest_interior_silence_seconds", "word_error_rate", "audio_status", "audio_file"]))
    played_models = set()
    for row in sample_rows:
        if len(played_models) >= CONFIG["preview_players"]:
            break
        if row["audio_status"] == "verified" and row["model"] not in played_models:
            print(row["model"], row["profile"], row["case"], "seed", row["seed"])
            display(Audio(filename=str(PACKAGE_DIR / row["audio_file"])))
            played_models.add(row["model"])
else:
    ZIP_PATH = ROOT.parent / f"tts_benchmark_{RUN_ID}_failure.zip"
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
        for file in RUN_DIR.rglob("*.json"):
            archive.write(file, file.relative_to(RUN_DIR))
        for file in ROOT.glob("*report.json"):
            archive.write(file, "reports/" + file.name)
    print("Kernel dependency setup failed. Diagnostic ZIP saved:", ZIP_PATH)
    from IPython.display import FileLink, display
    display(FileLink(str(ZIP_PATH)))


## Primary references

- [Kaggle notebook settings](https://www.kaggle.com/docs/notebooks)
- [Kaggle accelerator identifiers](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md)
- [ONNX Runtime CUDA requirements and arena limits](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html)
- [PyTorch CUDA wheel versions](https://pytorch.org/get-started/previous-versions/)
- [Kokoro](https://github.com/hexgrad/kokoro/tree/dfb907a02bba8152ca444717ca5d78747ccb4bec)
- [F5-TTS](https://github.com/SWivid/F5-TTS/tree/283252563dbf91be625e0c27926acfaac449186c)
- [EmotiVoice](https://github.com/netease-youdao/EmotiVoice/tree/59f0f36de4db12825f4705dd4e0780d79dd6bb01)
- [StyleTTS2](https://github.com/yl4579/StyleTTS2/tree/5cedc71c333f8d8b8551ca59378bdcc7af4c9529)
- [OmniVoice](https://github.com/k2-fsa/OmniVoice/tree/08be0b4ccbac3e13e374e86fbfead4b4cac343e2)
- [Supertonic 3](https://github.com/supertone-oss-archive/supertonic/tree/1e9799e964ea4c0dad7cde993b65c3c813a7b373)
- [Supertonic speaker descriptions](https://github.com/supertone-oss-archive/supertonic-py/blob/main/docs/voices.md)
- [EmotiVoice speaker descriptions](https://github.com/netease-youdao/EmotiVoice/tree/main/data/youdao/text)
- [OmniVoice design attributes](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/voice-design.md)
- [CPU ASR implementation](https://github.com/SYSTRAN/faster-whisper)

Use pretrained model licenses appropriate to your intended use; for example,
F5's published checkpoint is CC-BY-NC-4.0. This notebook is a benchmarking tool.
